# DSN Bootcamp Qualification Hackathon 2026 — Validation & Model Selection Report

## 1. Purpose

This document summarizes the work completed in `Notebooks/02_validation_baseline.ipynb`.

The objective was to establish a rigorous, leakage-safe validation framework, compare candidate algorithms, test feature hypotheses, perform targeted tuning, analyze errors, and freeze a final model configuration before full-data training and submission.

**Metric:** RMSE on the original `total_sales` scale. Lower is better.

---

## 2. Validation Protocol

- 5-fold `KFold`, `shuffle=True`, `random_state=42`.
- The same locked folds were used for all model and feature comparisons.
- Leaderboard feedback was not used as validation.
- `product_category_clean` was created by lowercasing/trimming `product_category`.
- `id` was excluded because it is unique.
- CatBoost categorical columns were explicitly declared.
- Missing categorical values were represented as `__MISSING__`.
- Target-derived features were only considered when constructed from training information within each fold.
- The final confirmation reproduced the champion using the same locked folds.

---

## 3. Baseline Algorithm Comparison

| Model | Mean CV RMSE |
|---|---:|
| Naive mean | 1697.7241 |
| Random Forest | 1276.3164 |
| Ridge | 1136.6424 |
| LightGBM | 1133.7387 |
| HistGradientBoosting | 1125.5758 |
| XGBoost | 1109.3365 |
| CatBoost — log target | 1112.2017 |
| **CatBoost — raw target** | **1085.9531** |

Tree boosting clearly outperformed the naive, linear, and Random Forest baselines. CatBoost on the raw target became the initial champion.

The log-target experiment was explicitly scored on original-scale RMSE and was rejected.

---

## 4. Feature Engineering Experiments

| Experiment | CV RMSE | Decision |
|---|---:|---|
| Product-relative price | 1085.0348 | Reject — gain too small |
| Weight missing flag | 1086.8995 | Reject |
| Product median weight | 1086.5230 | Reject |
| Explicit store × category | 1086.7950 | Reject |
| Product target encoding | 1185.4279 | Reject — poor and leakage-sensitive |
| Product price squared | 1075.3621 | Neutral/reject — CatBoost captures curvature |
| Store-price band | 1083.2568 | Reject |
| Store-relative price | 1075.8920 | Reject |

Feature engineering was hypothesis-driven rather than a bulk feature dump.

The repeated experiments showed that the existing CatBoost representation already captured most of the useful price/store/category interactions tested.

---

## 5. CatBoost Tuning

### Depth

Best tested depth was 4.

| Depth | CV RMSE |
|---:|---:|
| 4 | **1080.5855** |
| 5 | 1084.1052 |
| 6 | 1085.9531 |
| 7 | 1089.8798 |
| 8 | 1091.5018 |

### Iterations

With depth 4 and learning rate 0.05:

| Iterations | CV RMSE |
|---:|---:|
| 300 | **1075.7865** |
| 500 | 1080.5855 |
| 800 | 1086.5047 |
| 1000 | 1089.7414 |

More iterations hurt rather than helped, suggesting overfitting.

### Learning rate

The refined search identified `learning_rate=0.025` as the best tested value, producing **1075.3555 RMSE** before the later Ordered Boosting/L2 improvements.

### Ordered boosting

- Plain configuration: 1075.3555
- Ordered boosting: **1074.9861**

Decision: use `boosting_type="Ordered"`.

### L2 regularization

| `l2_leaf_reg` | CV RMSE |
|---:|---:|
| 1 | **1074.9159** |
| 3 | 1074.9861 |
| 5 | 1075.2883 |
| 10 | 1075.5594 |
| 20 | 1075.2896 |

Decision: use `l2_leaf_reg=1`; further L2 tuning was stopped because gains were tiny.

### Random strength

| `random_strength` | CV RMSE |
|---:|---:|
| 0.00 | 1076.1353 |
| 0.25 | 1075.9468 |
| 0.50 | 1075.0384 |
| **1.00** | **1074.9159** |
| 2.00 | 1076.2659 |

Decision: retain `random_strength=1.0`.

---

## 6. XGBoost and LightGBM Checks

### XGBoost

Depth testing strongly favored shallow trees:

- depth 3: 1096.9276
- depth 4: 1113.6015
- depth 5: 1129.4606
- depth 6: 1150.6766
- depth 7: 1160.5989

The best tested configuration after iteration tuning used depth 3, 200 estimators, learning rate 0.05, subsample 0.8, and colsample 0.8, with approximately **1082.36 CV RMSE**.

Further tuning was stopped because it remained behind CatBoost.

### LightGBM

The best tested configuration used `num_leaves=7` and `n_estimators=100`, with approximately **1081.99 CV RMSE**.

Further tuning was stopped because it remained behind CatBoost.

---

## 7. OOF Diversity and Blending

Strict OOF predictions were generated using the locked folds.

- CatBoost: 1075.5259 OOF RMSE
- XGBoost: 1082.5609
- LightGBM: 1082.1644

Residual correlations were extremely high:

- CatBoost / XGBoost: 0.9914
- CatBoost / LightGBM: 0.9939
- XGBoost / LightGBM: 0.9953

Best tested blend:

- 88% CatBoost
- 12% XGBoost
- 0% LightGBM
- OOF RMSE: **1075.3785**

The improvement was only about 0.15 RMSE over CatBoost alone.

**Decision:** CatBoost remains the primary model; ensemble complexity was not justified by the evidence.

---

## 8. Error Analysis

The champion's OOF predictions were analyzed by target decile, store, store × category, product frequency, and price within the high-sales tail.

### High-sales tail

- 90th-percentile cutoff: **4557.841**
- Tail observations: **682 / 6818 = 10%**
- Tail actual mean: **5818.53**
- Tail predicted mean: **3975.00**
- Tail contribution to total squared error: **40.60%**

The model systematically underpredicted the upper tail.

### Tail × price

High-tail observations were underpredicted across every price band. Mean underprediction remained roughly 1962 at the lower tail-price band and 1728 at the highest band.

**Inference:** price helps identify high-sales observations but does not explain the full upper-tail variation. Simple price transformations were therefore not pursued further.

### Store and store × category

High-volume stores generated much of the error because they contain many high-sales observations. No universal store bias or store × category correction was found.

### Product frequency

Extreme errors were not monotonically related to product observation count.

**Decision:** no product-frequency correction.

---

## 9. Tail/Regime Experiment

### Hypothesis

Because the top 10% contributed 40.60% of squared error, explicitly identifying and modeling the high-sales regime might improve RMSE.

### Design

- Tail cutoff: 4557.841
- Stage 1: CatBoost tail classifier
- Stage 2: separate CatBoost regressors for tail and non-tail
- Soft probability-weighted prediction
- Same locked 5-fold CV

### Result

- Two-stage CV RMSE: **1075.1865**
- Std: **22.0163**
- Champion: **1074.9159**

**Decision: Reject.** The regime model was 0.2706 RMSE worse and less stable.

The tail is genuinely difficult, but the evidence does not support a separately modeled regime with the available features.

---

## 10. Final Frozen Model

```text
CatBoostRegressor
boosting_type="Ordered"
iterations=300
learning_rate=0.025
depth=4
l2_leaf_reg=1
random_strength=1.0
loss_function="RMSE"
random_seed=42
```

Categorical features:

```text
product_code
fat_content
product_category_clean
store_code
store_size
store_location_tier
store_format
```

`id` is excluded.

### Final confirmation

The frozen model was retrained from scratch across the exact locked folds:

```text
Fold 1    1065.990810
Fold 2    1053.887357
Fold 3    1085.198997
Fold 4    1107.592529
Fold 5    1061.909675
```

Mean CV RMSE:

**1074.915874**

This exactly reproduced the frozen benchmark.

The final confirmation fold-level standard deviation was **21.590061**.


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

# Preserve the cleaned category representation established during EDA
train["product_category_clean"] = (
    train["product_category"].str.strip().str.lower()
)

test["product_category_clean"] = (
    test["product_category"].str.strip().str.lower()
)

X = train.drop(columns="total_sales")
y = train["total_sales"]

# ID is an identifier, not a predictive feature
X = X.drop(columns="id")

# Locked CV folds — reused for every experiment
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

folds = list(kf.split(X, y))

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Number of folds:", len(folds))

for i, (train_idx, valid_idx) in enumerate(folds, start=1):
    print(
        f"Fold {i}: "
        f"train={len(train_idx)}, "
        f"validation={len(valid_idx)}"
    )

X shape: (6818, 12)
y shape: (6818,)
Number of folds: 5
Fold 1: train=5454, validation=1364
Fold 2: train=5454, validation=1364
Fold 3: train=5454, validation=1364
Fold 4: train=5455, validation=1363
Fold 5: train=5455, validation=1363


In [2]:
# Naive mean baseline

from sklearn.metrics import mean_squared_error
import numpy as np

fold_rmse = []

for fold, (train_idx, valid_idx) in enumerate(folds, start=1):
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    mean_prediction = np.full(
        len(y_valid),
        y_train.mean()
    )

    rmse = np.sqrt(
        mean_squared_error(y_valid, mean_prediction)
    )

    fold_rmse.append(rmse)

    print(f"Fold {fold}: RMSE = {rmse:.4f}")

print("\nMean CV RMSE:", np.mean(fold_rmse))
print("Std CV RMSE:", np.std(fold_rmse))

Fold 1: RMSE = 1716.8663
Fold 2: RMSE = 1651.8001
Fold 3: RMSE = 1685.7045
Fold 4: RMSE = 1722.1846
Fold 5: RMSE = 1712.0648

Mean CV RMSE: 1697.7240714332984
Std CV RMSE: 26.168240707436414


In [3]:
# Ridge baseline — raw target

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
import numpy as np

numeric_features = [
    "product_weight_kg",
    "product_price",
    "shelf_visibility",
    "store_age_years",
]

categorical_features = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format",
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numeric_features,
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_features,
        ),
    ]
)

ridge_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Ridge(alpha=10.0)),
])

ridge_fold_rmse = []

for fold, (train_idx, valid_idx) in enumerate(folds, start=1):
    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    ridge_model.fit(X_train, y_train)

    predictions = ridge_model.predict(X_valid)

    rmse = np.sqrt(
        mean_squared_error(y_valid, predictions)
    )

    ridge_fold_rmse.append(rmse)

    print(f"Fold {fold}: RMSE = {rmse:.4f}")

print("\nRidge Mean CV RMSE:", np.mean(ridge_fold_rmse))
print("Ridge Std CV RMSE:", np.std(ridge_fold_rmse))
print(
    "Improvement vs naive:",
    np.mean(fold_rmse) - np.mean(ridge_fold_rmse)
)

Fold 1: RMSE = 1131.3343
Fold 2: RMSE = 1099.5975
Fold 3: RMSE = 1149.2786
Fold 4: RMSE = 1177.3026
Fold 5: RMSE = 1125.6992

Ridge Mean CV RMSE: 1136.6424317149786
Ridge Std CV RMSE: 25.819275144004955
Improvement vs naive: 561.0816397183198


In [4]:
# Random Forest baseline — raw target

from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_error
import numpy as np

numeric_features = [
    "product_weight_kg",
    "product_price",
    "shelf_visibility",
    "store_age_years",
]

categorical_features = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format",
]

preprocessor_rf = ColumnTransformer(
    transformers=[
        (
            "num",
            SimpleImputer(strategy="median"),
            numeric_features,
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_features,
        ),
    ]
)

rf_model = Pipeline([
    ("preprocessor", preprocessor_rf),
    (
        "model",
        RandomForestRegressor(
            n_estimators=400,
            max_depth=None,
            min_samples_leaf=2,
            max_features="sqrt",
            random_state=42,
            n_jobs=-1,
        ),
    ),
])

rf_fold_rmse = []

for fold, (train_idx, valid_idx) in enumerate(folds, start=1):
    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    rf_model.fit(X_train, y_train)

    predictions = rf_model.predict(X_valid)

    rmse = np.sqrt(
        mean_squared_error(y_valid, predictions)
    )

    rf_fold_rmse.append(rmse)

    print(f"Fold {fold}: RMSE = {rmse:.4f}")

print("\nRandom Forest Mean CV RMSE:", np.mean(rf_fold_rmse))
print("Random Forest Std CV RMSE:", np.std(rf_fold_rmse))
print(
    "Improvement vs Ridge:",
    np.mean(ridge_fold_rmse) - np.mean(rf_fold_rmse)
)

Fold 1: RMSE = 1275.5630
Fold 2: RMSE = 1234.5164
Fold 3: RMSE = 1284.5309
Fold 4: RMSE = 1309.2630
Fold 5: RMSE = 1277.7085

Random Forest Mean CV RMSE: 1276.3163544375893
Random Forest Std CV RMSE: 24.09438562737886
Improvement vs Ridge: -139.67392272261077


In [5]:
# HistGradientBoosting baseline — raw target

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import mean_squared_error
import numpy as np

numeric_features = [
    "product_weight_kg",
    "product_price",
    "shelf_visibility",
    "store_age_years",
]

categorical_features = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format",
]

preprocessor_hgb = ColumnTransformer(
    transformers=[
        (
            "num",
            SimpleImputer(strategy="median"),
            numeric_features,
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                (
                    "encoder",
                    OrdinalEncoder(
                        handle_unknown="use_encoded_value",
                        unknown_value=-1,
                    ),
                ),
            ]),
            categorical_features,
        ),
    ]
)

hgb_model = Pipeline([
    ("preprocessor", preprocessor_hgb),
    (
        "model",
        HistGradientBoostingRegressor(
            max_iter=300,
            learning_rate=0.05,
            max_leaf_nodes=31,
            l2_regularization=1.0,
            random_state=42,
        ),
    ),
])

hgb_fold_rmse = []

for fold, (train_idx, valid_idx) in enumerate(folds, start=1):
    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    hgb_model.fit(X_train, y_train)

    predictions = hgb_model.predict(X_valid)

    rmse = np.sqrt(
        mean_squared_error(y_valid, predictions)
    )

    hgb_fold_rmse.append(rmse)

    print(f"Fold {fold}: RMSE = {rmse:.4f}")

print("\nHGB Mean CV RMSE:", np.mean(hgb_fold_rmse))
print("HGB Std CV RMSE:", np.std(hgb_fold_rmse))
print(
    "Improvement vs Ridge:",
    np.mean(ridge_fold_rmse) - np.mean(hgb_fold_rmse)
)

Fold 1: RMSE = 1129.7642
Fold 2: RMSE = 1093.5782
Fold 3: RMSE = 1129.2000
Fold 4: RMSE = 1145.9154
Fold 5: RMSE = 1129.4215

HGB Mean CV RMSE: 1125.5758475672874
HGB Std CV RMSE: 17.222139299187553
Improvement vs Ridge: 11.066584147691174


In [6]:
# XGBoost baseline — raw target

from xgboost import XGBRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_error
import numpy as np

numeric_features = [
    "product_weight_kg",
    "product_price",
    "shelf_visibility",
    "store_age_years",
]

categorical_features = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format",
]

preprocessor_xgb = ColumnTransformer(
    transformers=[
        (
            "num",
            SimpleImputer(strategy="median"),
            numeric_features,
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_features,
        ),
    ]
)

xgb_model = Pipeline([
    ("preprocessor", preprocessor_xgb),
    (
        "model",
        XGBRegressor(
            n_estimators=500,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            eval_metric="rmse",
            random_state=42,
            n_jobs=-1,
        ),
    ),
])

xgb_fold_rmse = []

for fold, (train_idx, valid_idx) in enumerate(folds, start=1):
    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    xgb_model.fit(X_train, y_train)

    predictions = xgb_model.predict(X_valid)

    rmse = np.sqrt(
        mean_squared_error(y_valid, predictions)
    )

    xgb_fold_rmse.append(rmse)

    print(f"Fold {fold}: RMSE = {rmse:.4f}")

print("\nXGBoost Mean CV RMSE:", np.mean(xgb_fold_rmse))
print("XGBoost Std CV RMSE:", np.std(xgb_fold_rmse))
print(
    "Improvement vs HGB:",
    np.mean(hgb_fold_rmse) - np.mean(xgb_fold_rmse)
)

Fold 1: RMSE = 1117.8845
Fold 2: RMSE = 1073.6605
Fold 3: RMSE = 1117.7821
Fold 4: RMSE = 1140.9445
Fold 5: RMSE = 1096.4108

XGBoost Mean CV RMSE: 1109.33645585329
XGBoost Std CV RMSE: 22.730203827142514
Improvement vs HGB: 16.239391713997293


In [7]:
# LightGBM baseline — raw target

from lightgbm import LGBMRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_error
import numpy as np

preprocessor_lgb = ColumnTransformer(
    transformers=[
        (
            "num",
            SimpleImputer(strategy="median"),
            numeric_features,
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_features,
        ),
    ]
)

lgb_model = Pipeline([
    ("preprocessor", preprocessor_lgb),
    (
        "model",
        LGBMRegressor(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=-1,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            random_state=42,
            n_jobs=-1,
            verbosity=-1,
        ),
    ),
])

lgb_fold_rmse = []

for fold, (train_idx, valid_idx) in enumerate(folds, start=1):
    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    lgb_model.fit(X_train, y_train)

    predictions = lgb_model.predict(X_valid)

    rmse = np.sqrt(
        mean_squared_error(y_valid, predictions)
    )

    lgb_fold_rmse.append(rmse)

    print(f"Fold {fold}: RMSE = {rmse:.4f}")

print("\nLightGBM Mean CV RMSE:", np.mean(lgb_fold_rmse))
print("LightGBM Std CV RMSE:", np.std(lgb_fold_rmse))
print(
    "Improvement vs XGBoost:",
    np.mean(xgb_fold_rmse) - np.mean(lgb_fold_rmse)
)

Fold 1: RMSE = 1139.6191
Fold 2: RMSE = 1091.7394
Fold 3: RMSE = 1150.9243
Fold 4: RMSE = 1156.4491
Fold 5: RMSE = 1129.9617

LightGBM Mean CV RMSE: 1133.7387157602159
LightGBM Std CV RMSE: 22.908053432663543
Improvement vs XGBoost: -24.402259906925792


In [8]:
# CatBoost baseline — raw target

from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

X_cat = X.drop(columns="product_category").copy()
y_cat = y.copy()

cat_features = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format",
]

cat_fold_rmse = []

for fold, (train_idx, valid_idx) in enumerate(folds, start=1):
    X_train = X_cat.iloc[train_idx].copy()
    X_valid = X_cat.iloc[valid_idx].copy()
    y_train = y_cat.iloc[train_idx]
    y_valid = y_cat.iloc[valid_idx]

    for col in cat_features:
        X_train[col] = X_train[col].fillna("Missing").astype(str)
        X_valid[col] = X_valid[col].fillna("Missing").astype(str)

    cat_indices = [
        X_train.columns.get_loc(col)
        for col in cat_features
    ]

    model = CatBoostRegressor(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        thread_count=-1,
    )

    model.fit(X_train, y_train, cat_features=cat_indices)
    predictions = model.predict(X_valid)

    rmse = np.sqrt(mean_squared_error(y_valid, predictions))
    cat_fold_rmse.append(rmse)
    print(f"Fold {fold}: RMSE = {rmse:.4f}")

cat_mean_rmse = np.mean(cat_fold_rmse)
cat_std_rmse = np.std(cat_fold_rmse)

print("\nCatBoost Mean CV RMSE:", cat_mean_rmse)
print("CatBoost Std CV RMSE:", cat_std_rmse)
print(
    "Improvement vs XGBoost:",
    np.mean(xgb_fold_rmse) - cat_mean_rmse,
)

Fold 1: RMSE = 1076.0201
Fold 2: RMSE = 1057.1375
Fold 3: RMSE = 1093.5818
Fold 4: RMSE = 1119.1931
Fold 5: RMSE = 1083.8328

CatBoost Mean CV RMSE: 1085.9530521487982
CatBoost Std CV RMSE: 20.47727697063577
Improvement vs XGBoost: 23.38340370449191


In [9]:
# CatBoost baseline — log1p target

from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error
import numpy as np


X_cat_log = train.drop(
    columns=["total_sales", "id", "product_category"]
).copy()

y_log = np.log1p(train["total_sales"])

# Same categorical features as raw-target CatBoost
cat_features = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format",
]


cat_log_fold_rmse = []

for fold, (train_idx, valid_idx) in enumerate(folds, start=1):

    X_train = X_cat_log.iloc[train_idx].copy()
    X_valid = X_cat_log.iloc[valid_idx].copy()

    y_train = y_log.iloc[train_idx]
    y_valid_original = train["total_sales"].iloc[valid_idx]

    for col in cat_features:
        X_train[col] = (
            X_train[col]
            .fillna("__MISSING__")
            .astype(str)
        )

        X_valid[col] = (
            X_valid[col]
            .fillna("__MISSING__")
            .astype(str)
        )

    cat_indices = [
        X_train.columns.get_loc(col)
        for col in cat_features
    ]

    
    model = CatBoostRegressor(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        thread_count=-1,
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_indices,
    )

    
    log_predictions = model.predict(X_valid)

    predictions = np.expm1(log_predictions)


    rmse = np.sqrt(
        mean_squared_error(
            y_valid_original,
            predictions
        )
    )

    cat_log_fold_rmse.append(rmse)

    print(f"Fold {fold}: RMSE = {rmse:.4f}")



cat_log_mean_rmse = np.mean(cat_log_fold_rmse)
cat_log_std_rmse = np.std(cat_log_fold_rmse)

print("\nCatBoost Log Target Mean CV RMSE:", cat_log_mean_rmse)
print("CatBoost Log Target Std CV RMSE:", cat_log_std_rmse)

print(
    "Improvement vs raw-target CatBoost:",
    np.mean(cat_fold_rmse) - cat_log_mean_rmse
)

Fold 1: RMSE = 1122.7506
Fold 2: RMSE = 1069.1057
Fold 3: RMSE = 1117.7689
Fold 4: RMSE = 1142.4232
Fold 5: RMSE = 1108.9599

CatBoost Log Target Mean CV RMSE: 1112.2016602094416
CatBoost Log Target Std CV RMSE: 24.180100691870457
Improvement vs raw-target CatBoost: -26.24860806064339


In [10]:
# Phase 4 — Feature Engineering Experiment 1

from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

X_base = train.drop(
    columns=["total_sales", "id", "product_category"]
).copy()

y_base = train["total_sales"].copy()

cat_features = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format",
]

price_ratio_fold_rmse = []

for fold, (train_idx, valid_idx) in enumerate(folds, start=1):

    X_train = X_base.iloc[train_idx].copy()
    X_valid = X_base.iloc[valid_idx].copy()

    y_train = y_base.iloc[train_idx]
    y_valid = y_base.iloc[valid_idx]



    product_median_price = (
        X_train
        .groupby("product_code")["product_price"]
        .median()
    )

    global_median_price = X_train["product_price"].median()



    train_product_median = (
        X_train["product_code"]
        .map(product_median_price)
        .fillna(global_median_price)
    )

    valid_product_median = (
        X_valid["product_code"]
        .map(product_median_price)
        .fillna(global_median_price)
    )



    X_train["product_relative_price"] = (
        X_train["product_price"] /
        train_product_median
    )

    X_valid["product_relative_price"] = (
        X_valid["product_price"] /
        valid_product_median
    )



    for col in cat_features:
        X_train[col] = (
            X_train[col]
            .fillna("__MISSING__")
            .astype(str)
        )

        X_valid[col] = (
            X_valid[col]
            .fillna("__MISSING__")
            .astype(str)
        )

    cat_indices = [
        X_train.columns.get_loc(col)
        for col in cat_features
    ]



    model = CatBoostRegressor(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        thread_count=-1,
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_indices,
    )

    predictions = model.predict(X_valid)

    rmse = np.sqrt(
        mean_squared_error(y_valid, predictions)
    )

    price_ratio_fold_rmse.append(rmse)

    print(f"Fold {fold}: RMSE = {rmse:.4f}")


price_ratio_mean_rmse = np.mean(price_ratio_fold_rmse)
price_ratio_std_rmse = np.std(price_ratio_fold_rmse)

print("\nProduct-relative price CatBoost RMSE:", price_ratio_mean_rmse)
print("Product-relative price CatBoost Std:", price_ratio_std_rmse)

print(
    "Change vs current champion:",
    np.mean(cat_fold_rmse) - price_ratio_mean_rmse
)

Fold 1: RMSE = 1072.9828
Fold 2: RMSE = 1060.7495
Fold 3: RMSE = 1097.4152
Fold 4: RMSE = 1112.3106
Fold 5: RMSE = 1081.7160

Product-relative price CatBoost RMSE: 1085.0348410841539
Product-relative price CatBoost Std: 18.12888945212419
Change vs current champion: 0.9182110646443107


In [11]:
X_base = train.drop(
    columns=["total_sales", "id", "product_category"]
).copy()

y_base = train["total_sales"].copy()

cat_features = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format",
]

weight_flag_fold_rmse = []

for fold, (train_idx, valid_idx) in enumerate(folds, start=1):

    X_train = X_base.iloc[train_idx].copy()
    X_valid = X_base.iloc[valid_idx].copy()

    y_train = y_base.iloc[train_idx]
    y_valid = y_base.iloc[valid_idx]

    # One hypothesis: explicitly expose whether product weight is missing.
    X_train["weight_missing_flag"] = (
        X_train["product_weight_kg"].isna().astype(int)
    )

    X_valid["weight_missing_flag"] = (
        X_valid["product_weight_kg"].isna().astype(int)
    )

    # CatBoost categorical handling
    for col in cat_features:
        X_train[col] = X_train[col].fillna("__MISSING__").astype(str)
        X_valid[col] = X_valid[col].fillna("__MISSING__").astype(str)

    cat_indices = [
        X_train.columns.get_loc(col)
        for col in cat_features
    ]

    model = CatBoostRegressor(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        thread_count=-1,
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_indices,
    )

    predictions = model.predict(X_valid)

    rmse = np.sqrt(mean_squared_error(y_valid, predictions))
    weight_flag_fold_rmse.append(rmse)

    print(f"Fold {fold}: RMSE = {rmse:.4f}")

weight_flag_mean_rmse = np.mean(weight_flag_fold_rmse)
weight_flag_std_rmse = np.std(weight_flag_fold_rmse)

print("\nWeight-missing flag CatBoost RMSE:", weight_flag_mean_rmse)
print("Weight-missing flag CatBoost Std:", weight_flag_std_rmse)
print(
    "Change vs current champion:",
    np.mean(cat_fold_rmse) - weight_flag_mean_rmse
)

Fold 1: RMSE = 1081.5458
Fold 2: RMSE = 1058.7222
Fold 3: RMSE = 1096.8877
Fold 4: RMSE = 1117.3637
Fold 5: RMSE = 1079.9781

Weight-missing flag CatBoost RMSE: 1086.8994986917992
Weight-missing flag CatBoost Std: 19.4851922289055
Change vs current champion: -0.9464465430010023


In [12]:
X_base = train.drop(
    columns=["total_sales", "id", "product_category"]
).copy()

y_base = train["total_sales"].copy()

cat_features = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format",
]

product_weight_fold_rmse = []

for fold, (train_idx, valid_idx) in enumerate(folds, start=1):

    X_train = X_base.iloc[train_idx].copy()
    X_valid = X_base.iloc[valid_idx].copy()

    y_train = y_base.iloc[train_idx]
    y_valid = y_base.iloc[valid_idx]

    # Product's typical weight learned from TRAINING FOLD ONLY
    product_median_weight = (
        X_train.groupby("product_code")["product_weight_kg"].median()
    )

    global_median_weight = X_train["product_weight_kg"].median()

    train_product_weight = (
        X_train["product_code"]
        .map(product_median_weight)
        .fillna(global_median_weight)
    )

    valid_product_weight = (
        X_valid["product_code"]
        .map(product_median_weight)
        .fillna(global_median_weight)
    )

    # Add leakage-safe product-level typical weight
    X_train["product_median_weight"] = train_product_weight
    X_valid["product_median_weight"] = valid_product_weight

    # CatBoost categorical handling
    for col in cat_features:
        X_train[col] = X_train[col].fillna("__MISSING__").astype(str)
        X_valid[col] = X_valid[col].fillna("__MISSING__").astype(str)

    cat_indices = [
        X_train.columns.get_loc(col)
        for col in cat_features
    ]

    model = CatBoostRegressor(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        thread_count=-1,
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_indices,
    )

    predictions = model.predict(X_valid)

    rmse = np.sqrt(mean_squared_error(y_valid, predictions))
    product_weight_fold_rmse.append(rmse)

    print(f"Fold {fold}: RMSE = {rmse:.4f}")

product_weight_mean_rmse = np.mean(product_weight_fold_rmse)
product_weight_std_rmse = np.std(product_weight_fold_rmse)

print("\nProduct-median weight CatBoost RMSE:", product_weight_mean_rmse)
print("Product-median weight CatBoost Std:", product_weight_std_rmse)
print(
    "Change vs current champion:",
    np.mean(cat_fold_rmse) - product_weight_mean_rmse
)

Fold 1: RMSE = 1080.6905
Fold 2: RMSE = 1056.2755
Fold 3: RMSE = 1096.9270
Fold 4: RMSE = 1115.5934
Fold 5: RMSE = 1083.1285

Product-median weight CatBoost RMSE: 1086.522974645347
Product-median weight CatBoost Std: 19.564155266291344
Change vs current champion: -0.5699224965487701


In [13]:
X_base = train.drop(
    columns=["total_sales", "id", "product_category"]
).copy()

y_base = train["total_sales"].copy()

cat_features = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format",
    "store_category",
]

store_category_fold_rmse = []

for fold, (train_idx, valid_idx) in enumerate(folds, start=1):

    X_train = X_base.iloc[train_idx].copy()
    X_valid = X_base.iloc[valid_idx].copy()

    y_train = y_base.iloc[train_idx]
    y_valid = y_base.iloc[valid_idx]

    # Explicit store × category interaction
    X_train["store_category"] = (
        X_train["store_code"].astype(str)
        + "__"
        + X_train["product_category_clean"].astype(str)
    )

    X_valid["store_category"] = (
        X_valid["store_code"].astype(str)
        + "__"
        + X_valid["product_category_clean"].astype(str)
    )

    # CatBoost categorical handling
    for col in cat_features:
        X_train[col] = X_train[col].fillna("__MISSING__").astype(str)
        X_valid[col] = X_valid[col].fillna("__MISSING__").astype(str)

    cat_indices = [
        X_train.columns.get_loc(col)
        for col in cat_features
    ]

    model = CatBoostRegressor(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        thread_count=-1,
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_indices,
    )

    predictions = model.predict(X_valid)

    rmse = np.sqrt(mean_squared_error(y_valid, predictions))
    store_category_fold_rmse.append(rmse)

    print(f"Fold {fold}: RMSE = {rmse:.4f}")

store_category_mean_rmse = np.mean(store_category_fold_rmse)
store_category_std_rmse = np.std(store_category_fold_rmse)

print("\nStore x category CatBoost RMSE:", store_category_mean_rmse)
print("Store x category CatBoost Std:", store_category_std_rmse)
print(
    "Change vs current champion:",
    np.mean(cat_fold_rmse) - store_category_mean_rmse
)

Fold 1: RMSE = 1076.5959
Fold 2: RMSE = 1058.2771
Fold 3: RMSE = 1098.0173
Fold 4: RMSE = 1117.4426
Fold 5: RMSE = 1083.6420

Store x category CatBoost RMSE: 1086.7949722436883
Store x category CatBoost Std: 19.962225394272565
Change vs current champion: -0.8419200948901562


In [14]:
depth_results = {}

X_base = train.drop(
    columns=["total_sales", "id", "product_category"]
).copy()

y_base = train["total_sales"].copy()

cat_features = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format",
]

for depth in [4, 5, 7, 8]:

    fold_rmse = []

    print(f"\nCatBoost depth={depth}:")

    for fold, (train_idx, valid_idx) in enumerate(folds, start=1):

        X_train = X_base.iloc[train_idx].copy()
        X_valid = X_base.iloc[valid_idx].copy()

        y_train = y_base.iloc[train_idx]
        y_valid = y_base.iloc[valid_idx]

        for col in cat_features:
            X_train[col] = (
                X_train[col]
                .fillna("Missing")
                .astype(str)
            )
            X_valid[col] = (
                X_valid[col]
                .fillna("Missing")
                .astype(str)
            )

        cat_indices = [
            X_train.columns.get_loc(col)
            for col in cat_features
        ]

        model = CatBoostRegressor(
            iterations=500,
            learning_rate=0.05,
            depth=depth,
            loss_function="RMSE",
            random_seed=42,
            verbose=False,
            thread_count=-1,
        )

        model.fit(
            X_train,
            y_train,
            cat_features=cat_indices,
        )

        predictions = model.predict(X_valid)

        rmse = np.sqrt(
            mean_squared_error(y_valid, predictions)
        )

        fold_rmse.append(rmse)

        print(f"Fold {fold}: RMSE = {rmse:.4f}")

    mean_rmse = np.mean(fold_rmse)
    std_rmse = np.std(fold_rmse)

    depth_results[depth] = {
        "fold_rmse": fold_rmse,
        "mean_rmse": mean_rmse,
        "std_rmse": std_rmse,
    }

    print(f"Mean RMSE: {mean_rmse:.4f}")
    print(f"Std RMSE:  {std_rmse:.4f}")

print("\nDepth Summary")

for depth, result in depth_results.items():
    print(
        f"Depth {depth}: "
        f"RMSE = {result['mean_rmse']:.4f}, "
        f"Std = {result['std_rmse']:.4f}, "
        f"Change vs depth 6 = "
        f"{np.mean(cat_fold_rmse) - result['mean_rmse']:.4f}"
    )


CatBoost depth=4:
Fold 1: RMSE = 1070.6806
Fold 2: RMSE = 1052.7069
Fold 3: RMSE = 1094.0811
Fold 4: RMSE = 1113.1884
Fold 5: RMSE = 1072.2703
Mean RMSE: 1080.5855
Std RMSE:  20.9263

CatBoost depth=5:
Fold 1: RMSE = 1076.7280
Fold 2: RMSE = 1055.2974
Fold 3: RMSE = 1093.7923
Fold 4: RMSE = 1114.0312
Fold 5: RMSE = 1080.6770
Mean RMSE: 1084.1052
Std RMSE:  19.4189

CatBoost depth=7:
Fold 1: RMSE = 1079.8642
Fold 2: RMSE = 1059.5618
Fold 3: RMSE = 1105.5791
Fold 4: RMSE = 1123.5004
Fold 5: RMSE = 1080.8933
Mean RMSE: 1089.8798
Std RMSE:  22.2579

CatBoost depth=8:
Fold 1: RMSE = 1084.8218
Fold 2: RMSE = 1055.2788
Fold 3: RMSE = 1105.4645
Fold 4: RMSE = 1124.0274
Fold 5: RMSE = 1087.9165
Mean RMSE: 1091.5018
Std RMSE:  22.9017

Depth Summary
Depth 4: RMSE = 1080.5855, Std = 20.9263, Change vs depth 6 = 5.3676
Depth 5: RMSE = 1084.1052, Std = 19.4189, Change vs depth 6 = 1.8479
Depth 7: RMSE = 1089.8798, Std = 22.2579, Change vs depth 6 = -3.9267
Depth 8: RMSE = 1091.5018, Std = 22.9017,

In [15]:
iteration_results = {}

X_base = train.drop(
    columns=["total_sales", "id", "product_category"]
).copy()

y_base = train["total_sales"].copy()

cat_features = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format",
]

for iterations in [300, 500, 800, 1000]:

    fold_rmse = []

    print(f"\nCatBoost iterations={iterations}, depth=4: ")

    for fold, (train_idx, valid_idx) in enumerate(folds, start=1):

        X_train = X_base.iloc[train_idx].copy()
        X_valid = X_base.iloc[valid_idx].copy()

        y_train = y_base.iloc[train_idx]
        y_valid = y_base.iloc[valid_idx]

        for col in cat_features:
            X_train[col] = (
                X_train[col]
                .fillna("Missing")
                .astype(str)
            )
            X_valid[col] = (
                X_valid[col]
                .fillna("Missing")
                .astype(str)
            )

        cat_indices = [
            X_train.columns.get_loc(col)
            for col in cat_features
        ]

        model = CatBoostRegressor(
            iterations=iterations,
            learning_rate=0.05,
            depth=4,
            loss_function="RMSE",
            random_seed=42,
            verbose=False,
            thread_count=-1,
        )

        model.fit(
            X_train,
            y_train,
            cat_features=cat_indices,
        )

        predictions = model.predict(X_valid)

        rmse = np.sqrt(
            mean_squared_error(y_valid, predictions)
        )

        fold_rmse.append(rmse)

        print(f"Fold {fold}: RMSE = {rmse:.4f}")

    mean_rmse = np.mean(fold_rmse)
    std_rmse = np.std(fold_rmse)

    iteration_results[iterations] = {
        "fold_rmse": fold_rmse,
        "mean_rmse": mean_rmse,
        "std_rmse": std_rmse,
    }

    print(f"Mean RMSE: {mean_rmse:.4f}")
    print(f"Std RMSE:  {std_rmse:.4f}")

print("\n Iteration Summary ")

for iterations, result in iteration_results.items():
    print(
        f"{iterations} iterations: "
        f"RMSE = {result['mean_rmse']:.4f}, "
        f"Std = {result['std_rmse']:.4f}, "
        f"Change vs depth-4 champion = "
        f"{1080.5855 - result['mean_rmse']:.4f}"
    )


CatBoost iterations=300, depth=4: 
Fold 1: RMSE = 1065.9036
Fold 2: RMSE = 1049.3347
Fold 3: RMSE = 1089.6209
Fold 4: RMSE = 1109.3491
Fold 5: RMSE = 1064.7241
Mean RMSE: 1075.7865
Std RMSE:  21.1546

CatBoost iterations=500, depth=4: 
Fold 1: RMSE = 1070.6806
Fold 2: RMSE = 1052.7069
Fold 3: RMSE = 1094.0811
Fold 4: RMSE = 1113.1884
Fold 5: RMSE = 1072.2703
Mean RMSE: 1080.5855
Std RMSE:  20.9263

CatBoost iterations=800, depth=4: 
Fold 1: RMSE = 1079.1846
Fold 2: RMSE = 1055.8536
Fold 3: RMSE = 1097.3718
Fold 4: RMSE = 1118.7869
Fold 5: RMSE = 1081.3266
Mean RMSE: 1086.5047
Std RMSE:  20.8812

CatBoost iterations=1000, depth=4: 
Fold 1: RMSE = 1082.0348
Fold 2: RMSE = 1057.6134
Fold 3: RMSE = 1100.8600
Fold 4: RMSE = 1122.6104
Fold 5: RMSE = 1085.5882
Mean RMSE: 1089.7414
Std RMSE:  21.5074

 Iteration Summary 
300 iterations: RMSE = 1075.7865, Std = 21.1546, Change vs depth-4 champion = 4.7990
500 iterations: RMSE = 1080.5855, Std = 20.9263, Change vs depth-4 champion = 0.0000
800 

In [16]:
# Test different CatBoost learning rates while keeping depth=4 and iterations=300 fixed.

learning_rate_results = {}

X_base = train.drop(
    columns=["total_sales", "id", "product_category"]
).copy()

y_base = train["total_sales"].copy()

cat_features = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format",
]

for learning_rate in [0.02, 0.03, 0.05, 0.08, 0.10]:

    fold_rmse = []

    print(
        f"\n CatBoost "
        f"learning_rate={learning_rate}, "
        f"depth=4, iterations=300 :"
    )

    for fold, (train_idx, valid_idx) in enumerate(folds, start=1):

        X_train = X_base.iloc[train_idx].copy()
        X_valid = X_base.iloc[valid_idx].copy()

        y_train = y_base.iloc[train_idx]
        y_valid = y_base.iloc[valid_idx]

        for col in cat_features:
            X_train[col] = (
                X_train[col]
                .fillna("Missing")
                .astype(str)
            )
            X_valid[col] = (
                X_valid[col]
                .fillna("Missing")
                .astype(str)
            )

        cat_indices = [
            X_train.columns.get_loc(col)
            for col in cat_features
        ]

        model = CatBoostRegressor(
            iterations=300,
            learning_rate=learning_rate,
            depth=4,
            loss_function="RMSE",
            random_seed=42,
            verbose=False,
            thread_count=-1,
        )

        model.fit(
            X_train,
            y_train,
            cat_features=cat_indices,
        )

        predictions = model.predict(X_valid)

        rmse = np.sqrt(
            mean_squared_error(y_valid, predictions)
        )

        fold_rmse.append(rmse)

        print(f"Fold {fold}: RMSE = {rmse:.4f}")

    mean_rmse = np.mean(fold_rmse)
    std_rmse = np.std(fold_rmse)

    learning_rate_results[learning_rate] = {
        "fold_rmse": fold_rmse,
        "mean_rmse": mean_rmse,
        "std_rmse": std_rmse,
    }

    print(f"Mean RMSE: {mean_rmse:.4f}")
    print(f"Std RMSE:  {std_rmse:.4f}")

print("\n learning Rate Summary ")

for learning_rate, result in learning_rate_results.items():
    improvement = 1075.7865 - result["mean_rmse"]

    print(
        f"Learning rate {learning_rate}: "
        f"RMSE = {result['mean_rmse']:.4f}, "
        f"Std = {result['std_rmse']:.4f}, "
        f"RMSE improvement vs champion = {improvement:.4f}"
    )


 CatBoost learning_rate=0.02, depth=4, iterations=300 :
Fold 1: RMSE = 1067.4140
Fold 2: RMSE = 1050.8510
Fold 3: RMSE = 1086.8734
Fold 4: RMSE = 1106.8100
Fold 5: RMSE = 1065.5032
Mean RMSE: 1075.4903
Std RMSE:  19.4039

 CatBoost learning_rate=0.03, depth=4, iterations=300 :
Fold 1: RMSE = 1065.3625


KeyboardInterrupt: 

In [ ]:
# Refine the learning-rate search around the current best region while keeping depth=4 and iterations=300 fixed.

learning_rate_refined_results = {}

for learning_rate in [0.01, 0.015, 0.02, 0.025, 0.03]:

    fold_rmse = []

    print(
        f"\nCatBoost "
        f"learning_rate={learning_rate}, "
        f"depth=4, iterations=300"
    )

    for fold, (train_idx, valid_idx) in enumerate(folds, start=1):

        X_train = X_base.iloc[train_idx].copy()
        X_valid = X_base.iloc[valid_idx].copy()

        y_train = y_base.iloc[train_idx]
        y_valid = y_base.iloc[valid_idx]

        for col in cat_features:
            X_train[col] = (
                X_train[col]
                .fillna("Missing")
                .astype(str)
            )
            X_valid[col] = (
                X_valid[col]
                .fillna("Missing")
                .astype(str)
            )

        cat_indices = [
            X_train.columns.get_loc(col)
            for col in cat_features
        ]

        model = CatBoostRegressor(
            iterations=300,
            learning_rate=learning_rate,
            depth=4,
            loss_function="RMSE",
            random_seed=42,
            verbose=False,
            thread_count=-1,
        )

        model.fit(
            X_train,
            y_train,
            cat_features=cat_indices,
        )

        predictions = model.predict(X_valid)

        rmse = np.sqrt(
            mean_squared_error(y_valid, predictions)
        )

        fold_rmse.append(rmse)

        print(f"Fold {fold}: RMSE = {rmse:.4f}")

    mean_rmse = np.mean(fold_rmse)
    std_rmse = np.std(fold_rmse)

    learning_rate_refined_results[learning_rate] = {
        "fold_rmse": fold_rmse,
        "mean_rmse": mean_rmse,
        "std_rmse": std_rmse,
    }

    print(f"Mean RMSE: {mean_rmse:.4f}")
    print(f"Std RMSE:  {std_rmse:.4f}")

print("\n Refined Learning Rate Summary")

for learning_rate, result in learning_rate_refined_results.items():

    improvement = 1075.7865 - result["mean_rmse"]

    print(
        f"Learning rate {learning_rate}: "
        f"RMSE = {result['mean_rmse']:.4f}, "
        f"Std = {result['std_rmse']:.4f}, "
        f"RMSE improvement vs previous champion = {improvement:.4f}"
    )


===== CatBoost learning_rate=0.01, depth=4, iterations=300 =====
Fold 1: RMSE = 1080.3234
Fold 2: RMSE = 1058.4938
Fold 3: RMSE = 1092.4838
Fold 4: RMSE = 1116.1349
Fold 5: RMSE = 1075.5455
Mean RMSE: 1084.5963
Std RMSE:  19.1750

===== CatBoost learning_rate=0.015, depth=4, iterations=300 =====
Fold 1: RMSE = 1067.9981
Fold 2: RMSE = 1052.2464
Fold 3: RMSE = 1086.7534
Fold 4: RMSE = 1107.5217
Fold 5: RMSE = 1067.6385
Mean RMSE: 1076.4316
Std RMSE:  19.0079

===== CatBoost learning_rate=0.02, depth=4, iterations=300 =====
Fold 1: RMSE = 1067.4140
Fold 2: RMSE = 1050.8510
Fold 3: RMSE = 1086.8734
Fold 4: RMSE = 1106.8100
Fold 5: RMSE = 1065.5032
Mean RMSE: 1075.4903
Std RMSE:  19.4039

===== CatBoost learning_rate=0.025, depth=4, iterations=300 =====
Fold 1: RMSE = 1066.5354
Fold 2: RMSE = 1051.4447
Fold 3: RMSE = 1086.9416
Fold 4: RMSE = 1106.6580
Fold 5: RMSE = 1065.1977
Mean RMSE: 1075.3555
Std RMSE:  19.3225

===== CatBoost learning_rate=0.03, depth=4, iterations=300 =====
Fold 1: 

In [ ]:
# Aim: Test whether shallower XGBoost trees improve cross-validated RMSE while keeping all other settings fixed.

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error

numeric_features = [
    "product_weight_kg",
    "shelf_visibility",
    "product_price",
    "store_age_years",
]

categorical_features = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format",
]

depth_results = []

for depth in [3, 4, 5, 6, 7]:
    fold_scores = []

    for fold, (train_idx, valid_idx) in enumerate(folds, start=1):
        X_train = X.iloc[train_idx].copy()
        X_valid = X.iloc[valid_idx].copy()
        y_train = y.iloc[train_idx]
        y_valid = y.iloc[valid_idx]

        preprocessor = ColumnTransformer(
            transformers=[
                (
                    "num",
                    SimpleImputer(strategy="median"),
                    numeric_features,
                ),
                (
                    "cat",
                    Pipeline([
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        (
                            "encoder",
                            OrdinalEncoder(
                                handle_unknown="use_encoded_value",
                                unknown_value=-1,
                            ),
                        ),
                    ]),
                    categorical_features,
                ),
            ]
        )

        model = XGBRegressor(
            n_estimators=500,
            max_depth=depth,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            eval_metric="rmse",
            random_state=42,
            n_jobs=-1,
        )

        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model),
        ])

        pipeline.fit(X_train, y_train)

        predictions = pipeline.predict(X_valid)

        rmse = np.sqrt(
            mean_squared_error(
                y_valid,
                predictions,
            )
        )

        fold_scores.append(rmse)

    depth_results.append({
        "max_depth": depth,
        "mean_rmse": np.mean(fold_scores),
        "std_rmse": np.std(fold_scores),
    })

xgb_depth_results = pd.DataFrame(depth_results)

xgb_depth_results.sort_values("mean_rmse")

,max_depth,mean_rmse,std_rmse
0,3,1096.927586,18.839946
1,4,1113.601502,19.746073
2,5,1129.460619,20.480119
3,6,1150.676587,18.500571
4,7,1160.598936,17.471380


In [ ]:
# Test whether changing XGBoost's number of boosting rounds improves CV RMSE while keeping its best tree depth and all other settings fixed.

estimator_results = []

for n_estimators in [200, 300, 400, 500, 700]:
    fold_scores = []

    for fold, (train_idx, valid_idx) in enumerate(folds, start=1):
        X_train = X.iloc[train_idx].copy()
        X_valid = X.iloc[valid_idx].copy()
        y_train = y.iloc[train_idx]
        y_valid = y.iloc[valid_idx]

        preprocessor = ColumnTransformer(
            transformers=[
                (
                    "num",
                    SimpleImputer(strategy="median"),
                    numeric_features,
                ),
                (
                    "cat",
                    Pipeline([
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        (
                            "encoder",
                            OrdinalEncoder(
                                handle_unknown="use_encoded_value",
                                unknown_value=-1,
                            ),
                        ),
                    ]),
                    categorical_features,
                ),
            ]
        )

        model = XGBRegressor(
            n_estimators=n_estimators,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            eval_metric="rmse",
            random_state=42,
            n_jobs=-1,
        )

        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model),
        ])

        pipeline.fit(X_train, y_train)

        predictions = pipeline.predict(X_valid)

        rmse = np.sqrt(
            mean_squared_error(
                y_valid,
                predictions,
            )
        )

        fold_scores.append(rmse)

    estimator_results.append({
        "n_estimators": n_estimators,
        "mean_rmse": np.mean(fold_scores),
        "std_rmse": np.std(fold_scores),
    })

xgb_estimator_results = pd.DataFrame(estimator_results)

xgb_estimator_results.sort_values("mean_rmse")

,n_estimators,mean_rmse,std_rmse
0,200,1082.360095,20.994904
1,300,1087.020402,20.828558
2,400,1092.577716,19.844216
3,500,1096.927586,18.839946
4,700,1104.836639,18.904823


In [ ]:
# Test whether LightGBM tree complexity, controlled by num_leaves, can substantially improve its cross-validated RMSE.

from lightgbm import LGBMRegressor

lgbm_leaf_results = []

for num_leaves in [7, 15, 31, 63]:
    fold_scores = []

    for fold, (train_idx, valid_idx) in enumerate(folds, start=1):
        X_train = X.iloc[train_idx].copy()
        X_valid = X.iloc[valid_idx].copy()
        y_train = y.iloc[train_idx]
        y_valid = y.iloc[valid_idx]

        preprocessor = ColumnTransformer(
            transformers=[
                (
                    "num",
                    SimpleImputer(strategy="median"),
                    numeric_features,
                ),
                (
                    "cat",
                    Pipeline([
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        (
                            "encoder",
                            OrdinalEncoder(
                                handle_unknown="use_encoded_value",
                                unknown_value=-1,
                            ),
                        ),
                    ]),
                    categorical_features,
                ),
            ]
        )

        model = LGBMRegressor(
            n_estimators=500,
            learning_rate=0.05,
            num_leaves=num_leaves,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="regression",
            random_state=42,
            n_jobs=-1,
            verbosity=-1,
        )

        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model),
        ])

        pipeline.fit(X_train, y_train)

        predictions = pipeline.predict(X_valid)

        rmse = np.sqrt(
            mean_squared_error(
                y_valid,
                predictions,
            )
        )

        fold_scores.append(rmse)

    lgbm_leaf_results.append({
        "num_leaves": num_leaves,
        "mean_rmse": np.mean(fold_scores),
        "std_rmse": np.std(fold_scores),
    })

lgbm_leaf_results = pd.DataFrame(lgbm_leaf_results)

lgbm_leaf_results.sort_values("mean_rmse")

,num_leaves,mean_rmse,std_rmse
0,7,1096.108872,18.585480
1,15,1115.622266,21.740217
2,31,1138.483697,22.593714
3,63,1165.735000,26.843626


In [ ]:
# Test whether fewer LightGBM boosting rounds improve generalization after identifying num_leaves=7 as the best tree complexity.

lgbm_estimator_results = []

for n_estimators in [100, 200, 300, 400, 500]:
    fold_scores = []

    for fold, (train_idx, valid_idx) in enumerate(folds, start=1):
        X_train = X.iloc[train_idx].copy()
        X_valid = X.iloc[valid_idx].copy()
        y_train = y.iloc[train_idx]
        y_valid = y.iloc[valid_idx]

        preprocessor = ColumnTransformer(
            transformers=[
                (
                    "num",
                    SimpleImputer(strategy="median"),
                    numeric_features,
                ),
                (
                    "cat",
                    Pipeline([
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        (
                            "encoder",
                            OrdinalEncoder(
                                handle_unknown="use_encoded_value",
                                unknown_value=-1,
                            ),
                        ),
                    ]),
                    categorical_features,
                ),
            ]
        )

        model = LGBMRegressor(
            n_estimators=n_estimators,
            learning_rate=0.05,
            num_leaves=7,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="regression",
            random_state=42,
            n_jobs=-1,
            verbosity=-1,
        )

        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model),
        ])

        pipeline.fit(X_train, y_train)

        predictions = pipeline.predict(X_valid)

        rmse = np.sqrt(
            mean_squared_error(
                y_valid,
                predictions,
            )
        )

        fold_scores.append(rmse)

    lgbm_estimator_results.append({
        "n_estimators": n_estimators,
        "mean_rmse": np.mean(fold_scores),
        "std_rmse": np.std(fold_scores),
    })

lgbm_estimator_results = pd.DataFrame(lgbm_estimator_results)

lgbm_estimator_results.sort_values("mean_rmse")

,n_estimators,mean_rmse,std_rmse
0,100,1081.988825,19.648770
1,200,1085.432718,19.914547
2,300,1089.717055,19.639357
3,400,1092.876336,18.778516
4,500,1096.108872,18.585480


In [ ]:
# Generate leakage-safe out-of-fold predictions for our strongest models and measure whether their errors are sufficiently different to justify blending.

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import mean_squared_error

oof_predictions = {
    "catboost": np.zeros(len(y)),
    "xgboost": np.zeros(len(y)),
    "lightgbm": np.zeros(len(y)),
}

for fold, (train_idx, valid_idx) in enumerate(folds, start=1):
    X_train = X.iloc[train_idx].copy()
    X_valid = X.iloc[valid_idx].copy()
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    
    # CatBoost 
    X_train_cat = X_train.drop(columns=["product_category"]).copy()
    X_valid_cat = X_valid.drop(columns=["product_category"]).copy()

    for col in cat_features:
        X_train_cat[col] = X_train_cat[col].fillna("__MISSING__").astype(str)
        X_valid_cat[col] = X_valid_cat[col].fillna("__MISSING__").astype(str)

    cat_model = CatBoostRegressor(
        iterations=300,
        learning_rate=0.025,
        depth=4,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        thread_count=-1,
    )

    cat_model.fit(
        X_train_cat,
        y_train,
        cat_features=cat_features,
    )

    oof_predictions["catboost"][valid_idx] = cat_model.predict(X_valid_cat)

    # Shared preprocessing for XGBoost / LightGBM 
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                SimpleImputer(strategy="median"),
                numeric_features,
            ),
            (
                "cat",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    (
                        "encoder",
                        OrdinalEncoder(
                            handle_unknown="use_encoded_value",
                            unknown_value=-1,
                        ),
                    ),
                ]),
                categorical_features,
            ),
        ]
    )


    #  XGBoost 
    xgb_model = XGBRegressor(
        n_estimators=200,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        eval_metric="rmse",
        random_state=42,
        n_jobs=-1,
    )

    xgb_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", xgb_model),
    ])

    xgb_pipeline.fit(X_train, y_train)
    oof_predictions["xgboost"][valid_idx] = xgb_pipeline.predict(X_valid)

    
    # LightGBM
    lgbm_model = LGBMRegressor(
        n_estimators=100,
        learning_rate=0.05,
        num_leaves=7,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="regression",
        random_state=42,
        n_jobs=-1,
        verbosity=-1,
    )

    lgbm_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", lgbm_model),
    ])

    lgbm_pipeline.fit(X_train, y_train)
    oof_predictions["lightgbm"][valid_idx] = lgbm_pipeline.predict(X_valid)


oof_scores = pd.DataFrame({
    model_name: [
        np.sqrt(mean_squared_error(y, predictions))
    ]
    for model_name, predictions in oof_predictions.items()
}).T

oof_scores.columns = ["oof_rmse"]

oof_residuals = pd.DataFrame({
    model_name: y.to_numpy() - predictions
    for model_name, predictions in oof_predictions.items()
})

residual_correlations = oof_residuals.corr()

print("OOF RMSE")
display(oof_scores)

print("\nResidual correlation")
display(residual_correlations)

OOF RMSE


,oof_rmse
catboost,1075.525940
xgboost,1082.560946
lightgbm,1082.164379



Residual correlation


,catboost,xgboost,lightgbm
catboost,1.000000,0.991361,0.993905
xgboost,0.991361,1.000000,0.995317
lightgbm,0.993905,0.995317,1.000000


In [ ]:
# Find the best non-negative convex blend of CatBoost, XGBoost, and LightGBM using only locked-fold OOF predictions.

blend_results = []

cat_oof = oof_predictions["catboost"]
xgb_oof = oof_predictions["xgboost"]
lgbm_oof = oof_predictions["lightgbm"]

for cat_weight in np.arange(0.00, 1.01, 0.02):
    for xgb_weight in np.arange(0.00, 1.01 - cat_weight, 0.02):
        lgbm_weight = 1.0 - cat_weight - xgb_weight

        if lgbm_weight < -1e-9:
            continue

        blend_prediction = (
            cat_weight * cat_oof
            + xgb_weight * xgb_oof
            + lgbm_weight * lgbm_oof
        )

        rmse = np.sqrt(
            mean_squared_error(
                y,
                blend_prediction,
            )
        )

        blend_results.append({
            "catboost_weight": cat_weight,
            "xgboost_weight": xgb_weight,
            "lightgbm_weight": lgbm_weight,
            "oof_rmse": rmse,
        })

blend_results = pd.DataFrame(blend_results)

best_blend = (
    blend_results
    .sort_values("oof_rmse")
    .head(10)
    .reset_index(drop=True)
)

best_blend

,catboost_weight,xgboost_weight,lightgbm_weight,oof_rmse
0,0.88,0.12,0.000000e+00,1075.378547
1,0.86,0.14,0.000000e+00,1075.380349
2,0.90,0.10,-2.775558e-17,1075.384280
3,0.84,0.16,2.775558e-17,1075.389685
4,0.92,0.08,-4.163336e-17,1075.397546
5,0.82,0.18,-5.551115e-17,1075.406554
6,0.86,0.12,2.000000e-02,1075.407910
7,0.88,0.10,2.000000e-02,1075.409262
8,0.84,0.14,2.000000e-02,1075.414092
9,0.90,0.08,2.000000e-02,1075.418148


In [ ]:
# Aim: Identify where CatBoost's OOF RMSE is concentrated by quantifying absolute and squared error across sales-magnitude groups.

cat_oof = oof_predictions["catboost"]

error_analysis = pd.DataFrame({
    "actual_sales": y.to_numpy(),
    "prediction": cat_oof,
})

error_analysis["error"] = (
    error_analysis["actual_sales"] - error_analysis["prediction"]
)

error_analysis["absolute_error"] = (
    error_analysis["error"].abs()
)

error_analysis["squared_error"] = (
    error_analysis["error"] ** 2
)

error_analysis["sales_decile"] = pd.qcut(
    error_analysis["actual_sales"],
    q=10,
    duplicates="drop",
)

cat_error_by_decile = (
    error_analysis
    .groupby("sales_decile", observed=True)
    .agg(
        observations=("actual_sales", "size"),
        mean_actual=("actual_sales", "mean"),
        mean_prediction=("prediction", "mean"),
        mean_absolute_error=("absolute_error", "mean"),
        rmse=("squared_error", lambda x: np.sqrt(np.mean(x))),
        total_squared_error=("squared_error", "sum"),
    )
    .reset_index()
)

cat_error_by_decile["squared_error_share_pct"] = (
    100
    * cat_error_by_decile["total_squared_error"]
    / cat_error_by_decile["total_squared_error"].sum()
)

cat_error_by_decile

,sales_decile,observations,mean_actual,mean_prediction,mean_absolute_error,rmse,total_squared_error,squared_error_share_pct
0,"(32.699000000000005, 342.042]",682,191.706393,443.148825,278.912528,456.570289,1.421673e+08,1.802606
1,"(342.042, 669.366]",682,505.351349,1002.726318,553.759166,906.229264,5.600935e+08,7.101691
2,"(669.366, 1018.973]",682,839.744311,1315.385936,579.892275,912.598672,5.679944e+08,7.201870
3,"(1018.973, 1393.5]",681,1205.756990,1782.837931,729.915271,1008.881652,6.931505e+08,8.788784
4,"(1393.5, 1790.89]",682,1592.183871,2052.206786,663.732743,915.021790,5.710146e+08,7.240165
5,"(1790.89, 2256.888]",682,2006.526188,2389.732725,691.207391,885.209146,5.344119e+08,6.776062
6,"(2256.888, 2780.708]",681,2511.836623,2578.374824,618.424736,759.034820,3.923472e+08,4.974755
7,"(2780.708, 3466.5]",682,3104.332698,2970.657992,661.639373,826.484767,4.658586e+08,5.906841
8,"(3466.5, 4557.841]",682,3970.666393,3310.776147,884.015051,1053.936019,7.575527e+08,9.605370
9,"(4557.841, 12996.82]",682,5818.534355,3975.000525,1893.198274,2166.856442,3.202172e+09,40.601855


In [ ]:
# Determine whether CatBoost's OOF errors, especially large errors, are concentrated in particular stores.

store_error_analysis = error_analysis.copy()

store_error_analysis["store_code"] = train.loc[
    store_error_analysis.index,
    "store_code",
].to_numpy()

store_error_by_store = (
    store_error_analysis
    .groupby("store_code")
    .agg(
        observations=("actual_sales", "size"),
        mean_actual=("actual_sales", "mean"),
        mean_prediction=("prediction", "mean"),
        mean_error=("error", "mean"),
        mean_absolute_error=("absolute_error", "mean"),
        rmse=("squared_error", lambda x: np.sqrt(np.mean(x))),
        total_squared_error=("squared_error", "sum"),
    )
    .reset_index()
)

store_error_by_store["squared_error_share_pct"] = (
    100
    * store_error_by_store["total_squared_error"]
    / store_error_by_store["total_squared_error"].sum()
)

store_error_by_store.sort_values(
    "rmse",
    ascending=False,
)

,store_code,observations,mean_actual,mean_prediction,mean_error,mean_absolute_error,rmse,total_squared_error,squared_error_share_pct
0,STORE-7WS,748,3660.182219,3637.342498,22.839721,1032.716725,1401.568231,1.469366e+09,18.630792
9,STORE-YLW,754,2253.950690,2305.608134,-51.657444,846.910128,1156.462753,1.008404e+09,12.786034
7,STORE-OYG,744,2365.833656,2338.101392,27.732264,835.515038,1150.060890,9.840442e+08,12.477162
2,STORE-9RG,752,2479.168098,2417.831588,61.336510,832.781377,1119.796440,9.429659e+08,11.956312
4,STORE-DKU,736,2177.136481,2245.257419,-68.120938,799.179365,1105.479782,8.994550e+08,11.404615
1,STORE-89Z,728,2365.926745,2322.701637,43.225107,812.377426,1085.929277,8.584885e+08,10.885182
3,STORE-AGY,748,2254.890120,2293.363712,-38.473591,807.098883,1073.861897,8.625782e+08,10.937037
5,STORE-HL7,742,1971.661954,2021.384286,-49.722332,764.489428,1050.818780,8.193313e+08,10.388690
6,STORE-JOR,439,334.435011,351.455857,-17.020846,168.983758,229.150347,2.305184e+07,0.292285
8,STORE-T5G,427,338.326651,345.459130,-7.132479,157.578417,211.370574,1.907730e+07,0.241890


In [ ]:
# Determine whether the store-level error concentration is driven by specific product categories.

category_error_analysis = error_analysis.copy()

category_error_analysis["store_code"] = train.loc[
    category_error_analysis.index,
    "store_code",
].to_numpy()

category_error_analysis["product_category_clean"] = train.loc[
    category_error_analysis.index,
    "product_category_clean",
].to_numpy()

category_error_by_store_category = (
    category_error_analysis
    .groupby(["store_code", "product_category_clean"])
    .agg(
        observations=("actual_sales", "size"),
        mean_actual=("actual_sales", "mean"),
        mean_prediction=("prediction", "mean"),
        mean_error=("error", "mean"),
        mean_absolute_error=("absolute_error", "mean"),
        rmse=("squared_error", lambda x: np.sqrt(np.mean(x))),
        total_squared_error=("squared_error", "sum"),
    )
    .reset_index()
)

category_error_by_store_category["squared_error_share_pct"] = (
    100
    * category_error_by_store_category["total_squared_error"]
    / category_error_by_store_category["total_squared_error"].sum()
)

category_error_by_store_category.sort_values(
    "total_squared_error",
    ascending=False,
).head(20)

,store_code,product_category_clean,observations,mean_actual,mean_prediction,mean_error,mean_absolute_error,rmse,total_squared_error,squared_error_share_pct
6,STORE-7WS,fruits and vegetables,116,4021.409741,3771.535871,249.873870,1033.491999,1418.557546,2.334274e+08,2.959737
13,STORE-7WS,snack foods,103,3575.135631,3746.087342,-170.951711,1087.207699,1394.281389,2.002341e+08,2.538863
9,STORE-7WS,household,82,3877.592927,3816.886945,60.705982,1090.771490,1559.730598,1.994863e+08,2.529381
54,STORE-AGY,fruits and vegetables,124,2306.836774,2338.986538,-32.149764,857.462272,1184.756059,1.740522e+08,2.206891
22,STORE-89Z,fruits and vegetables,114,2444.613421,2341.495685,103.117736,913.899595,1218.533860,1.692700e+08,2.146255
118,STORE-OYG,fruits and vegetables,103,2558.418932,2465.668201,92.750731,884.859843,1263.297382,1.643798e+08,2.084249
77,STORE-DKU,snack foods,107,2138.355981,2301.938395,-163.582414,894.135779,1220.753604,1.594556e+08,2.021813
4,STORE-7WS,dairy,52,3855.248269,3783.771887,71.476383,1309.257259,1728.174228,1.553025e+08,1.969154
70,STORE-DKU,fruits and vegetables,106,2326.946321,2323.404826,3.541495,850.939711,1201.052831,1.529080e+08,1.938792
45,STORE-9RG,snack foods,111,2604.244414,2441.524563,162.719851,880.332357,1160.321945,1.494445e+08,1.894878


In [ ]:
# Inspect the largest CatBoost OOF errors to identify the feature patterns behind the remaining RMSE.

tail_errors = (
    error_analysis
    .join(
        train[
            [
                "product_code",
                "product_weight_kg",
                "fat_content",
                "product_category_clean",
                "product_price",
                "store_code",
                "store_age_years",
                "store_size",
                "store_location_tier",
                "store_format",
            ]
        ]
    )
    .sort_values("absolute_error", ascending=False)
)

tail_errors[
    [
        "actual_sales",
        "prediction",
        "error",
        "absolute_error",
        "product_code",
        "product_category_clean",
        "product_price",
        "product_weight_kg",
        "fat_content",
        "store_code",
        "store_size",
        "store_location_tier",
        "store_format",
    ]
].head(30)

,actual_sales,prediction,error,absolute_error,product_code,product_category_clean,product_price,product_weight_kg,fat_content,store_code,store_size,store_location_tier,store_format
5404,12996.82,5644.635794,7352.184206,7352.184206,PRD-I67HN3,household,232.78,NaN,Low Fat,STORE-7WS,Medium,Tier_3,Flagship Hypermarket
4220,12359.04,5817.634763,6541.405237,6541.405237,PRD-5I78U5,fruits and vegetables,241.26,NaN,Regular,STORE-7WS,Medium,Tier_3,Flagship Hypermarket
900,10130.90,3985.660883,6145.239117,6145.239117,PRD-PUWBWJ,dairy,262.30,12.342,Regular,STORE-OYG,NaN,Tier_2,Standard Supermarket
5950,9999.51,4056.881178,5942.628822,5942.628822,PRD-LV9PLM,health and hygiene,268.58,6.868,Low Fat,STORE-YLW,Small,Tier_1,Standard Supermarket
1555,9388.15,3953.194206,5434.955794,5434.955794,PRD-B9OF3M,fruits and vegetables,262.55,17.089,Low Fat,STORE-DKU,NaN,Tier_2,Standard Supermarket
4511,9072.04,3792.519616,5279.520384,5279.520384,PRD-07NNDG,frozen foods,262.86,9.666,Regular,STORE-DKU,NaN,Tier_2,Standard Supermarket
3444,8617.32,3662.666768,4954.653232,4954.653232,PRD-N5F3DX,snack foods,224.71,16.722,Regular,STORE-OYG,NaN,Tier_2,Standard Supermarket
625,8894.14,3976.191628,4917.948372,4917.948372,PRD-YW8YG4,frozen foods,255.71,5.525,Low Fat,STORE-OYG,NaN,Tier_2,Standard Supermarket
4597,8714.71,3812.342322,4902.367678,4902.367678,PRD-UX01UP,snack foods,242.05,10.675,Low Fat,STORE-DKU,NaN,Tier_2,Standard Supermarket
4363,1377.26,6192.311810,-4815.051810,4815.051810,PRD-6YLBXP,canned,258.91,NaN,Regular,STORE-7WS,Medium,Tier_3,Flagship Hypermarket


In [ ]:
#Fix categorical missing values so the product target-encoding experiment can run without changing its modeling hypothesis.

product_te_rmses = []

for fold_num, (train_idx, valid_idx) in enumerate(folds, start=1):
    X_train = X_cat.iloc[train_idx].copy()
    X_valid = X_cat.iloc[valid_idx].copy()
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    # Keep CatBoost categorical inputs valid and consistent with the champion pipeline.
    for col in cat_features:
        X_train[col] = X_train[col].fillna("Missing").astype(str)
        X_valid[col] = X_valid[col].fillna("Missing").astype(str)

    global_mean = y_train.mean()

    product_stats = (
        pd.DataFrame({
            "product_code": X_train["product_code"].values,
            "target": y_train.values,
        })
        .groupby("product_code")["target"]
        .agg(["mean", "count"])
    )

    smoothing = 10

    product_stats["product_te"] = (
        (product_stats["mean"] * product_stats["count"]
         + global_mean * smoothing)
        / (product_stats["count"] + smoothing)
    )

    X_train["product_te"] = (
        X_train["product_code"]
        .map(product_stats["product_te"])
        .fillna(global_mean)
    )

    X_valid["product_te"] = (
        X_valid["product_code"]
        .map(product_stats["product_te"])
        .fillna(global_mean)
    )

    model = CatBoostRegressor(
        iterations=300,
        learning_rate=0.025,
        depth=4,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        thread_count=-1,
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
    )

    pred = model.predict(X_valid)

    fold_rmse = np.sqrt(mean_squared_error(y_valid, pred))
    product_te_rmses.append(fold_rmse)

    print(f"Fold {fold_num}: {fold_rmse:.4f}")

print(f"\nMean RMSE: {np.mean(product_te_rmses):.4f}")
print(f"Std RMSE: {np.std(product_te_rmses):.4f}")
print(
    f"Change vs champion: "
    f"{1075.3555 - np.mean(product_te_rmses):+.4f}"
)

Fold 1: 1181.6405
Fold 2: 1105.5049
Fold 3: 1189.9296
Fold 4: 1240.7870
Fold 5: 1209.2775

Mean RMSE: 1185.4279
Std RMSE: 44.8458
Change vs champion: -110.0724


In [ ]:
# Test whether modestly emphasizing high-sales training observations reduces the upper-tail errors that dominate RMSE.

tail_weight_rmses = []

for fold_num, (train_idx, valid_idx) in enumerate(folds, start=1):
    X_train = X_cat.iloc[train_idx].copy()
    X_valid = X_cat.iloc[valid_idx].copy()
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    for col in cat_features:
        X_train[col] = X_train[col].fillna("__MISSING__").astype(str)
        X_valid[col] = X_valid[col].fillna("__MISSING__").astype(str)

    # Weight observations by their training-fold sales level.
    # sqrt keeps the weighting moderate rather than letting extreme rows dominate.
    train_mean = y_train.mean()
    sample_weight = np.sqrt(y_train / train_mean).clip(lower=0.5, upper=3.0)

    model = CatBoostRegressor(
        iterations=300,
        learning_rate=0.025,
        depth=4,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        thread_count=-1,
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
        sample_weight=sample_weight,
    )

    pred = model.predict(X_valid)

    fold_rmse = np.sqrt(mean_squared_error(y_valid, pred))
    tail_weight_rmses.append(fold_rmse)

    print(f"Fold {fold_num}: {fold_rmse:.4f}")

tail_weight_mean = np.mean(tail_weight_rmses)
tail_weight_std = np.std(tail_weight_rmses)

print(f"\nMean RMSE: {tail_weight_mean:.4f}")
print(f"Std RMSE: {tail_weight_std:.4f}")
print(f"Change vs champion: {1075.3555 - tail_weight_mean:+.4f}"
      )

Fold 1: 1082.2616
Fold 2: 1078.1458
Fold 3: 1124.8440
Fold 4: 1137.8189
Fold 5: 1087.0489

Mean RMSE: 1102.0238
Std RMSE: 24.4418
Change vs champion: -26.6683


In [ ]:
# Test whether an explicit nonlinear price term helps the shallow CatBoost capture curvature in the price-sales relationship.

price_squared_rmses = []

for fold_num, (train_idx, valid_idx) in enumerate(folds, start=1):
    X_train = X_cat.iloc[train_idx].copy()
    X_valid = X_cat.iloc[valid_idx].copy()
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    for col in cat_features:
        X_train[col] = X_train[col].fillna("__MISSING__").astype(str)
        X_valid[col] = X_valid[col].fillna("__MISSING__").astype(str)

    X_train["product_price_squared"] = X_train["product_price"] ** 2
    X_valid["product_price_squared"] = X_valid["product_price"] ** 2

    model = CatBoostRegressor(
        iterations=300,
        learning_rate=0.025,
        depth=4,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        thread_count=-1,
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
    )

    pred = model.predict(X_valid)

    fold_rmse = np.sqrt(mean_squared_error(y_valid, pred))
    price_squared_rmses.append(fold_rmse)

    print(f"Fold {fold_num}: {fold_rmse:.4f}")

price_squared_mean = np.mean(price_squared_rmses)
price_squared_std = np.std(price_squared_rmses)

print(f"\nMean RMSE: {price_squared_mean:.4f}")
print(f"Std RMSE: {price_squared_std:.4f}")
print(f"Change vs champion: {1075.3555 - price_squared_mean:+.4f}")

Fold 1: 1064.6394
Fold 2: 1052.1159
Fold 3: 1087.1516
Fold 4: 1107.7960
Fold 5: 1065.1078

Mean RMSE: 1075.3621
Std RMSE: 19.7558
Change vs champion: -0.0066


In [ ]:
# Determine whether high-sales behavior is concentrated in specific price bands within stores.

price_store_tail = train.copy()

price_store_tail["price_band"] = pd.qcut(
    price_store_tail["product_price"],
    q=5,
    duplicates="drop"
)

price_store_tail["high_sales"] = (
    price_store_tail["total_sales"] >= price_store_tail["total_sales"].quantile(0.90)
)

price_store_tail_profile = (
    price_store_tail
    .groupby(["store_code", "price_band"], observed=True)
    .agg(
        observations=("total_sales", "size"),
        mean_sales=("total_sales", "mean"),
        median_sales=("total_sales", "median"),
        high_sales_rate=("high_sales", "mean"),
        mean_price=("product_price", "mean"),
    )
    .reset_index()
)

price_store_tail_profile["high_sales_rate"] *= 100

price_store_tail_profile.sort_values(
    "mean_sales", ascending=False
).head(20)

,store_code,price_band,observations,mean_sales,median_sales,high_sales_rate,mean_price
4,STORE-7WS,"(194.228, 273.04]",142,5819.930704,5702.310,70.422535,228.621901
3,STORE-7WS,"(158.83, 194.228]",161,4754.122422,4702.160,53.416149,176.803106
14,STORE-9RG,"(194.228, 273.04]",158,4091.917658,4172.505,37.974684,230.710759
39,STORE-OYG,"(194.228, 273.04]",149,3990.268188,3705.160,36.241611,233.165235
9,STORE-89Z,"(194.228, 273.04]",146,3817.805890,3637.040,31.506849,231.625205
19,STORE-AGY,"(194.228, 273.04]",158,3633.293608,3399.955,28.481013,230.212342
2,STORE-7WS,"(117.848, 158.83]",141,3583.986312,3470.430,20.567376,138.462340
49,STORE-YLW,"(194.228, 273.04]",156,3575.762244,3478.970,29.487179,230.404423
24,STORE-DKU,"(194.228, 273.04]",146,3429.749658,3094.475,23.287671,229.049589
29,STORE-HL7,"(194.228, 273.04]",155,3272.581613,3115.810,21.290323,231.150000


In [ ]:
# Test whether an explicit store × price-band interaction captures the strong nonlinear price behavior seen within stores.

store_price_band_rmses = []

for fold_num, (train_idx, valid_idx) in enumerate(folds, start=1):
    X_train = X_cat.iloc[train_idx].copy()
    X_valid = X_cat.iloc[valid_idx].copy()
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    for col in cat_features:
        X_train[col] = X_train[col].fillna("Missing").astype(str)
        X_valid[col] = X_valid[col].fillna("Missing").astype(str)

    # Learn price-band boundaries from the training fold only.
    _, price_bins = pd.qcut(
        X_train["product_price"],
        q=5,
        retbins=True,
        duplicates="drop",
    )

    # Ensure validation prices are assigned using the training-fold boundaries.
    X_train["price_band"] = pd.cut(
        X_train["product_price"],
        bins=price_bins,
        include_lowest=True,
    ).astype(str)

    X_valid["price_band"] = pd.cut(
        X_valid["product_price"],
        bins=price_bins,
        include_lowest=True,
    ).astype(str)

    # Explicit store × price-band interaction.
    X_train["store_price_band"] = (
        X_train["store_code"] + "__" + X_train["price_band"]
    )
    X_valid["store_price_band"] = (
        X_valid["store_code"] + "__" + X_valid["price_band"]
    )

    interaction_cat_features = cat_features + [
        "price_band",
        "store_price_band",
    ]

    model = CatBoostRegressor(
        iterations=300,
        learning_rate=0.025,
        depth=4,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        thread_count=-1,
    )

    model.fit(
        X_train,
        y_train,
        cat_features=interaction_cat_features,
    )

    pred = model.predict(X_valid)

    fold_rmse = np.sqrt(mean_squared_error(y_valid, pred))
    store_price_band_rmses.append(fold_rmse)

    print(f"Fold {fold_num}: {fold_rmse:.4f}")

store_price_band_mean = np.mean(store_price_band_rmses)
store_price_band_std = np.std(store_price_band_rmses)

print(f"\nMean RMSE: {store_price_band_mean:.4f}")
print(f"Std RMSE: {store_price_band_std:.4f}")
print(f"Change vs champion: {1075.3555 - store_price_band_mean:+.4f}")

Fold 1: 1074.8842
Fold 2: 1059.3166
Fold 3: 1088.8471
Fold 4: 1123.3688
Fold 5: 1069.8672

Mean RMSE: 1083.2568
Std RMSE: 22.1934
Change vs champion: -7.9013


In [ ]:
# Generate leakage-safe OOF predictions from a deliberately different CatBoost configuration to test whether model diversity can improve the blend.

cat_diverse_oof = np.zeros(len(y))

for fold_num, (train_idx, valid_idx) in enumerate(folds, start=1):
    X_train = X_cat.iloc[train_idx].copy()
    X_valid = X_cat.iloc[valid_idx].copy()
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    for col in cat_features:
        X_train[col] = X_train[col].fillna("Missing").astype(str)
        X_valid[col] = X_valid[col].fillna("Missing").astype(str)

    model = CatBoostRegressor(
        iterations=500,
        learning_rate=0.015,
        depth=5,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        thread_count=-1,
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
    )

    cat_diverse_oof[valid_idx] = model.predict(X_valid)

    print(f"Fold {fold_num} complete")

cat_diverse_rmse = np.sqrt(mean_squared_error(y, cat_diverse_oof))
residual_corr = np.corrcoef(
    y - cat_oof,
    y - cat_diverse_oof
)[0, 1]

print(f"\nDiverse CatBoost OOF RMSE: {cat_diverse_rmse:.4f}")
print(f"Residual correlation vs champion: {residual_corr:.6f}")

Fold 1 complete
Fold 2 complete
Fold 3 complete
Fold 4 complete
Fold 5 complete

Diverse CatBoost OOF RMSE: 1076.0989
Residual correlation vs champion: 0.999394


In [ ]:
product_sales_stability = (
    train
    .groupby("product_code")["total_sales"]
    .agg(
        observations="size",
        mean_sales="mean",
        median_sales="median",
        std_sales="std",
        min_sales="min",
        max_sales="max",
    )
)

product_sales_stability["cv"] = (
    product_sales_stability["std_sales"]
    / product_sales_stability["mean_sales"]
)

product_sales_stability_profile = (
    product_sales_stability
    .query("observations >= 2")
    .groupby("observations")
    .agg(
        products=("mean_sales", "size"),
        median_cv=("cv", "median"),
        mean_cv=("cv", "mean"),
        median_range=("max_sales", lambda x: np.median(x)),
    )
    .reset_index()
)

print(product_sales_stability_profile)

print("\nOverall repeated-product statistics:")
print(
    product_sales_stability
    .query("observations >= 2")[["observations", "cv"]]
    .describe()
)

   observations  products  median_cv   mean_cv  median_range
0             2       140   0.371590  0.477942       2821.89
1             3       251   0.571104  0.569189       3260.23
2             4       394   0.576386  0.575928       3235.06
3             5       351   0.565867  0.573217       3505.48
4             6       240   0.590349  0.586617       3187.50
5             7        97   0.609231  0.606955       3791.41
6             8        35   0.621930  0.646345       4438.53
7             9         1   0.605521  0.605521       2161.90

Overall repeated-product statistics:
       observations           cv
count   1509.000000  1509.000000
mean       4.487740     0.570433
std        1.461491     0.234613
min        2.000000     0.002935
25%        3.000000     0.414485
50%        4.000000     0.573912
75%        5.000000     0.710269
max        9.000000     1.466994


In [ ]:
# Quantify whether repeated-product sales variability is substantially explained by the store, which determines whether product×store interactions are worth testing.

product_store_sales = (
    train
    .groupby(["product_code", "store_code"])["total_sales"]
    .agg(
        observations="size",
        mean_sales="mean",
        std_sales="std",
    )
    .reset_index()
)

product_store_sales["cv"] = (
    product_store_sales["std_sales"]
    / product_store_sales["mean_sales"]
)

print("Product-store combinations:", len(product_store_sales))
print("Repeated product-store combinations:", (product_store_sales["observations"] >= 2).sum())

print("\nProduct-store CV summary:")
print(
    product_store_sales.loc[
        product_store_sales["observations"] >= 2, "cv"
    ].describe()
)

print("\nProduct observations by number of stores:")
print(
    train.groupby("product_code")["store_code"]
    .nunique()
    .value_counts()
    .sort_index()
)

Product-store combinations: 6818
Repeated product-store combinations: 0

Product-store CV summary:
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: cv, dtype: float64

Product observations by number of stores:
store_code
1     46
2    140
3    251
4    394
5    351
6    240
7     97
8     35
9      1
Name: count, dtype: int64


In [ ]:
# Test whether CatBoost's ordered boosting improves generalization for our categorical-heavy dataset.

from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

ordered_scores = []

for fold, (train_idx, val_idx) in enumerate(folds, start=1):
    X_train = X_cat.iloc[train_idx].copy()
    X_val = X_cat.iloc[val_idx].copy()
    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    for col in cat_features:
        X_train[col] = X_train[col].fillna("Missing").astype(str)
        X_val[col] = X_val[col].fillna("Missing").astype(str)

    model = CatBoostRegressor(
        iterations=300,
        learning_rate=0.025,
        depth=4,
        loss_function="RMSE",
        random_seed=42,
        boosting_type="Ordered",
        verbose=False,
        thread_count=-1,
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
    )

    pred = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, pred))
    ordered_scores.append(rmse)

    print(f"Fold {fold}: {rmse:.4f}")

print(f"\nOrdered boosting CV RMSE: {np.mean(ordered_scores):.4f}")
print(f"Std: {np.std(ordered_scores):.4f}")
print(
    f"Change vs champion: "
    f"{1075.3555 - np.mean(ordered_scores):+.4f}"
)

Fold 1: 1065.3577
Fold 2: 1052.8215
Fold 3: 1085.5682
Fold 4: 1108.2951
Fold 5: 1062.8881

Ordered boosting CV RMSE: 1074.9861
Std: 19.7576
Change vs champion: +0.3694


In [ ]:
# Aim: Test whether CatBoost's L2 leaf regularization can improve the current Ordered boosting champion.

from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd

l2_results = []

for l2_value in [1, 3, 5, 10, 20]:
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(folds, start=1):
        X_train = X_cat.iloc[train_idx].copy()
        X_val = X_cat.iloc[val_idx].copy()
        y_train = y.iloc[train_idx]
        y_val = y.iloc[val_idx]

        for col in cat_features:
            X_train[col] = X_train[col].fillna("__MISSING__").astype(str)
            X_val[col] = X_val[col].fillna("__MISSING__").astype(str)

        model = CatBoostRegressor(
            iterations=300,
            learning_rate=0.025,
            depth=4,
            l2_leaf_reg=l2_value,
            loss_function="RMSE",
            random_seed=42,
            boosting_type="Ordered",
            verbose=False,
            thread_count=-1,
        )

        model.fit(
            X_train,
            y_train,
            cat_features=cat_features,
        )

        pred = model.predict(X_val)
        fold_scores.append(
            np.sqrt(mean_squared_error(y_val, pred))
        )

    l2_results.append({
        "l2_leaf_reg": l2_value,
        "mean_rmse": np.mean(fold_scores),
        "std_rmse": np.std(fold_scores),
    })

l2_results = pd.DataFrame(l2_results)

print(l2_results)
print("\nBest:")
print(l2_results.loc[l2_results["mean_rmse"].idxmin()])

   l2_leaf_reg    mean_rmse   std_rmse
0            1  1074.915874  19.310738
1            3  1074.986121  19.757582
2            5  1075.288297  19.431896
3           10  1075.559436  19.385734
4           20  1075.289631  18.660674

Best:
l2_leaf_reg       1.000000
mean_rmse      1074.915874
std_rmse         19.310738
Name: 0, dtype: float64


In [ ]:
# Aim: Measure high-sales-tail rates across categorical groups without changing the modeling dataset.

train["_sales_tail"] = tail_flag

for col in [
    "product_category_clean",
    "store_code",
    "fat_content",
    "store_size",
    "store_location_tier",
    "store_format",
]:
    profile = (
        train.groupby(col)
        .agg(
            observations=("total_sales", "size"),
            mean_sales=("total_sales", "mean"),
            tail_rate=("_sales_tail", "mean"),
        )
        .sort_values("tail_rate", ascending=False)
    )

    print(f"\n--- {col} ---")
    print(profile)

train.drop(columns="_sales_tail", inplace=True)


--- product_category_clean ---
                        observations   mean_sales  tail_rate
product_category_clean                                      
seafood                           52  2262.524231   0.134615
starchy foods                    113  2407.718584   0.123894
meat                             339  2264.310383   0.120944
fruits and vegetables           1006  2277.854672   0.119284
dairy                            554  2238.317076   0.117329
household                        740  2292.464662   0.106757
frozen foods                     675  2140.770133   0.106667
snack foods                      933  2223.830568   0.102894
canned                           517  2196.398743   0.094778
soft drinks                      366  1967.399454   0.087432
breads                           209  2186.152871   0.086124
others                           132  1947.435833   0.075758
baking goods                     516  1943.416860   0.073643
breakfast                         85  1852.197765   0

In [ ]:
# Test whether a row's price relative to its store's typical price adds signal beyond raw price and store identity.

store_relative_scores = []

for fold, (train_idx, val_idx) in enumerate(folds, start=1):
    X_train = X_cat.iloc[train_idx].copy()
    X_val = X_cat.iloc[val_idx].copy()
    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    train_price = train.iloc[train_idx].copy()
    val_price = train.iloc[val_idx].copy()

    store_median_price = (
        train_price
        .groupby("store_code")["product_price"]
        .median()
    )

    global_median_price = train_price["product_price"].median()

    train_store_median = (
        train_price["store_code"]
        .map(store_median_price)
        .fillna(global_median_price)
    )

    val_store_median = (
        val_price["store_code"]
        .map(store_median_price)
        .fillna(global_median_price)
    )

    X_train["store_relative_price"] = (
        train_price["product_price"] / train_store_median
    )

    X_val["store_relative_price"] = (
        val_price["product_price"] / val_store_median
    )

    for col in cat_features:
        X_train[col] = X_train[col].fillna("__MISSING__").astype(str)
        X_val[col] = X_val[col].fillna("__MISSING__").astype(str)

    model = CatBoostRegressor(
        iterations=300,
        learning_rate=0.025,
        depth=4,
        l2_leaf_reg=1,
        loss_function="RMSE",
        random_seed=42,
        boosting_type="Ordered",
        verbose=False,
        thread_count=-1,
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
    )

    pred = model.predict(X_val)

    rmse = np.sqrt(mean_squared_error(y_val, pred))
    store_relative_scores.append(rmse)

    print(f"Fold {fold}: {rmse:.4f}")

print(f"\nStore-relative price CV RMSE: {np.mean(store_relative_scores):.4f}")
print(f"Std: {np.std(store_relative_scores):.4f}")
print(
    f"Change vs current champion: "
    f"{1074.915874 - np.mean(store_relative_scores):+.4f}"
)

Fold 1: 1066.7600
Fold 2: 1054.3620
Fold 3: 1086.8134
Fold 4: 1109.7584
Fold 5: 1061.7661

Store-relative price CV RMSE: 1075.8920
Std: 20.0633
Change vs current champion: -0.9761


In [ ]:
# Determine whether extreme prediction errors are concentrated among products with many or few observations.

error_product_frequency = error_analysis.copy()

product_counts = (
    train["product_code"]
    .value_counts()
    .rename("product_observations")
)

error_product_frequency["product_observations"] = (
    train.loc[error_product_frequency.index, "product_code"]
    .map(product_counts)
)

error_product_frequency["error_direction"] = np.where(
    error_product_frequency["error"] < 0,
    "underprediction",
    "overprediction",
)

error_product_frequency["error_magnitude"] = (
    error_product_frequency["absolute_error"]
)

print(
    error_product_frequency.groupby("product_observations")
    .agg(
        rows=("error", "size"),
        mean_abs_error=("error_magnitude", "mean"),
        rmse=("squared_error", lambda x: np.sqrt(x.mean())),
        mean_error=("error", "mean"),
    )
    .reset_index()
)

print("\nExtreme errors:")
print(
    error_product_frequency[
        error_product_frequency["error_magnitude"]
        >= error_product_frequency["error_magnitude"].quantile(0.90)
    ][
        [
            "actual_sales",
            "prediction",
            "error",
            "absolute_error",
            "product_observations",
            "sales_decile",
        ]
    ]
    .sort_values("absolute_error", ascending=False)
    .head(20)
)

   product_observations  rows  mean_abs_error         rmse  mean_error
0                     1    46      809.584539  1181.023378 -161.349428
1                     2   280      822.208855  1132.137927   10.988473
2                     3   753      798.939544  1138.891493    6.685142
3                     4  1576      748.044501  1064.271169  -31.129113
4                     5  1755      776.923898  1093.716677    1.334467
5                     6  1440      694.778122  1021.483947  -16.957537
6                     7   679      757.898667  1059.397987   32.055875
7                     8   280      793.477297  1096.803865  -10.254613
8                     9     9      245.236499   355.231318  176.253066

Extreme errors:
      actual_sales   prediction        error  absolute_error  \
5404      12996.82  5644.635794  7352.184206     7352.184206   
4220      12359.04  5817.634763  6541.405237     6541.405237   
900       10130.90  3985.660883  6145.239117     6145.239117   
5950       9999.5

In [ ]:
# Measure whether price remains informative within each sales decile, especially inside the high-sales tail.

price_sales_by_decile = (
    train.assign(
        sales_decile=pd.qcut(
            train["total_sales"],
            q=10,
            duplicates="drop"
        )
    )
    .groupby("sales_decile", observed=True)
    .agg(
        observations=("total_sales", "size"),
        mean_sales=("total_sales", "mean"),
        mean_price=("product_price", "mean"),
        median_price=("product_price", "median"),
        price_sales_corr=(
            "product_price",
            lambda x: x.corr(
                train.loc[x.index, "total_sales"]
            )
        ),
    )
    .reset_index()
)

print(price_sales_by_decile)

                    sales_decile  observations   mean_sales  mean_price  \
0  (32.699000000000005, 342.042]           682   191.706393  102.666144   
1             (342.042, 669.366]           682   505.351349  108.053196   
2            (669.366, 1018.973]           682   839.744311   97.681276   
3             (1018.973, 1393.5]           681  1205.756990  112.234185   
4              (1393.5, 1790.89]           682  1592.183871  124.214795   
5            (1790.89, 2256.888]           682  2006.526188  141.783123   
6           (2256.888, 2780.708]           681  2511.836623  152.290587   
7             (2780.708, 3466.5]           682  3104.332698  170.356202   
8             (3466.5, 4557.841]           682  3970.666393  185.525777   
9           (4557.841, 12996.82]           682  5818.534355  209.906862   

   median_price  price_sales_corr  
0        95.460          0.269089  
1        96.070         -0.151661  
2        80.115          0.044071  
3       102.930          0.125

In [ ]:
# Identify which store-category groups dominate the high-sales tail and whether their tail behavior is unusually strong.

tail_cutoff = train["total_sales"].quantile(0.90)

tail_store_category = (
    train.loc[train["total_sales"] >= tail_cutoff]
    .groupby(["store_code", "product_category_clean"])
    .agg(
        observations=("total_sales", "size"),
        mean_sales=("total_sales", "mean"),
        median_sales=("total_sales", "median"),
        mean_price=("product_price", "mean"),
        max_sales=("total_sales", "max"),
    )
    .reset_index()
)

tail_store_category = tail_store_category.sort_values(
    ["mean_sales", "observations"],
    ascending=[False, False]
)

print(tail_store_category.head(25).to_string(index=False))

store_code product_category_clean  observations  mean_sales  median_sales  mean_price  max_sales
 STORE-AGY            hard drinks             1 8141.490000      8141.490  251.080000    8141.49
 STORE-YLW     health and hygiene             4 7612.530000      7860.975  230.840000    9999.51
 STORE-89Z              breakfast             1 7125.940000      7125.940  233.830000    7125.94
 STORE-7WS                  dairy            14 7099.707143      6393.600  205.020714   10477.97
 STORE-7WS            soft drinks            10 7027.707000      7085.070  185.966000    9803.38
 STORE-89Z     health and hygiene             1 6941.160000      6941.160  195.470000    6941.16
 STORE-HL7                 canned             2 6790.135000      6790.135  261.060000    7016.22
 STORE-DKU           frozen foods             6 6678.790000      6522.375  244.323333    9072.04
 STORE-OYG           baking goods             1 6670.190000      6670.190  258.440000    6670.19
 STORE-AGY            soft dri

In [ ]:
# Test whether high-sales tail behavior within store-category groups is primarily associated with price.

tail_store_category_price = (
    train.loc[train["total_sales"] >= tail_cutoff]
    .groupby(["store_code", "product_category_clean"])
    .agg(
        observations=("total_sales", "size"),
        mean_sales=("total_sales", "mean"),
        mean_price=("product_price", "mean"),
        min_price=("product_price", "min"),
        max_price=("product_price", "max"),
        price_sales_corr=(
            "product_price",
            lambda x: x.corr(train.loc[x.index, "total_sales"])
        ),
    )
    .reset_index()
)

tail_store_category_price = tail_store_category_price.sort_values(
    ["observations", "mean_sales"],
    ascending=[False, False]
)

print(
    tail_store_category_price[
        tail_store_category_price["observations"] >= 5
    ].to_string(index=False)
)

c:\Users\Administrator\Documents\python training and projects\env\lib\site-packages\numpy\lib\_function_base_impl.py:3037: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
c:\Users\Administrator\Documents\python training and projects\env\lib\site-packages\numpy\lib\_function_base_impl.py:2894: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
c:\Users\Administrator\Documents\python training and projects\env\lib\site-packages\numpy\lib\_function_base_impl.py:2894: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)


store_code product_category_clean  observations  mean_sales  mean_price  min_price  max_price  price_sales_corr
 STORE-7WS  fruits and vegetables            43 6135.629070  187.834186     110.48     266.80          0.572064
 STORE-7WS            snack foods            31 5779.822581  186.356774     111.76     258.26          0.339312
 STORE-7WS              household            30 6289.519667  199.084000     121.00     265.65          0.408024
 STORE-7WS           frozen foods            23 6363.699565  211.018261     134.30     268.75          0.125537
 STORE-7WS                   meat            17 6500.438235  200.357647     101.33     256.98          0.510455
 STORE-7WS                 canned            16 5997.521875  193.353750     101.51     260.28          0.461039
 STORE-89Z  fruits and vegetables            15 5425.553333  201.605333     140.90     255.00          0.762566
 STORE-7WS                  dairy            14 7099.707143  205.020714     151.19     257.16          0

In [ ]:
# Determine whether CatBoost systematically underpredicts or overpredicts high-sales observations at different price levels.

tail_error_price = error_analysis.copy()

tail_error_price["price"] = train.loc[tail_error_price.index, "product_price"]
tail_error_price["tail_flag"] = (
    tail_error_price["actual_sales"] >= tail_cutoff
)

tail_error_price = tail_error_price.loc[
    tail_error_price["tail_flag"]
].copy()

tail_error_price["price_bin"] = pd.qcut(
    tail_error_price["price"],
    q=5,
    duplicates="drop"
)

tail_price_error_profile = (
    tail_error_price
    .groupby("price_bin", observed=True)
    .agg(
        observations=("error", "size"),
        mean_price=("price", "mean"),
        mean_actual=("actual_sales", "mean"),
        mean_prediction=("prediction", "mean"),
        mean_error=("error", "mean"),
        mean_abs_error=("absolute_error", "mean"),
        rmse=("squared_error", lambda x: np.sqrt(x.mean())),
    )
    .reset_index()
)

print(tail_price_error_profile.to_string(index=False))

         price_bin  observations  mean_price  mean_actual  mean_prediction  mean_error  mean_abs_error        rmse
(101.329, 177.504]           137  156.727737  5365.294891      3403.303750 1961.991140     1965.893030 2152.577917
 (177.504, 195.11]           136  186.498750  5757.429485      3843.084701 1914.344784     1942.386083 2183.948095
 (195.11, 225.996]           136  212.006618  5755.714706      3939.133306 1816.581400     1834.835453 2096.396603
 (225.996, 248.77]           136  236.773750  6140.917059      4344.257990 1796.659069     1898.089746 2232.432575
  (248.77, 271.82]           137  257.568029  6074.764234      4346.693491 1728.070742     1824.755793 2166.779927


In [ ]:
# Test whether CatBoost split-score randomization improves the current Ordered CatBoost champion under the locked 5-fold CV.

from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

random_strength_values = [0, 0.25, 0.5, 1.0, 2.0]

random_strength_results = []

for random_strength in random_strength_values:
    fold_scores = []

    for train_idx, valid_idx in folds:
        X_train = X_cat.iloc[train_idx].copy()
        X_valid = X_cat.iloc[valid_idx].copy()
        y_train = y.iloc[train_idx]
        y_valid = y.iloc[valid_idx]

        for col in cat_features:
            X_train[col] = X_train[col].fillna("__MISSING__").astype(str)
            X_valid[col] = X_valid[col].fillna("__MISSING__").astype(str)

        model = CatBoostRegressor(
            iterations=300,
            learning_rate=0.025,
            depth=4,
            l2_leaf_reg=1,
            boosting_type="Ordered",
            random_strength=random_strength,
            loss_function="RMSE",
            random_seed=42,
            verbose=False,
            thread_count=-1,
        )

        model.fit(
            X_train,
            y_train,
            cat_features=cat_features,
        )

        pred = model.predict(X_valid)
        rmse = np.sqrt(mean_squared_error(y_valid, pred))
        fold_scores.append(rmse)

    random_strength_results.append({
        "random_strength": random_strength,
        "mean_rmse": np.mean(fold_scores),
        "std_rmse": np.std(fold_scores),
    })

random_strength_results = pd.DataFrame(random_strength_results)

print(random_strength_results)

   random_strength    mean_rmse   std_rmse
0             0.00  1076.135342  20.304341
1             0.25  1075.946831  20.400768
2             0.50  1075.038402  19.712375
3             1.00  1074.915874  19.310738
4             2.00  1076.265941  19.215002


In [ ]:
# Verify the tail threshold and class balance before designing the leakage-safe two-stage regime model.

tail_cutoff = y.quantile(0.90)
tail_flag = (y >= tail_cutoff).astype(int)

print(f"Tail cutoff (90th percentile): {tail_cutoff:.6f}")
print(f"Tail rows: {tail_flag.sum()} / {len(y)} ({tail_flag.mean():.2%})")

print("\nTail rate by locked fold:")
for fold_num, (train_idx, valid_idx) in enumerate(folds, start=1):
    print(
        f"Fold {fold_num}: "
        f"train={tail_flag.iloc[train_idx].mean():.2%}, "
        f"valid={tail_flag.iloc[valid_idx].mean():.2%}"
    )

Tail cutoff (90th percentile): 4557.841000
Tail rows: 682 / 6818 (10.00%)

Tail rate by locked fold:
Fold 1: train=9.90%, valid=10.41%
Fold 2: train=10.18%, valid=9.31%
Fold 3: train=10.18%, valid=9.31%
Fold 4: train=9.94%, valid=10.27%
Fold 5: train=9.83%, valid=10.71%


In [ ]:
# Test a leakage-safe two-stage tail model against the current Ordered CatBoost champion using the locked 5-fold CV.

from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd

tail_cutoff = y.quantile(0.90)
tail_models_results = []

for fold_num, (train_idx, valid_idx) in enumerate(folds, start=1):
    X_train = X_cat.iloc[train_idx].copy()
    X_valid = X_cat.iloc[valid_idx].copy()
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    # Create the tail label from training-fold targets only.
    y_tail_train = (y_train >= tail_cutoff).astype(int)

    # Prepare categorical features exactly as in the champion.
    for col in cat_features:
        X_train[col] = X_train[col].fillna("__MISSING__").astype(str)
        X_valid[col] = X_valid[col].fillna("__MISSING__").astype(str)

    # Stage 1: classify whether an observation belongs to the high-sales tail.
    classifier = CatBoostClassifier(
        iterations=300,
        learning_rate=0.025,
        depth=4,
        l2_leaf_reg=1,
        boosting_type="Ordered",
        loss_function="Logloss",
        random_seed=42,
        verbose=False,
        thread_count=-1,
    )

    classifier.fit(
        X_train,
        y_tail_train,
        cat_features=cat_features,
    )

    tail_probability = classifier.predict_proba(X_valid)[:, 1]

    # Stage 2a: model non-tail sales.
    non_tail_mask = y_train < tail_cutoff

    non_tail_model = CatBoostRegressor(
        iterations=300,
        learning_rate=0.025,
        depth=4,
        l2_leaf_reg=1,
        boosting_type="Ordered",
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        thread_count=-1,
    )

    non_tail_model.fit(
        X_train.loc[non_tail_mask],
        y_train.loc[non_tail_mask],
        cat_features=cat_features,
    )

    # Stage 2b: model tail sales.
    tail_mask = y_train >= tail_cutoff

    tail_model = CatBoostRegressor(
        iterations=300,
        learning_rate=0.025,
        depth=4,
        l2_leaf_reg=1,
        boosting_type="Ordered",
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        thread_count=-1,
    )

    tail_model.fit(
        X_train.loc[tail_mask],
        y_train.loc[tail_mask],
        cat_features=cat_features,
    )

    non_tail_pred = non_tail_model.predict(X_valid)
    tail_pred = tail_model.predict(X_valid)

    # Soft regime combination avoids a hard classifier boundary.
    pred = (
        (1 - tail_probability) * non_tail_pred
        + tail_probability * tail_pred
    )

    rmse = np.sqrt(mean_squared_error(y_valid, pred))

    tail_models_results.append({
        "fold": fold_num,
        "rmse": rmse,
        "validation_tail_rate": (y_valid >= tail_cutoff).mean(),
    })

tail_models_results = pd.DataFrame(tail_models_results)

print(tail_models_results.to_string(index=False))
print(f"\nMean CV RMSE: {tail_models_results['rmse'].mean():.6f}")
print(f"Std CV RMSE:  {tail_models_results['rmse'].std():.6f}")
print("Champion CV RMSE: 1074.915874")

 fold        rmse  validation_tail_rate
    1 1071.340663              0.104106
    2 1053.370274              0.093109
    3 1082.466843              0.093109
    4 1109.138003              0.102715
    5 1059.616649              0.107117

Mean CV RMSE: 1075.186487
Std CV RMSE:  22.016280
Champion CV RMSE: 1074.915874


In [ ]:
# Reproduce the final Ordered CatBoost champion across the locked folds before freezing the modeling decision.

from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd

final_confirmation = []

for fold_num, (train_idx, valid_idx) in enumerate(folds, start=1):
    X_train = X_cat.iloc[train_idx].copy()
    X_valid = X_cat.iloc[valid_idx].copy()
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    for col in cat_features:
        X_train[col] = X_train[col].fillna("__MISSING__").astype(str)
        X_valid[col] = X_valid[col].fillna("__MISSING__").astype(str)

    model = CatBoostRegressor(
        iterations=300,
        learning_rate=0.025,
        depth=4,
        l2_leaf_reg=1,
        boosting_type="Ordered",
        random_strength=1.0,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        thread_count=-1,
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
    )

    pred = model.predict(X_valid)
    rmse = np.sqrt(mean_squared_error(y_valid, pred))

    final_confirmation.append({
        "fold": fold_num,
        "rmse": rmse,
    })

final_confirmation = pd.DataFrame(final_confirmation)

print(final_confirmation.to_string(index=False))
print(f"\nMean CV RMSE: {final_confirmation['rmse'].mean():.6f}")
print(f"Std CV RMSE:  {final_confirmation['rmse'].std():.6f}")
print("Frozen benchmark: 1074.915874")

 fold        rmse
    1 1065.990810
    2 1053.887357
    3 1085.198997
    4 1107.592529
    5 1061.909675

Mean CV RMSE: 1074.915874
Std CV RMSE:  21.590061
Frozen benchmark: 1074.915874


In [ ]:
# Test whether price_per_kg provides incremental out-of-fold predictive value beyond the frozen CatBoost benchmark.

from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error

X_fe = train.drop(
    columns=["total_sales", "id", "product_category"]
).copy()

X_fe["price_per_kg"] = (
    X_fe["product_price"] / X_fe["product_weight_kg"]
)

cat_features = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format",
]

for col in cat_features:
    X_fe[col] = X_fe[col].fillna("__MISSING__").astype(str)

fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(folds, start=1):
    X_train = X_fe.iloc[train_idx]
    X_valid = X_fe.iloc[valid_idx]
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    model = CatBoostRegressor(
        iterations=300,
        learning_rate=0.025,
        depth=4,
        l2_leaf_reg=1,
        boosting_type="Ordered",
        random_strength=1.0,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        thread_count=-1,
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
    )

    pred = model.predict(X_valid)
    rmse = np.sqrt(mean_squared_error(y_valid, pred))
    fold_scores.append(rmse)

    print(f"Fold {fold}: RMSE = {rmse:.4f}")

print(f"\nMean CV RMSE: {np.mean(fold_scores):.4f}")
print(f"Std CV RMSE:  {np.std(fold_scores):.4f}")
print(f"Frozen benchmark: 1074.9159")
print(f"Change: {1074.9159 - np.mean(fold_scores):+.4f}")

Fold 1: RMSE = 1066.7261
Fold 2: RMSE = 1053.6681
Fold 3: RMSE = 1086.9136
Fold 4: RMSE = 1109.3689
Fold 5: RMSE = 1062.9534

Mean CV RMSE: 1075.9260
Std CV RMSE:  19.9374
Frozen benchmark: 1074.9159
Change: -1.0101


## Targeted Investigation: Residual Structure

The frozen CatBoost model has established a strong validation benchmark
(RMSE = 1074.915874), but substantial errors remain concentrated in the
upper sales tail.

Before modifying the model, we test whether its out-of-fold residuals
contain systematic structure that can be explained by the available
predictors.

This investigation is diagnostic only. No model changes are made until
the residual structure provides evidence for a specific hypothesis.

In [ ]:
# Diagnose whether the frozen CatBoost model's OOF residuals contain systematic structure that could support a residual-correction model.

residual_df = X_train.copy()

residual_df["actual_sales"] = y_train.values
residual_df["oof_prediction"] = cat_oof
residual_df["residual"] = residual_df["actual_sales"] - residual_df["oof_prediction"]
residual_df["abs_residual"] = residual_df["residual"].abs()

print("Residual summary:")
print(residual_df["residual"].describe())

print("\nMean residual:")
print(residual_df["residual"].mean())

print("\nMedian residual:")
print(residual_df["residual"].median())

print("\nResidual correlation with numeric variables:")
numeric_cols = [
    "product_weight_kg",
    "shelf_visibility",
    "product_price",
    "store_age_years",
]

print(
    residual_df[numeric_cols + ["residual"]]
    .corr(numeric_only=True)["residual"]
    .sort_values()
)

print("\nMean residual by categorical variables:")

for col in [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format",
]:
    stats = (
        residual_df.groupby(col, observed=True)["residual"]
        .agg(["mean", "count"])
        .sort_values("count", ascending=False)
    )

    print(f"\n{col}")
    print(stats.head(15))

ValueError: Length of values (6818) does not match length of index (5455)

In [ ]:
# Verify the current notebook objects and identify the full training feature dataframe needed to align the OOF predictions.

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("cat_oof length:", len(cat_oof))

print("\nAvailable key variables:")
for name in [
    "X",
    "X_train_full",
    "X_model",
    "X_train",
    "train_final",
    "train_df",
    "y",
    "y_train",
]:
    print(f"{name}: {'exists' if name in globals() else 'not found'}")

X_train shape: (5455, 12)
y_train shape: (5455,)
cat_oof length: 6818

Available key variables:
X: exists
X_train_full: not found
X_model: not found
X_train: exists
train_final: not found
train_df: not found
y: exists
y_train: exists


In [ ]:
# Verify that X and y contain the full training set and that their row order matches the OOF prediction vector.

print("X shape:", X.shape)
print("y shape:", y.shape)
print("cat_oof length:", len(cat_oof))

print("\nAlignment checks:")
print("X rows == y length:", len(X) == len(y))
print("X rows == cat_oof length:", len(X) == len(cat_oof))

print("\nFirst 5 X indices:")
print(X.index[:5].tolist())

print("\nFirst 5 y indices:")
print(y.index[:5].tolist())

print("\nFirst 5 cat_oof values:")
print(cat_oof[:5])

X shape: (6818, 12)
y shape: (6818,)
cat_oof length: 6818

Alignment checks:
X rows == y length: True
X rows == cat_oof length: True

First 5 X indices:
[0, 1, 2, 3, 4]

First 5 y indices:
[0, 1, 2, 3, 4]

First 5 cat_oof values:
[1349.99520477  720.01623514  739.71352107 4690.96678406 3347.1038525 ]


In [ ]:
# Build the leakage-safe OOF residual dataset and summarize the overall residual behavior.

residual_df = X.copy()

residual_df["actual_sales"] = y.to_numpy()
residual_df["oof_prediction"] = np.asarray(cat_oof)
residual_df["residual"] = (
    residual_df["actual_sales"] - residual_df["oof_prediction"]
)
residual_df["abs_residual"] = residual_df["residual"].abs()

print("Residual dataframe shape:", residual_df.shape)

print("\nResidual summary:")
print(
    residual_df["residual"]
    .describe()
    .round(3)
)

print("\nMean residual:", round(residual_df["residual"].mean(), 6))
print("Median residual:", round(residual_df["residual"].median(), 6))

print("\nLargest underpredictions:")
print(
    residual_df.nlargest(10, "residual")[
        ["actual_sales", "oof_prediction", "residual"]
    ].round(2)
)

print("\nLargest overpredictions:")
print(
    residual_df.nsmallest(10, "residual")[
        ["actual_sales", "oof_prediction", "residual"]
    ].round(2)
)

Residual dataframe shape: (6818, 16)

Residual summary:
count    6818.000
mean       -7.329
std      1075.580
min     -4815.052
25%      -550.343
50%       -63.509
75%       454.677
max      7352.184
Name: residual, dtype: float64

Mean residual: -7.328661
Median residual: -63.508857

Largest underpredictions:
      actual_sales  oof_prediction  residual
5404      12996.82         5644.64   7352.18
4220      12359.04         5817.63   6541.41
900       10130.90         3985.66   6145.24
5950       9999.51         4056.88   5942.63
1555       9388.15         3953.19   5434.96
4511       9072.04         3792.52   5279.52
3444       8617.32         3662.67   4954.65
625        8894.14         3976.19   4917.95
4597       8714.71         3812.34   4902.37
2053       9803.38         5034.16   4769.22

Largest overpredictions:
      actual_sales  oof_prediction  residual
4363       1377.26         6192.31  -4815.05
3216        575.13         4978.94  -4403.81
6685        787.89         5133.

In [ ]:
# Identify categorical groups associated with systematically larger OOF residual magnitude.

categorical_cols = [
    "product_category_clean",
    "fat_content",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format",
]

for col in categorical_cols:
    summary = (
        residual_df
        .groupby(col, observed=True)
        .agg(
            n=("abs_residual", "size"),
            mean_abs_error=("abs_residual", "mean"),
            median_abs_error=("abs_residual", "median"),
            mean_residual=("residual", "mean"),
        )
        .sort_values("mean_abs_error", ascending=False)
    )

    print(col)
    print(summary.round(2))


product_category_clean
                           n  mean_abs_error  median_abs_error  mean_residual
product_category_clean                                                       
seafood                   52          906.91            682.02         101.32
starchy foods            113          884.64            511.03          20.04
snack foods              933          803.80            578.09         -40.83
dairy                    554          789.66            478.34         -14.30
breakfast                 85          784.53            592.01        -131.86
household                740          780.15            542.22         -12.14
fruits and vegetables   1006          774.76            525.80          15.04
hard drinks              169          764.81            541.09         -38.18
canned                   517          747.74            553.09          18.04
frozen foods             675          746.86            472.77           1.88
meat                     339          73

In [ ]:
# Test whether residual magnitude has systematic relationships with the main continuous predictors.

numeric_cols = [
    "product_weight_kg",
    "shelf_visibility",
    "product_price",
    "store_age_years",
]

for col in numeric_cols:
    corr_pearson = residual_df[[col, "abs_residual"]].corr(
        method="pearson"
    ).iloc[0, 1]

    corr_spearman = residual_df[[col, "abs_residual"]].corr(
        method="spearman"
    ).iloc[0, 1]

    print(f"\n{col}")
    print(f"  Pearson correlation : {corr_pearson:.4f}")
    print(f"  Spearman correlation: {corr_spearman:.4f}")

print("\nResidual magnitude by product price quintile:")

residual_df["price_error_bin"] = pd.qcut(
    residual_df["product_price"],
    q=5,
    duplicates="drop"
)

print(
    residual_df.groupby("price_error_bin", observed=True)
    .agg(
        n=("abs_residual", "size"),
        mean_price=("product_price", "mean"),
        mean_abs_error=("abs_residual", "mean"),
        median_abs_error=("abs_residual", "median"),
        mean_residual=("residual", "mean"),
    )
    .round(2)
)


product_weight_kg
  Pearson correlation : 0.0093
  Spearman correlation: 0.0114

shelf_visibility
  Pearson correlation : -0.0878
  Spearman correlation: -0.0737

product_price
  Pearson correlation : 0.4408
  Spearman correlation: 0.4307

store_age_years
  Pearson correlation : -0.0287
  Spearman correlation: -0.0799

Residual magnitude by product price quintile:
                                 n  mean_price  mean_abs_error  \
price_error_bin                                                  
(31.108999999999998, 83.752]  1364       54.46          300.94   
(83.752, 117.848]             1363      101.36          536.11   
(117.848, 158.83]             1365      138.67          732.86   
(158.83, 194.228]             1362      177.28          954.44   
(194.228, 273.04]             1364      230.62         1253.26   

                              median_abs_error  mean_residual  
price_error_bin                                                
(31.108999999999998, 83.752]            2

In [ ]:
# Determine whether residual direction remains predictable within price regimes.

residual_df["price_regime"] = pd.qcut(
    residual_df["product_price"],
    q=5,
    labels=["Q1_low", "Q2", "Q3", "Q4", "Q5_high"],
)

print("Residual direction by price regime:")
print(
    residual_df.groupby("price_regime", observed=True)
    .agg(
        n=("residual", "size"),
        mean_residual=("residual", "mean"),
        median_residual=("residual", "median"),
        mean_abs_residual=("abs_residual", "mean"),
        rmse=("residual", lambda x: np.sqrt(np.mean(x ** 2))),
    )
    .round(3)
)

print("\nResidual direction by price regime × store format:")
print(
    residual_df.groupby(
        ["price_regime", "store_format"],
        observed=True
    )
    .agg(
        n=("residual", "size"),
        mean_residual=("residual", "mean"),
        mean_abs_residual=("abs_residual", "mean"),
    )
    .query("n >= 30")
    .sort_values("mean_residual")
    .round(3)
)

Residual direction by price regime:
                 n  mean_residual  median_residual  mean_abs_residual  \
price_regime                                                            
Q1_low        1364        -36.623          -40.053            300.940   
Q2            1363         -6.780          -58.866            536.114   
Q3            1365          3.423          -58.715            732.860   
Q4            1362         -9.960         -108.351            954.441   
Q5_high       1364         13.285         -114.917           1253.260   

                  rmse  
price_regime            
Q1_low         409.040  
Q2             704.831  
Q3             961.765  
Q4            1250.690  
Q5_high       1621.958  

Residual direction by price regime × store format:
                                     n  mean_residual  mean_abs_residual
price_regime store_format                                               
Q1_low       Flagship Hypermarket  147       -117.945            466.125
Q5_hig

In [ ]:
# Verify the exact columns and categorical-feature specification used by the current X matrix.

print("X columns:")
for i, col in enumerate(X.columns):
    print(f"{i}: {col} | dtype={X[col].dtype}")

print("\nCurrent cat_features:")
print(cat_features)

print("\nCategorical columns actually present in X:")
print([
    col for col in X.columns
    if X[col].dtype == "object"
])

X columns:
0: product_code | dtype=object
1: product_weight_kg | dtype=float64
2: fat_content | dtype=object
3: shelf_visibility | dtype=float64
4: product_category | dtype=object
5: product_price | dtype=float64
6: store_code | dtype=object
7: store_age_years | dtype=int64
8: store_size | dtype=object
9: store_location_tier | dtype=object
10: store_format | dtype=object
11: product_category_clean | dtype=object

Current cat_features:
['product_code', 'fat_content', 'product_category_clean', 'store_code', 'store_size', 'store_location_tier', 'store_format']

Categorical columns actually present in X:
['product_code', 'fat_content', 'product_category', 'store_code', 'store_size', 'store_location_tier', 'store_format', 'product_category_clean']


In [ ]:
# Reconstruct the exact frozen 11-feature modeling matrix for the Huber experiment.

model_features = [
    "product_code",
    "product_weight_kg",
    "fat_content",
    "shelf_visibility",
    "product_price",
    "store_code",
    "store_age_years",
    "store_size",
    "store_location_tier",
    "store_format",
    "product_category_clean",
]

X_model = X[model_features].copy()

cat_features_model = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format",
]

print("X_model shape:", X_model.shape)
print("\nX_model columns:")
print(X_model.columns.tolist())

print("\nCategorical features:")
print(cat_features_model)

print("\nCategorical columns in X_model:")
print(
    X_model.select_dtypes(include="object").columns.tolist()
)

print("\nFeature contract valid:",
      set(cat_features_model).issubset(X_model.columns))

X_model shape: (6818, 11)

X_model columns:
['product_code', 'product_weight_kg', 'fat_content', 'shelf_visibility', 'product_price', 'store_code', 'store_age_years', 'store_size', 'store_location_tier', 'store_format', 'product_category_clean']

Categorical features:
['product_code', 'fat_content', 'product_category_clean', 'store_code', 'store_size', 'store_location_tier', 'store_format']

Categorical columns in X_model:
['product_code', 'fat_content', 'store_code', 'store_size', 'store_location_tier', 'store_format', 'product_category_clean']

Feature contract valid: True


In [ ]:
# Identify any remaining NaN values inside the categorical columns used by the frozen CatBoost feature contract.

cat_nan_check = pd.DataFrame({
"column": cat_features_model,
"dtype": [X_model[c].dtype for c in cat_features_model],
"missing_count": [X_model[c].isna().sum() for c in cat_features_model],
"unique_count": [X_model[c].nunique(dropna=False) for c in cat_features_model],
})

display(cat_nan_check)

print("\nRows containing categorical NaNs:")
display(
X_model[
X_model[cat_features_model].isna().any(axis=1)
][cat_features_model].head(10)
)


,column,dtype,missing_count,unique_count
0,product_code,object,0,1555
1,fat_content,object,0,2
2,product_category_clean,object,0,16
3,store_code,object,0,10
4,store_size,object,1919,4
5,store_location_tier,object,0,3
6,store_format,object,0,4



Rows containing categorical NaNs:


,product_code,fat_content,product_category_clean,store_code,store_size,store_location_tier,store_format
9,PRD-DH2TA3,Low Fat,household,STORE-JOR,NaN,Tier_3,Corner Shop
11,PRD-UX01UP,Low Fat,snack foods,STORE-OYG,NaN,Tier_2,Standard Supermarket
14,PRD-9USY38,Low Fat,snack foods,STORE-DKU,NaN,Tier_2,Standard Supermarket
18,PRD-SUDBCI,Low Fat,snack foods,STORE-DKU,NaN,Tier_2,Standard Supermarket
19,PRD-5QVWC9,Low Fat,household,STORE-OYG,NaN,Tier_2,Standard Supermarket
21,PRD-SB61LG,Regular,fruits and vegetables,STORE-OYG,NaN,Tier_2,Standard Supermarket
27,PRD-0QD58X,Regular,breads,STORE-DKU,NaN,Tier_2,Standard Supermarket
28,PRD-99VH0W,Low Fat,fruits and vegetables,STORE-DKU,NaN,Tier_2,Standard Supermarket
30,PRD-NAMSQE,Regular,baking goods,STORE-JOR,NaN,Tier_3,Corner Shop
33,PRD-HZXGUJ,Low Fat,dairy,STORE-DKU,NaN,Tier_2,Standard Supermarket


In [ ]:
# Aim: Restore the established categorical missing-value representation without changing any feature values or the modeling hypothesis.

X_model = X_model.copy()

X_model["store_size"] = (
X_model["store_size"]
.astype(object)
.where(X_model["store_size"].notna(), "Missng")
)

# Verify the categorical feature contract again.

cat_nan_check = pd.DataFrame({
"column": cat_features_model,
"dtype": [X_model[c].dtype for c in cat_features_model],
"missing_count": [X_model[c].isna().sum() for c in cat_features_model],
"unique_count": [X_model[c].nunique(dropna=False) for c in cat_features_model],
})

display(cat_nan_check)

print(
"\nRemaining categorical NaNs:",
X_model[cat_features_model].isna().sum().sum()
)

print(
"store_size 'Missng' count:",
(X_model["store_size"] == "Missng").sum()
)


,column,dtype,missing_count,unique_count
0,product_code,object,0,1555
1,fat_content,object,0,2
2,product_category_clean,object,0,16
3,store_code,object,0,10
4,store_size,object,0,4
5,store_location_tier,object,0,3
6,store_format,object,0,4



Remaining categorical NaNs: 0
store_size 'Missng' count: 1919


In [ ]:
# Aim: Test whether robust Huber losses improve the frozen CatBoost model under the exact locked 5-fold validation protocol.

from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd

kf_huber = KFold(
n_splits=5,
shuffle=True,
random_state=42
)

huber_tests = {
"RMSE_control": "RMSE",
"Huber_500": "Huber:delta=500",
"Huber_1000": "Huber:delta=1000",
"Huber_1500": "Huber:delta=1500",
}

huber_results = []

for name, loss in huber_tests.items():
    fold_rmses = []

for fold, (train_idx, val_idx) in enumerate(
    kf_huber.split(X_model, y),
    start=1
):

    X_fold_train = X_model.iloc[train_idx]
    X_fold_val = X_model.iloc[val_idx]

    y_fold_train = y.iloc[train_idx]
    y_fold_val = y.iloc[val_idx]

    model = CatBoostRegressor(
        iterations=300,
        learning_rate=0.025,
        depth=4,
        l2_leaf_reg=1,
        random_strength=1.0,
        loss_function=loss,
        boosting_type="Ordered",
        random_seed=42,
        verbose=False
    )

    model.fit(
        X_fold_train,
        y_fold_train,
        cat_features=cat_features_model
    )

    predictions = model.predict(X_fold_val)

    rmse = np.sqrt(
        mean_squared_error(
            y_fold_val,
            predictions
        )
    )

    fold_rmses.append(rmse)

    print(
        f"{name} | Fold {fold}: {rmse:.6f}"
    )

mean_rmse = np.mean(fold_rmses)
std_rmse = np.std(fold_rmses, ddof=1)

huber_results.append({
    "model": name,
    "mean_rmse": mean_rmse,
    "std_rmse": std_rmse,
    "change_vs_champion": mean_rmse - 1074.915874
})

print(
    f"{name} | "
    f"Mean: {mean_rmse:.6f} | "
    f"Std: {std_rmse:.6f}\n"
)

huber_results_df = (
pd.DataFrame(huber_results)
.sort_values("mean_rmse")
.reset_index(drop=True)
)


print("\nHuber objective comparison:")
display(huber_results_df.round(6))


Huber_1500 | Fold 1: 1073.175349
Huber_1500 | Fold 2: 1051.795973
Huber_1500 | Fold 3: 1084.109659
Huber_1500 | Fold 4: 1109.640062
Huber_1500 | Fold 5: 1066.113942
Huber_1500 | Mean: 1076.966997 | Std: 21.704220


Huber objective comparison:


,model,mean_rmse,std_rmse,change_vs_champion
0,Huber_1500,1076.966997,21.70422,2.051123


In [ ]:
# Aim: Determine which Huber experiments actually completed before deciding whether another training run is necessary.

print("Completed Huber experiments:")
print(huber_results)

print("\nNumber of completed experiments:", len(huber_results))

print("\nModels completed:")
print([result["model"] for result in huber_results])


Completed Huber experiments:
[{'model': 'Huber_1500', 'mean_rmse': np.float64(1076.9669968325118), 'std_rmse': np.float64(21.704219787429924), 'change_vs_champion': np.float64(2.051122832511737)}]

Number of completed experiments: 1

Models completed:
['Huber_1500']


In [ ]:
# Aim: Complete the Huber-loss hypothesis by testing the remaining thresholds while reproducing the frozen RMSE control.

remaining_huber_tests = {
"RMSE_control": "RMSE",
"Huber_500": "Huber:delta=500",
"Huber_1000": "Huber:delta=1000"
}

remaining_results = []

for name, loss in remaining_huber_tests.items():
    fold_rmses = []

for fold, (train_idx, val_idx) in enumerate(
    kf_huber.split(X_model, y),
    start=1
):
    X_fold_train = X_model.iloc[train_idx]
    X_fold_val = X_model.iloc[val_idx]

    y_fold_train = y.iloc[train_idx]
    y_fold_val = y.iloc[val_idx]

    model = CatBoostRegressor(
        iterations=300,
        learning_rate=0.025,
        depth=4,
        l2_leaf_reg=1,
        random_strength=1.0,
        loss_function=loss,
        boosting_type="Ordered",
        random_seed=42,
        verbose=False
    )

    model.fit(
        X_fold_train,
        y_fold_train,
        cat_features=cat_features_model
    )

    predictions = model.predict(X_fold_val)

    rmse = np.sqrt(
        mean_squared_error(y_fold_val, predictions)
    )

    fold_rmses.append(rmse)

    print(f"{name} | Fold {fold}: {rmse:.6f}")

mean_rmse = np.mean(fold_rmses)
std_rmse = np.std(fold_rmses, ddof=1)

remaining_results.append({
    "model": name,
    "mean_rmse": mean_rmse,
    "std_rmse": std_rmse,
    "change_vs_champion": mean_rmse - 1074.915874
})

print(
    f"{name} | Mean: {mean_rmse:.6f} | "
    f"Std: {std_rmse:.6f}\n"
)

remaining_huber_df = (
pd.DataFrame(remaining_results)
.sort_values("mean_rmse")
.reset_index(drop=True)
)

print("Completed remaining Huber experiments:")
display(remaining_huber_df.round(6))


Huber_1000 | Fold 1: 1079.302814
Huber_1000 | Fold 2: 1053.186435
Huber_1000 | Fold 3: 1085.868098
Huber_1000 | Fold 4: 1109.471250
Huber_1000 | Fold 5: 1068.167840
Huber_1000 | Mean: 1079.199287 | Std: 20.971628

Completed remaining Huber experiments:


,model,mean_rmse,std_rmse,change_vs_champion
0,Huber_1000,1079.199287,20.971628,4.283413


In [ ]:
# Aim: Complete the Huber experiment with the remaining delta=500 test and reproduce the frozen RMSE control.

final_huber_tests = {
"RMSE_control": "RMSE",
"Huber_500": "Huber:delta=500"
}

final_huber_results = []

for name, loss in final_huber_tests.items():
    fold_rmses = []


for fold, (train_idx, val_idx) in enumerate(
    kf_huber.split(X_model, y),
    start=1
):
    X_fold_train = X_model.iloc[train_idx]
    X_fold_val = X_model.iloc[val_idx]

    y_fold_train = y.iloc[train_idx]
    y_fold_val = y.iloc[val_idx]

    model = CatBoostRegressor(
        iterations=300,
        learning_rate=0.025,
        depth=4,
        l2_leaf_reg=1,
        random_strength=1.0,
        loss_function=loss,
        boosting_type="Ordered",
        random_seed=42,
        verbose=False
    )

    model.fit(
        X_fold_train,
        y_fold_train,
        cat_features=cat_features_model
    )

    predictions = model.predict(X_fold_val)

    rmse = np.sqrt(
        mean_squared_error(y_fold_val, predictions)
    )

    fold_rmses.append(rmse)

    print(f"{name} | Fold {fold}: {rmse:.6f}")

mean_rmse = np.mean(fold_rmses)
std_rmse = np.std(fold_rmses, ddof=1)

final_huber_results.append({
    "model": name,
    "mean_rmse": mean_rmse,
    "std_rmse": std_rmse,
    "change_vs_champion": mean_rmse - 1074.915874
})

print(
    f"{name} | Mean: {mean_rmse:.6f} | "
    f"Std: {std_rmse:.6f}\n"
)


final_huber_df = (
pd.DataFrame(final_huber_results)
.sort_values("mean_rmse")
.reset_index(drop=True)
)

print("Final Huber threshold check:")
display(final_huber_df.round(6))


Huber_500 | Fold 1: 1081.733918
Huber_500 | Fold 2: 1054.861350
Huber_500 | Fold 3: 1086.268969
Huber_500 | Fold 4: 1111.024994
Huber_500 | Fold 5: 1072.790524
Huber_500 | Mean: 1081.335951 | Std: 20.493054

Final Huber threshold check:


,model,mean_rmse,std_rmse,change_vs_champion
0,Huber_500,1081.335951,20.493054,6.420077


In [ ]:
# Aim: Compare the target distribution and upper-tail concentration across the exact locked validation folds.

fold_profile = []

for fold, (train_idx, val_idx) in enumerate(
    kf_huber.split(X_model, y),
    start=1
):
    y_fold = y.iloc[val_idx]

    fold_profile.append({
        "fold": fold,
        "n": len(y_fold),
        "mean_sales": y_fold.mean(),
        "median_sales": y_fold.median(),
        "std_sales": y_fold.std(),
        "max_sales": y_fold.max(),
        "q90_sales": y_fold.quantile(0.90),
        "q95_sales": y_fold.quantile(0.95),
        "q99_sales": y_fold.quantile(0.99),
        "top_10_pct_mean": y_fold[
            y_fold >= y.quantile(0.90)
        ].mean(),
        "top_10_pct_count": (
            y_fold >= y.quantile(0.90)
        ).sum(),
        "top_1_pct_count": (
            y_fold >= y.quantile(0.99)
        ).sum()
    })

fold_profile_df = pd.DataFrame(fold_profile)

display(fold_profile_df.round(3))

print("\nOverall target:")
print(f"Mean:   {y.mean():.3f}")
print(f"Median: {y.median():.3f}")
print(f"Std:    {y.std():.3f}")
print(f"Max:    {y.max():.3f}")
print(f"Q90:    {y.quantile(0.90):.3f}")
print(f"Q95:    {y.quantile(0.95):.3f}")
print(f"Q99:    {y.quantile(0.99):.3f}")

,fold,n,mean_sales,median_sales,std_sales,max_sales,q90_sales,q95_sales,q99_sales,top_10_pct_mean,top_10_pct_count,top_1_pct_count
0,1,1364,2221.086,1839.350,1716.519,9803.38,4647.479,5703.774,7373.280,5852.311,142,15
1,2,1364,2133.376,1800.625,1651.595,9999.51,4466.315,5326.737,7038.474,5729.562,127,13
2,3,1364,2152.429,1773.825,1686.092,12359.04,4441.266,5502.650,7142.750,5914.226,127,13
3,4,1363,2181.864,1755.030,1722.794,12996.82,4571.606,5603.301,7386.820,5840.023,140,16
4,5,1363,2185.041,1790.520,1712.645,10477.97,4608.854,5543.884,7172.898,5759.232,146,12



Overall target:
Mean:   2174.757
Median: 1790.890
Std:    1697.895
Max:    12996.820
Q90:    4557.841
Q95:    5557.191
Q99:    7261.389


In [ ]:
# Aim: Determine whether the validation folds differ materially
# in product-price distribution, which may explain differences
# in residual difficulty.

fold_price_profile = []

for fold, (train_idx, val_idx) in enumerate(
    kf_huber.split(X_model, y),
    start=1
):
    X_fold = X_model.iloc[val_idx]
    y_fold = y.iloc[val_idx]

    price = X_fold["product_price"]

    fold_price_profile.append({
        "fold": fold,
        "mean_price": price.mean(),
        "median_price": price.median(),
        "std_price": price.std(),
        "max_price": price.max(),
        "q90_price": price.quantile(0.90),
        "q95_price": price.quantile(0.95),
        "q99_price": price.quantile(0.99),
        "high_price_25pct": (price >= X_model["product_price"].quantile(0.75)).sum(),
        "high_price_10pct": (price >= X_model["product_price"].quantile(0.90)).sum(),
        "high_price_5pct": (price >= X_model["product_price"].quantile(0.95)).sum()
    })

fold_price_profile_df = pd.DataFrame(fold_price_profile)

display(
    fold_price_profile_df.round(3)
)

,fold,mean_price,median_price,std_price,max_price,q90_price,q95_price,q99_price,high_price_25pct,high_price_10pct,high_price_5pct
0,1,142.382,144.575,62.807,271.82,232.635,251.604,264.040,354,146,77
1,2,137.986,133.770,62.117,271.12,227.988,248.698,263.304,326,126,61
2,3,139.579,140.960,63.180,269.61,233.537,251.445,264.428,329,145,75
3,4,141.624,141.420,63.016,273.04,234.214,251.476,263.221,362,150,74
4,5,140.799,143.910,60.914,269.90,225.896,247.016,261.012,334,115,54


In [ ]:
# Aim: Determine whether the locked validation folds differ materially
# in store composition, especially exposure to high-error stores.

fold_store_profile = []

for fold, (train_idx, val_idx) in enumerate(
    kf_huber.split(X_model, y),
    start=1
):
    X_fold = X_model.iloc[val_idx].copy()

    store_counts = X_fold["store_code"].value_counts()

    row = {"fold": fold}

    for store in sorted(X_model["store_code"].dropna().unique()):
        row[f"{store}_n"] = store_counts.get(store, 0)
        row[f"{store}_share"] = (
            store_counts.get(store, 0) / len(X_fold)
        )

    fold_store_profile.append(row)

fold_store_profile_df = pd.DataFrame(fold_store_profile)

display(
    fold_store_profile_df.round(4)
)

,fold,STORE-7WS_n,STORE-7WS_share,STORE-89Z_n,STORE-89Z_share,STORE-9RG_n,STORE-9RG_share,STORE-AGY_n,STORE-AGY_share,STORE-DKU_n,STORE-DKU_share,STORE-HL7_n,STORE-HL7_share,STORE-JOR_n,STORE-JOR_share,STORE-OYG_n,STORE-OYG_share,STORE-T5G_n,STORE-T5G_share,STORE-YLW_n,STORE-YLW_share
0,1,140,0.1026,153,0.1122,160,0.1173,156,0.1144,141,0.1034,145,0.1063,93,0.0682,143,0.1048,84,0.0616,149,0.1092
1,2,151,0.1107,142,0.1041,138,0.1012,146,0.1070,149,0.1092,155,0.1136,94,0.0689,151,0.1107,91,0.0667,147,0.1078
2,3,147,0.1078,124,0.0909,144,0.1056,165,0.1210,152,0.1114,151,0.1107,81,0.0594,161,0.1180,79,0.0579,160,0.1173
3,4,150,0.1101,157,0.1152,157,0.1152,145,0.1064,145,0.1064,131,0.0961,90,0.0660,154,0.1130,76,0.0558,158,0.1159
4,5,160,0.1174,152,0.1115,153,0.1123,136,0.0998,149,0.1093,160,0.1174,81,0.0594,135,0.0990,97,0.0712,140,0.1027


In [ ]:
# Aim: Determine whether the folds differ in the joint distribution
# of product price and store, especially whether Fold 4 contains
# unusually expensive products within high-error stores.

fold_store_price_profile = []

for fold, (train_idx, val_idx) in enumerate(
    kf_huber.split(X_model, y),
    start=1
):
    X_fold = X_model.iloc[val_idx].copy()

    grouped = (
        X_fold
        .groupby("store_code")["product_price"]
        .agg(
            mean_price="mean",
            median_price="median",
            q90_price=lambda x: x.quantile(0.90),
            high_price_25pct=lambda x: (
                x >= X_model["product_price"].quantile(0.75)
            ).sum(),
            count="count"
        )
        .reset_index()
    )

    grouped["fold"] = fold

    fold_store_price_profile.append(grouped)

fold_store_price_profile_df = pd.concat(
    fold_store_price_profile,
    ignore_index=True
)

display(
    fold_store_price_profile_df
    .sort_values(["store_code", "fold"])
    .round(3)
)

,store_code,mean_price,median_price,q90_price,high_price_25pct,count,fold
0,STORE-7WS,140.584,142.545,223.317,32,140,1
10,STORE-7WS,137.065,127.150,225.970,32,151,2
20,STORE-7WS,139.880,127.480,226.716,31,147,3
30,STORE-7WS,139.484,143.050,228.702,40,150,4
40,STORE-7WS,140.228,146.205,226.524,40,160,5
1,STORE-89Z,138.120,139.320,238.136,34,153,1
11,STORE-89Z,149.009,146.680,236.523,44,142,2
21,STORE-89Z,141.107,146.260,231.244,31,124,3
31,STORE-89Z,133.257,122.500,225.022,37,157,4
41,STORE-89Z,140.188,142.875,230.447,38,152,5


In [ ]:
# Aim: Determine whether model error changes systematically with
# how many times a product appears in the dataset.

product_counts = X_model["product_code"].value_counts()

residual_product_repeat = residual_df.copy()

residual_product_repeat["product_obs_count"] = (
    residual_product_repeat["product_code"]
    .map(product_counts)
)

repeat_profile = (
    residual_product_repeat
    .groupby("product_obs_count")
    .agg(
        n=("residual", "size"),
        mean_residual=("residual", "mean"),
        mean_abs_error=("abs_residual", "mean"),
        rmse=("residual", lambda x: np.sqrt(np.mean(x ** 2))),
        median_abs_error=("abs_residual", "median"),
        mean_price=("product_price", "mean")
    )
    .reset_index()
)

display(
    repeat_profile.round(3)
)

,product_obs_count,n,mean_residual,mean_abs_error,rmse,median_abs_error,mean_price
0,1,46,-161.349,809.585,1181.023,560.955,146.602
1,2,280,10.988,822.209,1132.138,603.613,146.842
2,3,753,6.685,798.940,1138.891,535.916,143.334
3,4,1576,-31.129,748.045,1064.271,513.388,140.270
4,5,1755,1.334,776.924,1093.717,528.429,141.220
5,6,1440,-16.958,694.778,1021.484,459.356,133.659
6,7,679,32.056,757.899,1059.398,483.641,138.683
7,8,280,-10.255,793.477,1096.804,538.214,164.234
8,9,9,176.253,245.236,355.231,140.113,48.061


In [ ]:
# Aim: Test whether deviations from a product's own typical
# characteristics are associated with model error magnitude.

product_relative_diagnostics = residual_df.copy()

product_relative_diagnostics["product_price_median"] = (
    product_relative_diagnostics
    .groupby("product_code")["product_price"]
    .transform("median")
)

product_relative_diagnostics["price_deviation_from_product"] = (
    product_relative_diagnostics["product_price"]
    - product_relative_diagnostics["product_price_median"]
)

product_relative_diagnostics["price_abs_deviation_from_product"] = (
    product_relative_diagnostics["price_deviation_from_product"].abs()
)

product_relative_diagnostics["product_visibility_median"] = (
    product_relative_diagnostics
    .groupby("product_code")["shelf_visibility"]
    .transform("median")
)

product_relative_diagnostics["visibility_abs_deviation_from_product"] = (
    product_relative_diagnostics["shelf_visibility"]
    - product_relative_diagnostics["product_visibility_median"]
).abs()

product_relative_diagnostics["product_weight_median"] = (
    product_relative_diagnostics
    .groupby("product_code")["product_weight_kg"]
    .transform("median")
)

product_relative_diagnostics["weight_abs_deviation_from_product"] = (
    product_relative_diagnostics["product_weight_kg"]
    - product_relative_diagnostics["product_weight_median"]
).abs()

diagnostic_correlations = (
    product_relative_diagnostics[
        [
            "price_abs_deviation_from_product",
            "visibility_abs_deviation_from_product",
            "weight_abs_deviation_from_product",
            "abs_residual"
        ]
    ]
    .corr(method="spearman")["abs_residual"]
    .drop("abs_residual")
)

display(
    diagnostic_correlations.to_frame("spearman_corr_with_abs_residual")
    .round(4)
)

,spearman_corr_with_abs_residual
price_abs_deviation_from_product,0.1180
visibility_abs_deviation_from_product,-0.1948
weight_abs_deviation_from_product,-0.0020


In [ ]:
# Aim: Determine whether within-product price and visibility
# deviations show a stable, monotonic relationship with model error.

for feature in [
    "price_abs_deviation_from_product",
    "visibility_abs_deviation_from_product"
]:
    temp = product_relative_diagnostics[
        [feature, "residual", "abs_residual"]
    ].copy()

    temp["deviation_bin"] = pd.qcut(
        temp[feature],
        q=5,
        duplicates="drop"
    )

    profile = (
        temp
        .groupby("deviation_bin", observed=True)
        .agg(
            n=("residual", "size"),
            mean_deviation=(feature, "mean"),
            mean_residual=("residual", "mean"),
            mean_abs_error=("abs_residual", "mean"),
            rmse=("residual", lambda x: np.sqrt(np.mean(x ** 2)))
        )
        .reset_index()
    )

    print(f"\n{feature}")
    display(profile.round(3))


price_abs_deviation_from_product


,deviation_bin,n,mean_deviation,mean_residual,mean_abs_error,rmse
0,"(-0.001, 0.245]",1364,0.054,-35.413,736.168,1063.427
1,"(0.245, 0.939]",1363,0.581,-11.311,646.645,958.317
2,"(0.939, 1.87]",1366,1.375,-13.393,695.983,998.027
3,"(1.87, 3.223]",1361,2.482,-13.315,731.393,1021.089
4,"(3.223, 13.26]",1364,5.004,36.782,967.233,1302.175



visibility_abs_deviation_from_product


,deviation_bin,n,mean_deviation,mean_residual,mean_abs_error,rmse
0,"(-0.001, 0.00015]",1368,0.000,-30.612,813.842,1113.679
1,"(0.00015, 0.00065]",1375,0.000,55.371,847.906,1166.737
2,"(0.00065, 0.0016]",1367,0.001,-26.242,836.007,1148.114
3,"(0.0016, 0.00658]",1344,0.003,-43.018,844.465,1146.302
4,"(0.00658, 0.184]",1364,0.047,6.939,435.459,741.691


In [ ]:
# Aim: Test whether fold-safe within-product price and visibility
# deviations provide predictive signal beyond the frozen CatBoost model.

feature_experiment_results = []

for fold, (train_idx, val_idx) in enumerate(
    kf_huber.split(X_model, y),
    start=1
):
    X_train_fold = X_model.iloc[train_idx].copy()
    X_val_fold = X_model.iloc[val_idx].copy()

    y_train_fold = y.iloc[train_idx]
    y_val_fold = y.iloc[val_idx]

    # Product-level medians learned from training data only
    product_price_median = (
        X_train_fold
        .groupby("product_code")["product_price"]
        .median()
    )

    product_visibility_median = (
        X_train_fold
        .groupby("product_code")["shelf_visibility"]
        .median()
    )

    global_price_median = X_train_fold["product_price"].median()
    global_visibility_median = X_train_fold["shelf_visibility"].median()

    # Map training medians
    train_price_median = (
        X_train_fold["product_code"]
        .map(product_price_median)
        .fillna(global_price_median)
    )

    train_visibility_median = (
        X_train_fold["product_code"]
        .map(product_visibility_median)
        .fillna(global_visibility_median)
    )

    # Map validation medians
    val_price_median = (
        X_val_fold["product_code"]
        .map(product_price_median)
        .fillna(global_price_median)
    )

    val_visibility_median = (
        X_val_fold["product_code"]
        .map(product_visibility_median)
        .fillna(global_visibility_median)
    )

    # Candidate features
    X_train_candidate = X_train_fold.copy()
    X_val_candidate = X_val_fold.copy()

    X_train_candidate["product_price_abs_dev"] = (
        X_train_fold["product_price"] - train_price_median
    ).abs()

    X_val_candidate["product_price_abs_dev"] = (
        X_val_fold["product_price"] - val_price_median
    ).abs()

    X_train_candidate["visibility_abs_dev"] = (
        X_train_fold["shelf_visibility"] - train_visibility_median
    ).abs()

    X_val_candidate["visibility_abs_dev"] = (
        X_val_fold["shelf_visibility"] - val_visibility_median
    ).abs()

    candidate_features = cat_features_model

    # Add engineered numeric features to the feature list
    candidate_features = candidate_features + [
        "product_price_abs_dev",
        "visibility_abs_dev"
    ]

    model = CatBoostRegressor(
        iterations=300,
        learning_rate=0.025,
        depth=4,
        l2_leaf_reg=1,
        random_strength=1.0,
        loss_function="RMSE",
        boosting_type="Ordered",
        random_seed=42,
        verbose=False
    )

    model.fit(
        X_train_candidate,
        y_train_fold,
        cat_features=candidate_features[:len(cat_features_model)]
    )

    pred = model.predict(X_val_candidate)

    rmse = np.sqrt(np.mean((y_val_fold - pred) ** 2))

    feature_experiment_results.append({
        "fold": fold,
        "rmse": rmse
    })

feature_experiment_df = pd.DataFrame(feature_experiment_results)

display(
    feature_experiment_df.round(6)
)

print(
    "Mean RMSE:",
    feature_experiment_df["rmse"].mean()
)

print(
    "Change vs frozen champion:",
    feature_experiment_df["rmse"].mean() - 1074.915874
)

,fold,rmse
0,1,1065.848436
1,2,1054.412855
2,3,1086.961193
3,4,1107.670058
4,5,1062.917103


Mean RMSE: 1075.5619292971403
Δ vs frozen champion: 0.6460552971402649


In [ ]:
# Aim: Test whether allowing CatBoost to construct second-order
# categorical target-statistic interactions improves predictive
# performance while keeping the entire frozen architecture unchanged.

ctr_experiment_results = []

for fold, (train_idx, val_idx) in enumerate(
    kf_huber.split(X_model, y),
    start=1
):
    X_train_fold = X_model.iloc[train_idx].copy()
    X_val_fold = X_model.iloc[val_idx].copy()

    y_train_fold = y.iloc[train_idx]
    y_val_fold = y.iloc[val_idx]

    model = CatBoostRegressor(
        iterations=300,
        learning_rate=0.025,
        depth=4,
        l2_leaf_reg=1,
        random_strength=1.0,
        max_ctr_complexity=2,
        loss_function="RMSE",
        boosting_type="Ordered",
        random_seed=42,
        verbose=False
    )

    model.fit(
        X_train_fold,
        y_train_fold,
        cat_features=cat_features_model
    )

    pred = model.predict(X_val_fold)

    rmse = np.sqrt(
        np.mean((y_val_fold - pred) ** 2)
    )

    ctr_experiment_results.append({
        "fold": fold,
        "rmse": rmse
    })

ctr_experiment_df = pd.DataFrame(
    ctr_experiment_results
)

display(
    ctr_experiment_df.round(6)
)

ctr_mean_rmse = ctr_experiment_df["rmse"].mean()

print(
    f"Mean RMSE: {ctr_mean_rmse:.6f}"
)

print(
    f"Change vs frozen champion: "
    f"{ctr_mean_rmse - 1074.915874:.6f}"
)

,fold,rmse
0,1,1066.458184
1,2,1052.598265
2,3,1085.386724
3,4,1108.400225
4,5,1062.269398


Mean RMSE: 1075.022559
Δ vs frozen champion: 0.106685


In [ ]:
# Aim: Test whether depth 5 becomes effective in the low-iteration
# regime that produced the current CatBoost champion.

depth5_results = []

for fold, (train_idx, val_idx) in enumerate(
    kf_huber.split(X_model, y),
    start=1
):
    X_train_fold = X_model.iloc[train_idx].copy()
    X_val_fold = X_model.iloc[val_idx].copy()

    y_train_fold = y.iloc[train_idx]
    y_val_fold = y.iloc[val_idx]

    model = CatBoostRegressor(
        iterations=300,
        learning_rate=0.025,
        depth=5,
        l2_leaf_reg=1,
        random_strength=1.0,
        loss_function="RMSE",
        boosting_type="Ordered",
        random_seed=42,
        verbose=False
    )

    model.fit(
        X_train_fold,
        y_train_fold,
        cat_features=cat_features_model
    )

    pred = model.predict(X_val_fold)

    rmse = np.sqrt(
        np.mean((y_val_fold - pred) ** 2)
    )

    depth5_results.append({
        "fold": fold,
        "rmse": rmse
    })

depth5_experiment_df = pd.DataFrame(
    depth5_results
)

display(
    depth5_experiment_df.round(6)
)

depth5_mean_rmse = depth5_experiment_df["rmse"].mean()

print(
    f"Mean RMSE: {depth5_mean_rmse:.6f}"
)

print(
    f"Change vs frozen champion: "
    f"{depth5_mean_rmse - 1074.915874:.6f}"
)

,fold,rmse
0,1,1065.186671
1,2,1051.482174
2,3,1086.612254
3,4,1108.814997
4,5,1062.461002


Mean RMSE: 1074.911420
Δ vs frozen champion: -0.004454


In [ ]:
# Aim: Investigate whether product_code contains systematic
# prefix/suffix/character-position structure that could represent
# latent product families beyond product_category_clean.

product_code_structure = pd.DataFrame({
    "product_code": X["product_code"].astype(str),
    "sales": y.values
})

product_code_structure["code_prefix_1"] = (
    product_code_structure["product_code"].str[0]
)

product_code_structure["code_prefix_2"] = (
    product_code_structure["product_code"].str[:2]
)

product_code_structure["code_prefix_3"] = (
    product_code_structure["product_code"].str[:3]
)

product_code_structure["code_suffix_1"] = (
    product_code_structure["product_code"].str[-1]
)

product_code_structure["code_suffix_2"] = (
    product_code_structure["product_code"].str[-2:]
)

product_code_structure["code_length"] = (
    product_code_structure["product_code"].str.len()
)

display(
    product_code_structure.head(10)
)

print("Unique prefix-1:", product_code_structure["code_prefix_1"].nunique())
print("Unique prefix-2:", product_code_structure["code_prefix_2"].nunique())
print("Unique prefix-3:", product_code_structure["code_prefix_3"].nunique())
print("Unique suffix-1:", product_code_structure["code_suffix_1"].nunique())
print("Unique suffix-2:", product_code_structure["code_suffix_2"].nunique())
print("Unique code lengths:", product_code_structure["code_length"].unique())

,product_code,sales,code_prefix_1,code_prefix_2,code_prefix_3,code_suffix_1,code_suffix_2,code_length
0,PRD-PRFP9S,1764.98,P,PR,PRD,S,9S,10
1,PRD-PXXK71,342.13,P,PR,PRD,1,71,10
2,PRD-V5MOIJ,378.85,P,PR,PRD,J,IJ,10
3,PRD-UN5Z3J,5595.72,P,PR,PRD,J,3J,10
4,PRD-6RDQYB,2375.36,P,PR,PRD,B,YB,10
5,PRD-E6VA05,842.75,P,PR,PRD,5,05,10
6,PRD-MHLD93,4512.70,P,PR,PRD,3,93,10
7,PRD-EF6Y39,2373.55,P,PR,PRD,9,39,10
8,PRD-IAZMTU,2037.05,P,PR,PRD,U,TU,10
9,PRD-DH2TA3,213.93,P,PR,PRD,3,A3,10


Unique prefix-1: 1
Unique prefix-2: 1
Unique prefix-3: 1
Unique suffix-1: 36
Unique suffix-2: 883
Unique code lengths: [10]


In [ ]:
# Aim: Determine whether individual characters inside the variable
# portion of product_code contain systematic target or category structure.

code_structure = pd.DataFrame({
    "product_code": X["product_code"].astype(str),
    "sales": y.values,
    "category": X["product_category_clean"].values
})

# Remove the invariant "PRD-" prefix
code_structure["code_body"] = (
    code_structure["product_code"].str[4:]
)

print("Code body length distribution:")
print(code_structure["code_body"].str.len().value_counts().sort_index())

# Extract each character position
for i in range(6):
    code_structure[f"char_{i+1}"] = (
        code_structure["code_body"].str[i]
    )

char_results = []

for i in range(1, 7):
    col = f"char_{i}"

    grouped = (
        code_structure
        .groupby(col)["sales"]
        .agg(["count", "mean", "std"])
    )

    overall_mean = code_structure["sales"].mean()

    # Between-group weighted variance
    between_var = (
        (
            grouped["count"]
            * (grouped["mean"] - overall_mean) ** 2
        ).sum()
        / len(code_structure)
    )

    total_var = code_structure["sales"].var()

    eta_squared = between_var / total_var

    char_results.append({
        "position": i,
        "unique_values": code_structure[col].nunique(),
        "largest_group": grouped["count"].max(),
        "smallest_group": grouped["count"].min(),
        "eta_squared": eta_squared
    })

char_results_df = pd.DataFrame(char_results)

display(
    char_results_df.round(6)
)

Code body length distribution:
code_body
6    6818
Name: count, dtype: int64


,position,unique_values,largest_group,smallest_group,eta_squared
0,1,36,265,137,0.009347
1,2,36,238,112,0.009736
2,3,36,265,103,0.012123
3,4,36,266,123,0.006391
4,5,36,252,147,0.010803
5,6,36,262,120,0.011292


In [ ]:
# Aim: Test whether combinations of two product-code characters
# contain stronger target structure than individual characters.

pair_results = []

for i in range(1, 7):
    for j in range(i + 1, 7):

        col_i = f"char_{i}"
        col_j = f"char_{j}"

        pair_col = f"pair_{i}_{j}"

        code_structure[pair_col] = (
            code_structure[col_i]
            + code_structure[col_j]
        )

        grouped = (
            code_structure
            .groupby(pair_col)["sales"]
            .agg(["count", "mean"])
        )

        overall_mean = code_structure["sales"].mean()

        between_var = (
            (
                grouped["count"]
                * (grouped["mean"] - overall_mean) ** 2
            ).sum()
            / len(code_structure)
        )

        total_var = code_structure["sales"].var()

        eta_squared = between_var / total_var

        pair_results.append({
            "positions": f"{i}+{j}",
            "unique_pairs": code_structure[pair_col].nunique(),
            "smallest_group": grouped["count"].min(),
            "largest_group": grouped["count"].max(),
            "eta_squared": eta_squared
        })

pair_results_df = (
    pd.DataFrame(pair_results)
    .sort_values("eta_squared", ascending=False)
    .reset_index(drop=True)
)

display(
    pair_results_df.round(6)
)

,positions,unique_pairs,smallest_group,largest_group,eta_squared
0,1+5,915,1,33,0.296635
1,4+5,915,1,25,0.290368
2,1+4,915,1,32,0.286633
3,1+2,906,1,32,0.282802
4,1+3,927,1,26,0.281083
5,2+5,923,1,27,0.275793
6,3+5,912,1,43,0.275071
7,2+4,897,1,27,0.273478
8,2+3,901,1,25,0.271567
9,5+6,883,1,27,0.271007


In [ ]:
# Aim: Determine whether the large pairwise product-code eta-squared
# values represent genuine structure or are mainly caused by the
# very high number of tiny groups.

pair_diagnostic_results = []

rng = np.random.default_rng(42)

shuffled_sales = rng.permutation(
    code_structure["sales"].to_numpy()
)

for row in pair_results_df.itertuples(index=False):

    pair_label = f"pair_{row.positions.replace('+', '_')}"

    grouped = (
        code_structure
        .groupby(pair_label)["sales"]
        .agg(["count", "mean"])
    )

    overall_mean = code_structure["sales"].mean()

    between_var = (
        (
            grouped["count"]
            * (grouped["mean"] - overall_mean) ** 2
        ).sum()
        / len(code_structure)
    )

    total_var = code_structure["sales"].var()

    observed_eta = between_var / total_var

    # Same grouping, but with sales randomly shuffled.
    shuffled_df = pd.DataFrame({
        "group": code_structure[pair_label].to_numpy(),
        "sales": shuffled_sales
    })

    shuffled_grouped = (
        shuffled_df
        .groupby("group")["sales"]
        .agg(["count", "mean"])
    )

    shuffled_between_var = (
        (
            shuffled_grouped["count"]
            * (shuffled_grouped["mean"] - shuffled_sales.mean()) ** 2
        ).sum()
        / len(shuffled_df)
    )

    shuffled_total_var = shuffled_sales.var()

    shuffled_eta = (
        shuffled_between_var / shuffled_total_var
    )

    # Category purity within the pair groups.
    category_counts = (
        code_structure
        .groupby(pair_label)["category"]
        .value_counts()
    )

    pair_group_sizes = (
        code_structure[pair_label]
        .value_counts()
    )

    purity_sum = 0.0

    for group_name, group_size in pair_group_sizes.items():
        counts = category_counts.loc[group_name]

        purity = counts.max() / group_size

        purity_sum += group_size * purity

    weighted_purity = (
        purity_sum / len(code_structure)
    )

    pair_diagnostic_results.append({
        "positions": row.positions,
        "unique_pairs": row.unique_pairs,
        "eta_squared": observed_eta,
        "permuted_eta_squared": shuffled_eta,
        "eta_excess": observed_eta - shuffled_eta,
        "weighted_category_purity": weighted_purity
    })

pair_diagnostic_df = (
    pd.DataFrame(pair_diagnostic_results)
    .sort_values("eta_squared", ascending=False)
    .reset_index(drop=True)
)

display(
    pair_diagnostic_df.round(6)
)

,positions,unique_pairs,eta_squared,permuted_eta_squared,eta_excess,weighted_category_purity
0,1+5,915,0.296635,0.134379,0.162256,0.697712
1,4+5,915,0.290368,0.133208,0.157160,0.692578
2,1+4,915,0.286633,0.146727,0.139906,0.690378
3,1+2,906,0.282802,0.132375,0.150427,0.687738
4,1+3,927,0.281083,0.134289,0.146794,0.693312
5,2+5,923,0.275793,0.128680,0.147113,0.692872
6,3+5,912,0.275071,0.137217,0.137854,0.695219
7,2+4,897,0.273478,0.135315,0.138163,0.678351
8,2+3,901,0.271567,0.128660,0.142906,0.684365
9,5+6,883,0.271007,0.130693,0.140314,0.667791


In [ ]:
# Aim: Determine whether the strongest product-code character pair
# contains information beyond full product identity and product category.

best_pair = "pair_1_5"

# 1. Full product-code target structure
product_grouped = (
    code_structure
    .groupby("product_code")["sales"]
    .agg(["count", "mean"])
)

overall_mean = code_structure["sales"].mean()

product_between_var = (
    (
        product_grouped["count"]
        * (product_grouped["mean"] - overall_mean) ** 2
    ).sum()
    / len(code_structure)
)

product_eta_squared = (
    product_between_var
    / code_structure["sales"].var()
)

# 2. Full product-code permutation benchmark

product_perm_df = pd.DataFrame({
    "product_code": code_structure["product_code"].to_numpy(),
    "sales": shuffled_sales
})

product_perm_grouped = (
    product_perm_df
    .groupby("product_code")["sales"]
    .agg(["count", "mean"])
)

product_perm_between_var = (
    (
        product_perm_grouped["count"]
        * (
            product_perm_grouped["mean"]
            - shuffled_sales.mean()
        ) ** 2
    ).sum()
    / len(product_perm_df)
)

product_permuted_eta = (
    product_perm_between_var
    / shuffled_sales.var()
)

# ---------------------------------------------------------
# 3. Category target structure
# ---------------------------------------------------------

category_grouped = (
    code_structure
    .groupby("category")["sales"]
    .agg(["count", "mean"])
)

category_between_var = (
    (
        category_grouped["count"]
        * (
            category_grouped["mean"]
            - overall_mean
        ) ** 2
    ).sum()
    / len(code_structure)
)

category_eta_squared = (
    category_between_var
    / code_structure["sales"].var()
)

# 4. Print comparison

comparison_df = pd.DataFrame({
    "grouping": [
        "product_category",
        "product_code",
        "char_1+char_5"
    ],
    "eta_squared": [
        category_eta_squared,
        product_eta_squared,
        pair_diagnostic_df.loc[
            pair_diagnostic_df["positions"] == "1+5",
            "eta_squared"
        ].iloc[0]
    ],
    "permuted_eta_squared": [
        np.nan,
        product_permuted_eta,
        pair_diagnostic_df.loc[
            pair_diagnostic_df["positions"] == "1+5",
            "permuted_eta_squared"
        ].iloc[0]
    ]
})

display(
    comparison_df.round(6)
)

,grouping,eta_squared,permuted_eta_squared
0,product_category,0.005490,NaN
1,product_code,0.465068,0.229999
2,char_1+char_5,0.296635,0.134379


In [ ]:
# Aim: Measure how strongly a product's other observed sales predict
# the current row's sales, using a strict leave-one-out product mean.

product_signal = pd.DataFrame({
    "product_code": X["product_code"].values,
    "sales": y.values
})

product_stats = (
    product_signal
    .groupby("product_code")["sales"]
    .agg(["count", "sum"])
)

product_signal["product_obs_count"] = (
    product_signal["product_code"]
    .map(product_stats["count"])
)

product_signal["other_product_count"] = (
    product_signal["product_obs_count"] - 1
)

product_signal["other_product_sales"] = (
    product_signal["product_code"]
    .map(product_stats["sum"])
    - product_signal["sales"]
)

product_signal["loo_product_mean"] = np.where(
    product_signal["other_product_count"] > 0,
    (
        product_signal["other_product_sales"]
        / product_signal["other_product_count"]
    ),
    np.nan
)

# Only rows with at least one other observation of the product
repeated_product_signal = (
    product_signal[
        product_signal["other_product_count"] > 0
    ]
    .copy()
)

print(
    "Rows with another observation of the same product:",
    len(repeated_product_signal)
)

print(
    "Rows without another observation:",
    len(product_signal)
    - len(repeated_product_signal)
)

print(
    "Median other observations:",
    repeated_product_signal[
        "other_product_count"
    ].median()
)

print(
    "Mean other observations:",
    repeated_product_signal[
        "other_product_count"
    ].mean()
)

print(
    "Pearson correlation:",
    repeated_product_signal[
        ["sales", "loo_product_mean"]
    ].corr()
    .iloc[0, 1]
)

print(
    "Spearman correlation:",
    repeated_product_signal[
        ["sales", "loo_product_mean"]
    ].corr(method="spearman")
    .iloc[0, 1]
)

display(
    repeated_product_signal[
        [
            "product_code",
            "sales",
            "other_product_count",
            "loo_product_mean"
        ]
    ]
    .head(15)
)

Rows with another observation of the same product: 6772
Rows without another observation: 46
Median other observations: 4.0
Mean other observations: 3.963378617838157
Pearson correlation: 0.4261621683630575
Spearman correlation: 0.466129282210878


,product_code,sales,other_product_count,loo_product_mean
0,PRD-PRFP9S,1764.98,4,1257.275000
1,PRD-PXXK71,342.13,4,1259.660000
2,PRD-V5MOIJ,378.85,5,419.148000
3,PRD-UN5Z3J,5595.72,3,2581.150000
4,PRD-6RDQYB,2375.36,3,1937.826667
5,PRD-E6VA05,842.75,5,502.890000
6,PRD-MHLD93,4512.70,5,3042.642000
7,PRD-EF6Y39,2373.55,5,2544.156000
8,PRD-IAZMTU,2037.05,3,1688.386667
9,PRD-DH2TA3,213.93,3,585.790000


In [ ]:
# Aim: Test whether a strictly fold-safe product target statistic
# provides predictive information beyond CatBoost's native product_code
# representation.

product_te_results = []

for fold, (train_idx, val_idx) in enumerate(
    kf_huber.split(X_model, y),
    start=1
):
    X_train_fold = X_model.iloc[train_idx].copy()
    X_val_fold = X_model.iloc[val_idx].copy()

    y_train_fold = y.iloc[train_idx].copy()
    y_val_fold = y.iloc[val_idx].copy()

    # Build product statistics using TRAINING DATA ONLY

    train_product_stats = (
        pd.DataFrame({
            "product_code": X_train_fold["product_code"].values,
            "target": y_train_fold.values
        })
        .groupby("product_code")["target"]
        .agg(["count", "sum"])
    )

    train_global_mean = y_train_fold.mean()

    # Training feature: strict leave-one-out product mean

    train_product_counts = (
        X_train_fold["product_code"]
        .map(train_product_stats["count"])
    )

    train_product_sums = (
        X_train_fold["product_code"]
        .map(train_product_stats["sum"])
    )

    train_loo_mean = (
        (train_product_sums - y_train_fold.values)
        / (train_product_counts - 1)
    )

    train_loo_mean = (
        train_loo_mean
        .replace([np.inf, -np.inf], np.nan)
        .fillna(train_global_mean)
    )

    # Validation feature: training-fold product mean only

    val_product_mean = (
        X_val_fold["product_code"]
        .map(
            train_product_stats["sum"]
            / train_product_stats["count"]
        )
        .fillna(train_global_mean)
    )

    # Add feature

    X_train_te = X_train_fold.copy()
    X_val_te = X_val_fold.copy()

    X_train_te["product_target_mean"] = (
        train_loo_mean.to_numpy()
    )

    X_val_te["product_target_mean"] = (
        val_product_mean.to_numpy()
    )

    # Frozen champion architecture

    model = CatBoostRegressor(
        iterations=300,
        learning_rate=0.025,
        depth=4,
        l2_leaf_reg=1,
        random_strength=1.0,
        loss_function="RMSE",
        boosting_type="Ordered",
        random_seed=42,
        verbose=False
    )

    model.fit(
        X_train_te,
        y_train_fold,
        cat_features=cat_features_model
    )

    pred = model.predict(X_val_te)

    rmse = np.sqrt(
        np.mean((y_val_fold - pred) ** 2)
    )

    product_te_results.append({
        "fold": fold,
        "rmse": rmse
    })

product_te_experiment_df = pd.DataFrame(
    product_te_results
)

display(
    product_te_experiment_df.round(6)
)

product_te_mean_rmse = (
    product_te_experiment_df["rmse"].mean()
)

print(
    f"Mean RMSE: {product_te_mean_rmse:.6f}"
)

print(
    f"Change vs frozen champion: "
    f"{product_te_mean_rmse - 1074.915874:.6f}"
)

,fold,rmse
0,1,1073.768714
1,2,1052.562347
2,3,1099.225890
3,4,1120.278293
4,5,1078.403451


Mean RMSE: 1084.847739
Δ vs frozen champion: 9.931865


In [ ]:
# Aim: Test whether repeated products retain residual persistence after
# the frozen CatBoost model's leakage-safe OOF predictions.

residual_diagnostic = pd.DataFrame({
    "product_code": X["product_code"].values,
    "sales": y.values,
    "oof_prediction": cat_oof
})

residual_diagnostic["residual"] = (
    residual_diagnostic["sales"]
    - residual_diagnostic["oof_prediction"]
)

# Product-level residual statistics
product_residual_stats = (
    residual_diagnostic
    .groupby("product_code")["residual"]
    .agg(["count", "sum"])
)

residual_diagnostic["product_count"] = (
    residual_diagnostic["product_code"]
    .map(product_residual_stats["count"])
)

residual_diagnostic["other_residual_count"] = (
    residual_diagnostic["product_count"] - 1
)

residual_diagnostic["other_residual_sum"] = (
    residual_diagnostic["product_code"]
    .map(product_residual_stats["sum"])
    - residual_diagnostic["residual"]
)

residual_diagnostic["loo_product_residual_mean"] = np.where(
    residual_diagnostic["other_residual_count"] > 0,
    (
        residual_diagnostic["other_residual_sum"]
        / residual_diagnostic["other_residual_count"]
    ),
    np.nan
)

repeated_residual = residual_diagnostic[
    residual_diagnostic["other_residual_count"] > 0
].copy()

print(
    "Rows with repeated products:",
    len(repeated_residual)
)

print(
    "Rows without another product observation:",
    len(residual_diagnostic)
    - len(repeated_residual)
)

print(
    "Median other observations:",
    repeated_residual[
        "other_residual_count"
    ].median()
)

print(
    "Mean other observations:",
    repeated_residual[
        "other_residual_count"
    ].mean()
)

print(
    "Pearson correlation:",
    repeated_residual[
        ["residual", "loo_product_residual_mean"]
    ].corr().iloc[0, 1]
)

print(
    "Spearman correlation:",
    repeated_residual[
        ["residual", "loo_product_residual_mean"]
    ].corr(method="spearman").iloc[0, 1]
)

display(
    repeated_residual[
        [
            "product_code",
            "residual",
            "other_residual_count",
            "loo_product_residual_mean"
        ]
    ].head(15)
)

Rows with repeated products: 6772
Rows without another product observation: 46
Median other observations: 4.0
Mean other observations: 3.963378617838157
Pearson correlation: 0.010173956645180768
Spearman correlation: 0.01785420463345341


,product_code,residual,other_residual_count,loo_product_residual_mean
0,PRD-PRFP9S,414.984795,4,-147.692810
1,PRD-PXXK71,-377.886235,4,387.422039
2,PRD-V5MOIJ,-360.863521,5,-16.031789
3,PRD-UN5Z3J,904.753216,3,-440.083920
4,PRD-6RDQYB,-971.743853,3,-1236.298050
5,PRD-E6VA05,106.909563,5,-150.589154
6,PRD-MHLD93,-1082.360242,5,175.629915
7,PRD-EF6Y39,326.170095,5,353.531069
8,PRD-IAZMTU,629.954190,3,-16.883405
9,PRD-DH2TA3,-39.772239,3,-621.376669


In [ ]:
# Aim: Identify whether product category changes sales behavior or
# residual difficulty within comparable price regimes.

interaction_diagnostic = pd.DataFrame({
    "product_category": X["product_category_clean"].values,
    "product_price": X["product_price"].values,
    "sales": y.values,
    "prediction": cat_oof
})

interaction_diagnostic["residual"] = (
    interaction_diagnostic["sales"]
    - interaction_diagnostic["prediction"]
)

interaction_diagnostic["abs_residual"] = (
    interaction_diagnostic["residual"].abs()
)

# Create price quintiles using the full training distribution.
interaction_diagnostic["price_quintile"] = pd.qcut(
    interaction_diagnostic["product_price"],
    q=5,
    labels=["Q1", "Q2", "Q3", "Q4", "Q5"],
    duplicates="drop"
)

interaction_table = (
    interaction_diagnostic
    .groupby(
        ["price_quintile", "product_category"],
        observed=True
    )
    .agg(
        n=("sales", "size"),
        mean_sales=("sales", "mean"),
        mean_prediction=("prediction", "mean"),
        mean_residual=("residual", "mean"),
        mae=("abs_residual", "mean"),
        rmse=(
            "residual",
            lambda x: np.sqrt(np.mean(x ** 2))
        )
    )
    .reset_index()
)

# Keep only reasonably supported groups so tiny cells do not dominate.
interaction_table_supported = interaction_table[
    interaction_table["n"] >= 40
].copy()

display(
    interaction_table_supported
    .sort_values(
        ["price_quintile", "rmse"],
        ascending=[True, False]
    )
    .round(2)
)

,price_quintile,product_category,n,mean_sales,mean_prediction,mean_residual,mae,rmse
0,Q1,baking goods,133,874.37,895.40,-21.03,335.55,477.04
9,Q1,household,123,806.67,844.03,-37.36,322.76,446.91
4,Q1,dairy,117,845.96,828.90,17.06,304.83,427.42
3,Q1,canned,95,946.45,929.24,17.20,312.13,406.63
6,Q1,fruits and vegetables,177,877.63,907.06,-29.42,316.49,404.94
13,Q1,snack foods,133,741.48,852.08,-110.61,300.98,399.24
5,Q1,frozen foods,173,818.68,867.71,-49.03,281.07,392.25
8,Q1,health and hygiene,92,732.57,795.02,-62.45,273.18,390.28
14,Q1,soft drinks,98,843.77,831.36,12.42,278.46,371.15
10,Q1,meat,64,911.59,926.04,-14.46,278.81,350.09


In [ ]:
# Aim: Verify Optuna is available and confirm the exact data/model
# objects that will be used for the controlled hyperparameter search.

import optuna
import numpy as np
import pandas as pd

print("Optuna version:", optuna.__version__)
print("\nData:")
print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nCategorical features:")
print(cat_features_model)

print("\nLocked benchmark:")
print("Frozen CatBoost CV RMSE:", 1074.915874)

Optuna version: 5.0.0

Data:
X shape: (6818, 12)
y shape: (6818,)

Categorical features:
['product_code', 'fat_content', 'product_category_clean', 'store_code', 'store_size', 'store_location_tier', 'store_format']

Locked benchmark:
Frozen CatBoost CV RMSE: 1074.915874


In [ ]:
# Aim: Build a leakage-safe Optuna objective using the exact locked
# 5-fold validation protocol and a targeted CatBoost search space.

from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

# Locked validation protocol
optuna_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Search objective
def objective(trial):

    params = {
        "iterations": trial.suggest_int(
            "iterations",
            100,
            500
        ),
        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.05,
            log=True
        ),
        "depth": trial.suggest_int(
            "depth",
            3,
            6
        ),
        "l2_leaf_reg": trial.suggest_float(
            "l2_leaf_reg",
            0.5,
            10.0,
            log=True
        ),
        "random_strength": trial.suggest_float(
            "random_strength",
            0.0,
            2.0
        ),
        "bagging_temperature": trial.suggest_float(
            "bagging_temperature",
            0.0,
            3.0
        ),
        "border_count": trial.suggest_int(
            "border_count",
            32,
            255
        ),
        "loss_function": "RMSE",
        "boosting_type": "Ordered",
        "random_seed": 42,
        "verbose": False
    }

    fold_rmse = []

    for fold, (train_idx, val_idx) in enumerate(
        optuna_cv.split(X, y),
        start=1
    ):

        X_train_fold = X.iloc[train_idx].copy()
        X_val_fold = X.iloc[val_idx].copy()

        y_train_fold = y.iloc[train_idx]
        y_val_fold = y.iloc[val_idx]

        # Protect categorical columns from missing values
        for col in cat_features_model:
            X_train_fold[col] = (
                X_train_fold[col]
                .astype(object)
                .where(
                    X_train_fold[col].notna(),
                    "Missng"
                )
            )

            X_val_fold[col] = (
                X_val_fold[col]
                .astype(object)
                .where(
                    X_val_fold[col].notna(),
                    "Missng"
                )
            )

        model = CatBoostRegressor(**params)

        model.fit(
            X_train_fold,
            y_train_fold,
            cat_features=cat_features_model
        )

        predictions = model.predict(X_val_fold)

        rmse = np.sqrt(
            mean_squared_error(
                y_val_fold,
                predictions
            )
        )

        fold_rmse.append(rmse)

    return np.mean(fold_rmse)

In [ ]:
# Aim: Run a small Optuna pilot to test whether targeted hyperparameter
# optimization can beat the frozen CatBoost benchmark.

study = optuna.create_study(
    direction="minimize",
    study_name="catboost_dsn_pilot"
)

study.optimize(
    objective,
    n_trials=20,
    show_progress_bar=True
)

print("\nBest CV RMSE:", study.best_value)
print("\nBest parameters:")
for param, value in study.best_params.items():
    print(f"{param}: {value}")

[I 2026-09-26 08:43:29,417] A new study created in memory with name: catboost_dsn_pilot


  0%|          | 0/20 [00:00<?, ?it/s]

[W 2026-09-26 08:43:42,695] Trial 0 failed with parameters: {'iterations': 413, 'learning_rate': 0.043934254103759805, 'depth': 6, 'l2_leaf_reg': 0.8909571114179319, 'random_strength': 1.1371605485391334, 'bagging_temperature': 0.03395375161710057, 'border_count': 181} because of the following error: CatBoostError('Bad value for num_feature[non_default_doc_idx=0,feature_idx=4]="Frozen Foods": Cannot convert \'Frozen Foods\' to float').
Traceback (most recent call last):
  File "_catboost.pyx", line 2606, in _catboost.get_float_feature
  File "_catboost.pyx", line 1290, in _catboost._FloatOrNan
  File "_catboost.pyx", line 1039, in _catboost._FloatOrNanFromString
TypeError: Cannot convert 'Frozen Foods' to float

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "c:\Users\Administrator\Documents\python training and projects\env\lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(tri

CatBoostError: Bad value for num_feature[non_default_doc_idx=0,feature_idx=4]="Frozen Foods": Cannot convert 'Frozen Foods' to float

In [ ]:
# Aim: Reconstruct and verify the exact 11-feature matrix used by
# the frozen CatBoost benchmark before restarting Optuna.

optuna_features = [
    "product_code",
    "product_weight_kg",
    "fat_content",
    "shelf_visibility",
    "product_price",
    "store_code",
    "store_age_years",
    "store_size",
    "store_location_tier",
    "store_format",
    "product_category_clean"
]

X_optuna = X[optuna_features].copy()

# Ensure categorical columns are valid CatBoost categorical inputs.
for col in cat_features_model:
    X_optuna[col] = (
        X_optuna[col]
        .astype(object)
        .where(
            X_optuna[col].notna(),
            "Missng"
        )
    )

print("X_optuna shape:", X_optuna.shape)
print("\nColumns:")
print(X_optuna.columns.tolist())

print("\nCategorical columns:")
print(cat_features_model)

print("\nNon-model columns excluded:")
print(sorted(set(X.columns) - set(optuna_features)))

X_optuna shape: (6818, 11)

Columns:
['product_code', 'product_weight_kg', 'fat_content', 'shelf_visibility', 'product_price', 'store_code', 'store_age_years', 'store_size', 'store_location_tier', 'store_format', 'product_category_clean']

Categorical columns:
['product_code', 'fat_content', 'product_category_clean', 'store_code', 'store_size', 'store_location_tier', 'store_format']

Non-model columns excluded:
['product_category']


In [17]:
# Aim: Rebuild the Optuna objective using the exact 11-feature
# modeling matrix and the locked 5-fold validation protocol.

def objective(trial):

    params = {
        "iterations": trial.suggest_int(
            "iterations",
            100,
            500
        ),
        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.05,
            log=True
        ),
        "depth": trial.suggest_int(
            "depth",
            3,
            6
        ),
        "l2_leaf_reg": trial.suggest_float(
            "l2_leaf_reg",
            0.5,
            10.0,
            log=True
        ),
        "random_strength": trial.suggest_float(
            "random_strength",
            0.0,
            2.0
        ),
        "bagging_temperature": trial.suggest_float(
            "bagging_temperature",
            0.0,
            3.0
        ),
        "border_count": trial.suggest_int(
            "border_count",
            32,
            255
        ),
        "loss_function": "RMSE",
        "boosting_type": "Ordered",
        "random_seed": 42,
        "verbose": False
    }

    fold_rmse = []

    for train_idx, val_idx in optuna_cv.split(X_optuna, y):

        X_train_fold = X_optuna.iloc[train_idx].copy()
        X_val_fold = X_optuna.iloc[val_idx].copy()

        y_train_fold = y.iloc[train_idx]
        y_val_fold = y.iloc[val_idx]

        model = CatBoostRegressor(**params)

        model.fit(
            X_train_fold,
            y_train_fold,
            cat_features=cat_features_model
        )

        predictions = model.predict(X_val_fold)

        rmse = np.sqrt(
            mean_squared_error(
                y_val_fold,
                predictions
            )
        )

        fold_rmse.append(rmse)

    return np.mean(fold_rmse)

In [ ]:
# Run a fresh 20-trial Optuna pilot against the locked CatBoost
# benchmark using the corrected objective.

study = optuna.create_study(
    direction="minimize",
    study_name="catboost_dsn_pilot_v2"
)

study.optimize(
    objective,
    n_trials=20,
    show_progress_bar=True
)

print("Optuna Pilot Result:")
print(f"\nBest CV RMSE: {study.best_value:.6f}")
print(f"Frozen benchmark: {1074.915874:.6f}")
print(f"Improvement: {1074.915874 - study.best_value:.6f}")

print("\nBest parameters:")
for param, value in study.best_params.items():
    print(f"{param}: {value}")

[I 2026-09-26 10:16:26,093] A new study created in memory with name: catboost_dsn_pilot_v2


  0%|          | 0/20 [00:01<?, ?it/s]

[I 2026-09-26 10:31:41,643] Trial 0 finished with value: 1077.7937183713525 and parameters: {'iterations': 489, 'learning_rate': 0.04583539962253089, 'depth': 3, 'l2_leaf_reg': 1.2684217608603878, 'random_strength': 1.1427469900405236, 'bagging_temperature': 0.6640346731521712, 'border_count': 74}. Best is trial 0 with value: 1077.7937183713525.
[I 2026-09-26 10:32:59,618] Trial 1 finished with value: 1130.4282627122066 and parameters: {'iterations': 173, 'learning_rate': 0.010888897461288228, 'depth': 3, 'l2_leaf_reg': 0.8100705751334922, 'random_strength': 0.9093429295689655, 'bagging_temperature': 2.700650926053693, 'border_count': 239}. Best is trial 0 with value: 1077.7937183713525.
[I 2026-09-26 10:41:57,429] Trial 2 finished with value: 1075.87637598156 and parameters: {'iterations': 413, 'learning_rate': 0.016629649103980428, 'depth': 5, 'l2_leaf_reg': 6.135588057570418, 'random_strength': 0.9035677106353348, 'bagging_temperature': 0.8455315736497948, 'border_count': 44}. Best 

In [18]:
# Identify which OOF prediction arrays are currently available
# in the notebook so we can continue without rerunning expensive models.

oof_candidates = {
    name: (
        type(value).__name__,
        getattr(value, "shape", None),
        len(value) if hasattr(value, "__len__") else None
    )
    for name, value in globals().items()
    if (
        ("oof" in name.lower() or "pred" in name.lower())
        and not name.startswith("_")
    )
}

for name, info in sorted(oof_candidates.items()):
    print(f"{name}: type={info[0]}, shape={info[1]}, length={info[2]}")

log_predictions: type=ndarray, shape=(1363,), length=1363
mean_prediction: type=ndarray, shape=(1363,), length=1363
predictions: type=ndarray, shape=(1364,), length=1364


In [ ]:
# Safely inspect existing 6818-row prediction/residual arrays
# without modifying or rerunning any model.

for name, value in list(globals().items()):
    if name.startswith("_"):
        continue

    shape = getattr(value, "shape", None)

    if shape == (6818,) or shape == (6818, 1):
        print(
            f"{name}: "
            f"type={type(value).__name__}, "
            f"shape={shape}"
        )

y: type=Series, shape=(6818,)
onehot_xgb_oof: type=ndarray, shape=(6818,)
onehot_xgb_residual: type=ndarray, shape=(6818,)


In [21]:
# Reconstruct the frozen CatBoost OOF predictions so they can
# be compared directly with the completed one-hot XGBoost OOF model.

from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error

# Exact frozen CatBoost feature set
ensemble_features = [
    "product_code",
    "product_weight_kg",
    "fat_content",
    "shelf_visibility",
    "product_price",
    "store_code",
    "store_age_years",
    "store_size",
    "store_location_tier",
    "store_format",
    "product_category_clean",
]

ensemble_cat_features = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format",
]

ensemble_cat_X = X[ensemble_features].copy()

# CatBoost requires categorical missing values to be explicit
for col in ensemble_cat_features:
    ensemble_cat_X[col] = (
        ensemble_cat_X[col]
        .astype(object)
        .where(
            ensemble_cat_X[col].notna(),
            "Missng"
        )
    )

cat_oof = np.zeros(len(ensemble_cat_X))
cat_fold_rmses = []

for fold, (train_idx, val_idx) in enumerate(
    kf.split(ensemble_cat_X),
    start=1
):
    X_tr = ensemble_cat_X.iloc[train_idx]
    X_va = ensemble_cat_X.iloc[val_idx]

    y_tr = y.iloc[train_idx]
    y_va = y.iloc[val_idx]

    model = CatBoostRegressor(
        boosting_type="Ordered",
        iterations=300,
        learning_rate=0.025,
        depth=4,
        l2_leaf_reg=1,
        random_strength=1.0,
        loss_function="RMSE",
        random_seed=42,
        verbose=False
    )

    model.fit(
        X_tr,
        y_tr,
        cat_features=ensemble_cat_features
    )

    cat_oof[val_idx] = model.predict(X_va)

    fold_rmse = np.sqrt(
        mean_squared_error(
            y_va,
            cat_oof[val_idx]
        )
    )

    cat_fold_rmses.append(fold_rmse)

    print(
        f"Fold {fold}: RMSE = {fold_rmse:.6f}"
    )

cat_oof_rmse = np.sqrt(
    mean_squared_error(y, cat_oof)
)

cat_residual = y.to_numpy() - cat_oof

onehot_xgb_residual = (
    y.to_numpy() - onehot_xgb_oof
)

residual_corr = np.corrcoef(
    cat_residual,
    onehot_xgb_residual
)[0, 1]

print("\n" + "=" * 55)
print("Ensemble Baseline: Catboost + One-hot XGBoost")
print(f"CatBoost OOF RMSE:       {cat_oof_rmse:.6f}")
print(f"One-hot XGB OOF RMSE:    {onehot_xgb_rmse:.6f}")
print(f"Residual correlation:    {residual_corr:.6f}")

Fold 1: RMSE = 1065.990810
Fold 2: RMSE = 1053.887357
Fold 3: RMSE = 1085.198997
Fold 4: RMSE = 1107.592529
Fold 5: RMSE = 1061.909675


NameError: name 'onehot_xgb_oof' is not defined

In [ ]:
# Find the best non-negative two-model blend between the
# frozen CatBoost and one-hot XGBoost OOF predictions.

blend_results = []

for cat_weight in np.linspace(0, 1, 1001):

    xgb_weight = 1.0 - cat_weight

    blended_pred = (
        cat_weight * cat_oof
        + xgb_weight * onehot_xgb_oof
    )

    blend_rmse = np.sqrt(
        mean_squared_error(
            y,
            blended_pred
        )
    )

    blend_results.append(
        {
            "cat_weight": cat_weight,
            "xgb_weight": xgb_weight,
            "rmse": blend_rmse
        }
    )

blend_results = pd.DataFrame(blend_results)

best_blend = (
    blend_results
    .loc[blend_results["rmse"].idxmin()]
)

print("NON-NEGATIVE CATBOOST + ONE-HOT XGBOOST BLEND")
print(f"CatBoost weight: {best_blend['cat_weight']:.3f}")
print(f"XGBoost weight:  {best_blend['xgb_weight']:.3f}")
print(f"Blend OOF RMSE:  {best_blend['rmse']:.6f}")
print(f"CatBoost OOF:    {cat_oof_rmse:.6f}")
print(
    f"Improvement:"
    f"{cat_oof_rmse - best_blend['rmse']:.6f}"
)

NON-NEGATIVE CATBOOST + ONE-HOT XGBOOST BLEND
CatBoost weight: 1.000
XGBoost weight:  0.000
Blend OOF RMSE:  1075.086399
CatBoost OOF:    1075.086399
Improvement:0.000000


In [ ]:
# Inspect whether the first two product-code body characters
# form meaningful groups before using product_broad_type in a model.

product_broad_type_diagnostic = pd.DataFrame({
    "product_code": X["product_code"],
    "product_broad_type": (
        X["product_code"]
        .astype(str)
        .str.split("-")
        .str[1]
        .str[:2]
    ),
    "total_sales": y
})

broad_type_summary = (
    product_broad_type_diagnostic
    .groupby("product_broad_type")
    .agg(
        n=("total_sales", "size"),
        mean_sales=("total_sales", "mean"),
        median_sales=("total_sales", "median"),
        std_sales=("total_sales", "std")
    )
    .sort_values("n", ascending=False)
)

print(
    "Unique product_broad_type values:",
    product_broad_type_diagnostic["product_broad_type"].nunique()
)

display(
    broad_type_summary.round(2)
)

Unique product_broad_type values: 906


,n,mean_sales,median_sales,std_sales
product_broad_type,,,,
MH,32,1416.75,896.39,1391.34
1Z,29,2497.53,2313.30,1803.11
U0,28,3049.62,2832.42,1902.88
OE,27,2943.18,2550.14,2126.52
3I,23,3440.55,3742.35,1855.47
...,...,...,...,...
EJ,1,1841.84,1841.84,NaN
QN,1,5011.59,5011.59,NaN
KM,1,3298.79,3298.79,NaN


In [ ]:
# Aim: Rebuild Model C using the exact locked modeling features,
# replacing full product_code with product_broad_type.

from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd

# Start from the exact competition modeling feature set
broad_features = [
    "product_code",
    "product_weight_kg",
    "fat_content",
    "shelf_visibility",
    "product_price",
    "store_code",
    "store_age_years",
    "store_size",
    "store_location_tier",
    "store_format",
    "product_category_clean"
]

X_broad = X[broad_features].copy()

# Create compressed product-code representation
X_broad["product_broad_type"] = (
    X_broad["product_code"]
    .astype(str)
    .str.split("-")
    .str[1]
    .str[:2]
)

# Replace full product identity with broad type
X_broad = X_broad.drop(columns=["product_code"])

cat_features_broad = [
    "product_broad_type",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format"
]

# Match our established CatBoost missing-category handling
for col in cat_features_broad:
    X_broad[col] = (
        X_broad[col]
        .astype(object)
        .where(X_broad[col].notna(), "Missng")
    )

print("Model C shape:", X_broad.shape)
print("Categorical columns:", cat_features_broad)
print("Remaining raw product_code:", "product_code" in X_broad.columns)
print("Remaining raw product_category:", "product_category" in X_broad.columns)

Model C shape: (6818, 11)
Categorical columns: ['product_broad_type', 'fat_content', 'product_category_clean', 'store_code', 'store_size', 'store_location_tier', 'store_format']
Remaining raw product_code: False
Remaining raw product_category: False


In [ ]:
# Aim: Calculate Model C OOF RMSE and residual diversity
# using the predictions already generated before the metric error.

broad_fold_rmse = []

for fold, (train_idx, val_idx) in enumerate(
    kf.split(X_broad, y), start=1
):
    fold_rmse = np.sqrt(
        mean_squared_error(
            y.iloc[val_idx],
            broad_oof[val_idx]
        )
    )

    broad_fold_rmse.append(fold_rmse)

broad_oof_rmse = np.sqrt(
    mean_squared_error(
        y,
        broad_oof
    )
)

broad_residual = y.to_numpy() - broad_oof

print("Model C fold RMSE:")
for i, score in enumerate(broad_fold_rmse, start=1):
    print(f"F{i}: {score:.6f}")

print(f"\nModel C fold-mean RMSE: {np.mean(broad_fold_rmse):.6f}")
print(f"Model C OOF RMSE:       {broad_oof_rmse:.6f}")

print(f"\nChampion OOF RMSE:       {cat_oof_rmse:.6f}")

print(
    f"Residual correlation:    "
    f"{np.corrcoef(cat_residual, broad_residual)[0, 1]:.6f}"
)

Model C fold RMSE:
F1: 1066.754221
F2: 2697.602379
F3: 2733.819773
F4: 2779.635449
F5: 2775.861277

Model C fold-mean RMSE: 2410.734620
Model C OOF RMSE:       2502.736241

Champion OOF RMSE:       1075.086399
Residual correlation:    0.589656


In [ ]:
# Recompute Model C OOF predictions cleanly across all five locked folds
# after fixing the scikit-learn RMSE compatibility issue.

broad_oof = np.zeros(len(X_broad))
broad_fold_rmse = []

for fold, (train_idx, val_idx) in enumerate(
    kf.split(X_broad, y), start=1
):
    X_tr = X_broad.iloc[train_idx].copy()
    X_va = X_broad.iloc[val_idx].copy()

    y_tr = y.iloc[train_idx]
    y_va = y.iloc[val_idx]

    model = CatBoostRegressor(
        boosting_type="Ordered",
        iterations=300,
        learning_rate=0.025,
        depth=4,
        l2_leaf_reg=1,
        random_strength=1.0,
        loss_function="RMSE",
        random_seed=42,
        verbose=False
    )

    model.fit(
        X_tr,
        y_tr,
        cat_features=cat_features_broad
    )

    pred = model.predict(X_va)

    broad_oof[val_idx] = pred

    fold_rmse = np.sqrt(
        mean_squared_error(
            y_va,
            pred
        )
    )

    broad_fold_rmse.append(fold_rmse)

    print(f"F{fold}: {fold_rmse:.6f}")

broad_oof_rmse = np.sqrt(
    mean_squared_error(
        y,
        broad_oof
    )
)

broad_residual = y.to_numpy() - broad_oof

print(f"\nModel C fold-mean RMSE: {np.mean(broad_fold_rmse):.6f}")
print(f"Model C OOF RMSE:       {broad_oof_rmse:.6f}")
print(f"Champion OOF RMSE:      {cat_oof_rmse:.6f}")
print(
    f"Residual correlation:   "
    f"{np.corrcoef(cat_residual, broad_residual)[0, 1]:.6f}"
)

F1: 1066.754221
F2: 1051.734868
F3: 1085.786040
F4: 1107.416395
F5: 1059.573353

Model C fold-mean RMSE: 1074.252976
Model C OOF RMSE:       1074.437471
Champion OOF RMSE:      1075.086399
Residual correlation:   0.999295


In [ ]:
# Aim: Quantify the observations affected by the Non-Edible fat-content
# representation before testing it in a model.

non_edible_categories = [
    "health and hygiene",
    "household",
    "others"
]

non_edible_mask = (
    X["product_category_clean"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(non_edible_categories)
)

non_edible_diagnostic = pd.DataFrame({
    "product_category_clean": X["product_category_clean"],
    "fat_content": X["fat_content"],
    "non_edible": non_edible_mask,
    "total_sales": y
})

print("Non-Edible observations:", non_edible_mask.sum())
print(
    "Share of dataset:",
    f"{100 * non_edible_mask.mean():.2f}%"
)

display(
    pd.crosstab(
        non_edible_diagnostic["product_category_clean"],
        non_edible_diagnostic["fat_content"],
        margins=True
    )
)

print("\nSales summary by Non-Edible status:")
display(
    non_edible_diagnostic
    .groupby("non_edible")["total_sales"]
    .agg(["count", "mean", "median", "std"])
    .round(2)
)

Non-Edible observations: 1284
Share of dataset: 18.83%


fat_content,Low Fat,Regular,All
product_category_clean,,,
baking goods,263,253,516
breads,121,88,209
breakfast,30,55,85
canned,265,252,517
dairy,345,209,554
frozen foods,358,317,675
fruits and vegetables,509,497,1006
hard drinks,169,0,169
health and hygiene,412,0,412



Sales summary by Non-Edible status:


,count,mean,median,std
non_edible,,,,
False,5534,2177.73,1771.38,1708.86
True,1284,2161.95,1919.09,1650.41


In [ ]:
# Aim: Create a semantic fat-content representation where Low Fat is
# treated as Non-Edible for categories where fat content is not applicable.

X_fat_semantic = X[broad_features].copy()

non_edible_categories = [
    "health and hygiene",
    "household",
    "others"
]

non_edible_mask = (
    X_fat_semantic["product_category_clean"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(non_edible_categories)
)

X_fat_semantic["fat_content_semantic"] = np.where(
    non_edible_mask,
    "Non-Edible",
    X_fat_semantic["fat_content"].astype(str)
)

# Remove the original representation
X_fat_semantic = X_fat_semantic.drop(columns=["fat_content"])

cat_features_fat_semantic = [
    "product_code",
    "fat_content_semantic",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format"
]

for col in cat_features_fat_semantic:
    X_fat_semantic[col] = (
        X_fat_semantic[col]
        .astype(object)
        .where(X_fat_semantic[col].notna(), "Missng")
    )

print("Model D shape:", X_fat_semantic.shape)
print("Categorical columns:", cat_features_fat_semantic)
print("\nSemantic fat-content distribution:")
display(
    X_fat_semantic["fat_content_semantic"]
    .value_counts(dropna=False)
)

Model D shape: (6818, 11)
Categorical columns: ['product_code', 'fat_content_semantic', 'product_category_clean', 'store_code', 'store_size', 'store_location_tier', 'store_format']

Semantic fat-content distribution:


fat_content_semantic
Low Fat       3141
Regular       2393
Non-Edible    1284
Name: count, dtype: int64

In [ ]:
# Aim: Evaluate the semantic fat-content representation with the exact
# locked 5-fold CatBoost protocol and measure ensemble diversity.

fat_oof = np.zeros(len(X_fat_semantic))
fat_fold_rmse = []

for fold, (train_idx, val_idx) in enumerate(
    kf.split(X_fat_semantic, y), start=1
):
    X_tr = X_fat_semantic.iloc[train_idx].copy()
    X_va = X_fat_semantic.iloc[val_idx].copy()

    y_tr = y.iloc[train_idx]
    y_va = y.iloc[val_idx]

    model = CatBoostRegressor(
        boosting_type="Ordered",
        iterations=300,
        learning_rate=0.025,
        depth=4,
        l2_leaf_reg=1,
        random_strength=1.0,
        loss_function="RMSE",
        random_seed=42,
        verbose=False
    )

    model.fit(
        X_tr,
        y_tr,
        cat_features=cat_features_fat_semantic
    )

    pred = model.predict(X_va)

    fat_oof[val_idx] = pred

    fold_rmse = np.sqrt(
        mean_squared_error(
            y_va,
            pred
        )
    )

    fat_fold_rmse.append(fold_rmse)

    print(f"F{fold}: {fold_rmse:.6f}")

fat_oof_rmse = np.sqrt(
    mean_squared_error(
        y,
        fat_oof
    )
)

fat_residual = y.to_numpy() - fat_oof

print(f"\nModel D fold-mean RMSE: {np.mean(fat_fold_rmse):.6f}")
print(f"Model D OOF RMSE:       {fat_oof_rmse:.6f}")
print(f"Champion OOF RMSE:      {cat_oof_rmse:.6f}")

print(
    f"Residual correlation:   "
    f"{np.corrcoef(cat_residual, fat_residual)[0, 1]:.6f}"
)

F1: 1066.883894
F2: 1052.359843
F3: 1087.942685
F4: 1108.244889
F5: 1062.433859

Model D fold-mean RMSE: 1075.573034
Model D OOF RMSE:       1075.756764
Champion OOF RMSE:      1075.086399
Residual correlation:   0.999331


In [ ]:
# Aim: Test whether the shelf-visibility × product-price interaction provides
# additional predictive signal beyond the frozen CatBoost feature set.

X_vis_price = X[broad_features].copy()

X_vis_price["visibility_to_price"] = (
    X_vis_price["shelf_visibility"] * X_vis_price["product_price"]
)

cat_features_vis_price = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format"
]

for col in cat_features_vis_price:
    X_vis_price[col] = (
        X_vis_price[col]
        .astype(object)
        .where(X_vis_price[col].notna(), "Missng")
    )

print("Shape:", X_vis_price.shape)
print("New feature summary:")
print(X_vis_price["visibility_to_price"].describe())
print("\nMissing values:")
print(X_vis_price["visibility_to_price"].isna().sum())

Shape: (6818, 12)
New feature summary:
count    6818.000000
mean        9.217343
std         8.902052
min         0.000000
25%         2.945857
50%         6.342222
75%        13.129652
max        80.377720
Name: visibility_to_price, dtype: float64

Missing values:
0


In [ ]:
# Aim: Evaluate whether visibility_to_price improves the frozen CatBoost
# model under the exact locked 5-fold validation protocol.

vis_price_oof = np.zeros(len(X_vis_price))
vis_price_fold_rmse = []

for fold, (train_idx, val_idx) in enumerate(
    kf.split(X_vis_price, y), start=1
):
    X_tr = X_vis_price.iloc[train_idx].copy()
    X_va = X_vis_price.iloc[val_idx].copy()
    y_tr = y.iloc[train_idx]
    y_va = y.iloc[val_idx]

    model = CatBoostRegressor(
        boosting_type="Ordered",
        iterations=300,
        learning_rate=0.025,
        depth=4,
        l2_leaf_reg=1,
        random_strength=1.0,
        loss_function="RMSE",
        random_seed=42,
        verbose=False
    )

    model.fit(
        X_tr,
        y_tr,
        cat_features=cat_features_vis_price
    )

    pred = model.predict(X_va)

    vis_price_oof[val_idx] = pred

    fold_rmse = np.sqrt(mean_squared_error(y_va, pred))
    vis_price_fold_rmse.append(fold_rmse)

    print(f"F{fold}: {fold_rmse:.6f}")

vis_price_oof_rmse = np.sqrt(
    mean_squared_error(y, vis_price_oof)
)

vis_price_residual = y.to_numpy() - vis_price_oof

print(f"\nModel visibility_to_price fold-mean RMSE: "
      f"{np.mean(vis_price_fold_rmse):.6f}")

print(f"Model visibility_to_price OOF RMSE:       "
      f"{vis_price_oof_rmse:.6f}")

print(f"Champion OOF RMSE:                         "
      f"{cat_oof_rmse:.6f}")

print(
    f"OOF improvement:                           "
    f"{cat_oof_rmse - vis_price_oof_rmse:.6f}"
)

print(
    f"Residual correlation with champion:       "
    f"{np.corrcoef(cat_residual, vis_price_residual)[0, 1]:.6f}"
)

F1: 1065.851559
F2: 1053.209641
F3: 1086.961096
F4: 1108.731918
F5: 1060.714645

Model visibility_to_price fold-mean RMSE: 1075.093772
Model visibility_to_price OOF RMSE:       1075.280967
Champion OOF RMSE:                         1075.086399
OOF improvement:                           -0.194568
Residual correlation with champion:       0.999407


In [ ]:
# Aim: Construct fold-safe item_vis_ratio features using only
# training-fold product visibility means, with global-mean fallback
# for products unseen within a training fold.

item_vis_oof = np.zeros(len(X))
item_vis_fold_rmse = []

for fold, (train_idx, val_idx) in enumerate(
    kf.split(X, y), start=1
):
    X_tr = X.iloc[train_idx].copy()
    X_va = X.iloc[val_idx].copy()

    # Product-level visibility means learned only from the training fold
    product_visibility_mean = (
        X_tr.groupby("product_code")["shelf_visibility"]
        .mean()
    )

    global_visibility_mean = X_tr["shelf_visibility"].mean()

    # Map training and validation observations
    X_tr["item_vis_ratio"] = (
        X_tr["shelf_visibility"]
        / X_tr["product_code"].map(product_visibility_mean)
    )

    X_va["item_vis_ratio"] = (
        X_va["shelf_visibility"]
        / X_va["product_code"]
        .map(product_visibility_mean)
        .fillna(global_visibility_mean)
    )

    # We only need to inspect the feature construction here.
    # Model fitting comes after confirming the feature behaves sensibly.
    print(
        f"F{fold}: "
        f"train missing={X_tr['item_vis_ratio'].isna().sum()}, "
        f"validation missing={X_va['item_vis_ratio'].isna().sum()}, "
        f"train median={X_tr['item_vis_ratio'].median():.4f}, "
        f"validation median={X_va['item_vis_ratio'].median():.4f}"
    )

F1: train missing=13, validation missing=0, train median=0.9972, validation median=0.9957
F2: train missing=6, validation missing=0, train median=0.9969, validation median=0.9878
F3: train missing=13, validation missing=0, train median=0.9962, validation median=0.9928
F4: train missing=9, validation missing=0, train median=0.9958, validation median=0.9871
F5: train missing=6, validation missing=1, train median=0.9956, validation median=0.9952


In [ ]:
# Aim: Rebuild the fold-safe item_vis_ratio experiment using the exact
# locked 11-feature modeling set, preventing accidental inclusion of
# raw product_category.

item_vis_oof = np.zeros(len(X))
item_vis_fold_rmse = []

item_vis_features = [
    "product_code",
    "product_weight_kg",
    "fat_content",
    "shelf_visibility",
    "product_price",
    "store_code",
    "store_age_years",
    "store_size",
    "store_location_tier",
    "store_format",
    "product_category_clean"
]

cat_features_item_vis = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format"
]

for fold, (train_idx, val_idx) in enumerate(
    kf.split(X, y), start=1
):
    X_tr = X.iloc[train_idx][item_vis_features].copy()
    X_va = X.iloc[val_idx][item_vis_features].copy()

    y_tr = y.iloc[train_idx]
    y_va = y.iloc[val_idx]

    # Learn product visibility means from training fold only
    product_visibility_mean = (
        X_tr.groupby("product_code")["shelf_visibility"]
        .mean()
    )

    global_visibility_mean = X_tr["shelf_visibility"].mean()

    # Fold-safe relative visibility
    X_tr["item_vis_ratio"] = (
        X_tr["shelf_visibility"]
        / X_tr["product_code"].map(product_visibility_mean)
    )

    X_va["item_vis_ratio"] = (
        X_va["shelf_visibility"]
        / X_va["product_code"]
        .map(product_visibility_mean)
        .fillna(global_visibility_mean)
    )

    # Preserve genuine missing categorical values for CatBoost
    for col in cat_features_item_vis:
        X_tr[col] = (
            X_tr[col]
            .astype(object)
            .where(X_tr[col].notna(), "Missng")
        )

        X_va[col] = (
            X_va[col]
            .astype(object)
            .where(X_va[col].notna(), "Missng")
        )

    model = CatBoostRegressor(
        boosting_type="Ordered",
        iterations=300,
        learning_rate=0.025,
        depth=4,
        l2_leaf_reg=1,
        random_strength=1.0,
        loss_function="RMSE",
        random_seed=42,
        verbose=False
    )

    model.fit(
        X_tr,
        y_tr,
        cat_features=cat_features_item_vis
    )

    pred = model.predict(X_va)

    item_vis_oof[val_idx] = pred

    fold_rmse = np.sqrt(mean_squared_error(y_va, pred))
    item_vis_fold_rmse.append(fold_rmse)

    print(f"F{fold}: {fold_rmse:.6f}")

item_vis_oof_rmse = np.sqrt(
    mean_squared_error(y, item_vis_oof)
)

item_vis_residual = y.to_numpy() - item_vis_oof

print(
    f"\nModel item_vis_ratio fold-mean RMSE: "
    f"{np.mean(item_vis_fold_rmse):.6f}"
)

print(
    f"Model item_vis_ratio OOF RMSE:       "
    f"{item_vis_oof_rmse:.6f}"
)

print(
    f"Champion OOF RMSE:                    "
    f"{cat_oof_rmse:.6f}"
)

print(
    f"OOF improvement:                      "
    f"{cat_oof_rmse - item_vis_oof_rmse:.6f}"
)

print(
    f"Residual correlation with champion:   "
    f"{np.corrcoef(cat_residual, item_vis_residual)[0, 1]:.6f}"
)

F1: 1065.921211
F2: 1053.945913
F3: 1086.882556
F4: 1108.317126
F5: 1061.393211

Model item_vis_ratio fold-mean RMSE: 1075.292003
Model item_vis_ratio OOF RMSE:       1075.471578
Champion OOF RMSE:                    1075.086399
OOF improvement:                      -0.385179
Residual correlation with champion:   0.999387


In [ ]:
# Aim: Test whether coarse store-vintage regimes capture nonlinear store-age
# structure that the raw store_age_years feature does not.

X_vintage = X[[
    "product_code",
    "product_weight_kg",
    "fat_content",
    "shelf_visibility",
    "product_price",
    "store_code",
    "store_age_years",
    "store_size",
    "store_location_tier",
    "store_format",
    "product_category_clean"
]].copy()

X_vintage["store_vintage"] = pd.cut(
    X_vintage["store_age_years"],
    bins=[-1, 25, 35, 100],
    labels=["Young", "Mature", "Old"]
)

print(X_vintage["store_vintage"].value_counts(dropna=False).sort_index())
print("\nStore-vintage sales means:")
print(
    pd.DataFrame({
        "store_vintage": X_vintage["store_vintage"],
        "total_sales": y
    })
    .groupby("store_vintage", observed=False)["total_sales"]
    .agg(["count", "mean", "median", "std"])
)

store_vintage
Young     1486
Mature    3409
Old       1923
Name: count, dtype: int64

Store-vintage sales means:
               count         mean    median          std
store_vintage                                           
Young           1486  2169.013062  1809.095  1497.194482
Mature          3409  2063.771581  1728.100  1558.106332
Old             1923  2375.943625  1905.910  2031.252356


In [ ]:
# Aim: Test whether explicit store-age regimes add predictive structure beyond raw store_age_years.

X_vintage_model = X[[
    "product_code",
    "product_weight_kg",
    "fat_content",
    "shelf_visibility",
    "product_price",
    "store_code",
    "store_age_years",
    "store_size",
    "store_location_tier",
    "store_format",
    "product_category_clean"
]].copy()

X_vintage_model["store_vintage"] = pd.cut(
    X_vintage_model["store_age_years"],
    bins=[-1, 25, 35, 100],
    labels=["Young", "Mature", "Old"]
)

cat_features_vintage = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format",
    "store_vintage"
]

vintage_oof = np.zeros(len(X_vintage_model))

vintage_fold_rmse = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_vintage_model), start=1):

    X_train = X_vintage_model.iloc[train_idx].copy()
    X_val = X_vintage_model.iloc[val_idx].copy()

    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    for col in cat_features_vintage:
        X_train[col] = X_train[col].astype(object).where(
            X_train[col].notna(), "Missng"
        )
        X_val[col] = X_val[col].astype(object).where(
            X_val[col].notna(), "Missng"
        )

    model = CatBoostRegressor(
        boosting_type="Ordered",
        iterations=300,
        learning_rate=0.025,
        depth=4,
        l2_leaf_reg=1,
        random_strength=1.0,
        loss_function="RMSE",
        random_seed=42,
        verbose=False
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features_vintage
    )

    pred = model.predict(X_val)

    vintage_oof[val_idx] = pred

    fold_rmse = np.sqrt(mean_squared_error(y_val, pred))
    vintage_fold_rmse.append(fold_rmse)

    print(f"Fold {fold}: RMSE = {fold_rmse:.6f}")

vintage_oof_rmse = np.sqrt(mean_squared_error(y, vintage_oof))

print("\nFold RMSEs:")
print(vintage_fold_rmse)

print(f"\nMean fold RMSE: {np.mean(vintage_fold_rmse):.6f}")
print(f"OOF RMSE:       {vintage_oof_rmse:.6f}")

print(f"\nChampion OOF RMSE: {cat_oof_rmse:.6f}")
print(f"OOF difference:    {vintage_oof_rmse - cat_oof_rmse:+.6f}")

Fold 1: RMSE = 1066.852297
Fold 2: RMSE = 1051.985309
Fold 3: RMSE = 1087.541765
Fold 4: RMSE = 1109.439263
Fold 5: RMSE = 1062.460408

Fold RMSEs:
[np.float64(1066.852297022153), np.float64(1051.9853089386581), np.float64(1087.5417646061057), np.float64(1109.4392630741302), np.float64(1062.4604078487146)]

Mean fold RMSE: 1075.655808
OOF RMSE:       1075.847458

Champion OOF RMSE: 1075.086399
OOF difference:    +0.761059


In [ ]:
# Aim: Build strict OOF predictions from a one-hot Ridge model and test whether its residuals provide ensemble diversity.

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

ridge_features = [
    "product_code",
    "product_weight_kg",
    "fat_content",
    "shelf_visibility",
    "product_price",
    "store_code",
    "store_age_years",
    "store_size",
    "store_location_tier",
    "store_format",
    "product_category_clean"
]

ridge_cat = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format"
]

ridge_num = [
    "product_weight_kg",
    "shelf_visibility",
    "product_price",
    "store_age_years"
]

X_ridge = X[ridge_features].copy()

ridge_oof = np.zeros(len(X_ridge))
ridge_fold_rmse = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_ridge), start=1):

    X_train = X_ridge.iloc[train_idx].copy()
    X_val = X_ridge.iloc[val_idx].copy()

    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True
        ))
    ])

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    preprocessor = ColumnTransformer([
        ("cat", categorical_pipeline, ridge_cat),
        ("num", numeric_pipeline, ridge_num)
    ])

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("ridge", Ridge(alpha=10.0))
    ])

    model.fit(X_train, y_train)

    pred = model.predict(X_val)

    ridge_oof[val_idx] = pred

    fold_rmse = np.sqrt(mean_squared_error(y_val, pred))
    ridge_fold_rmse.append(fold_rmse)

    print(f"Fold {fold}: RMSE = {fold_rmse:.6f}")

ridge_oof_rmse = np.sqrt(mean_squared_error(y, ridge_oof))
ridge_residual = y.to_numpy() - ridge_oof

print("\nFold RMSEs:")
print(ridge_fold_rmse)

print(f"\nMean fold RMSE: {np.mean(ridge_fold_rmse):.6f}")
print(f"OOF RMSE:       {ridge_oof_rmse:.6f}")

print(f"\nChampion CatBoost OOF RMSE: {cat_oof_rmse:.6f}")
print(f"Difference:                {ridge_oof_rmse - cat_oof_rmse:+.6f}")

residual_corr_ridge = np.corrcoef(
    cat_residual,
    ridge_residual
)[0, 1]

print(f"\nCatBoost/Ridge residual correlation: {residual_corr_ridge:.6f}")

Fold 1: RMSE = 1131.333180
Fold 2: RMSE = 1099.593137
Fold 3: RMSE = 1149.277975
Fold 4: RMSE = 1177.318416
Fold 5: RMSE = 1125.699321

Fold RMSEs:
[np.float64(1131.3331804731174), np.float64(1099.5931372790499), np.float64(1149.277975309772), np.float64(1177.3184157095766), np.float64(1125.6993205699666)]

Mean fold RMSE: 1136.644406
OOF RMSE:       1136.933369

Champion CatBoost OOF RMSE: 1075.086399
Difference:                +61.846970

CatBoost/Ridge residual correlation: 0.940739


In [ ]:
# Aim: Determine whether Ridge's lower residual correlation provides enough complementary signal to improve the CatBoost OOF prediction.

from scipy.optimize import minimize

P_cat_ridge = np.column_stack([
    cat_oof,
    ridge_oof
])

def blend_rmse(weights):
    pred = P_cat_ridge @ weights
    return np.sqrt(mean_squared_error(y, pred))

result_cat_ridge = minimize(
    blend_rmse,
    x0=np.array([0.9, 0.1]),
    bounds=[(0, 1), (0, 1)],
    constraints={
        "type": "eq",
        "fun": lambda w: np.sum(w) - 1
    },
    method="SLSQP"
)

best_weights_cat_ridge = result_cat_ridge.x
best_blend_rmse_cat_ridge = result_cat_ridge.fun

print("Optimized weights:")
print(f"CatBoost: {best_weights_cat_ridge[0]:.6f}")
print(f"Ridge:    {best_weights_cat_ridge[1]:.6f}")

print(f"\nCatBoost OOF RMSE: {cat_oof_rmse:.6f}")
print(f"Ridge OOF RMSE:    {ridge_oof_rmse:.6f}")
print(f"Blend OOF RMSE:    {best_blend_rmse_cat_ridge:.6f}")

print(
    f"\nBlend improvement vs CatBoost: "
    f"{cat_oof_rmse - best_blend_rmse_cat_ridge:+.6f}"
)

Optimized weights:
CatBoost: 0.959947
Ridge:    0.040053

CatBoost OOF RMSE: 1075.086399
Ridge OOF RMSE:    1136.933369
Blend OOF RMSE:    1074.975447

Blend improvement vs CatBoost: +0.110952


In [ ]:
# Aim: Generate strict OOF ExtraTrees predictions and measure both standalone performance and residual diversity versus CatBoost and Ridge.

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

extra_features = [
    "product_code",
    "product_weight_kg",
    "fat_content",
    "shelf_visibility",
    "product_price",
    "store_code",
    "store_age_years",
    "store_size",
    "store_location_tier",
    "store_format",
    "product_category_clean"
]

extra_cat = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format"
]

extra_num = [
    "product_weight_kg",
    "shelf_visibility",
    "product_price",
    "store_age_years"
]

X_extra = X[extra_features].copy()

extra_oof = np.zeros(len(X_extra))
extra_fold_rmse = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_extra), start=1):

    X_train = X_extra.iloc[train_idx].copy()
    X_val = X_extra.iloc[val_idx].copy()

    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ))
    ])

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])

    preprocessor = ColumnTransformer([
        ("cat", categorical_pipeline, extra_cat),
        ("num", numeric_pipeline, extra_num)
    ])

    X_train_enc = preprocessor.fit_transform(X_train)
    X_val_enc = preprocessor.transform(X_val)

    model = ExtraTreesRegressor(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=2,
        max_features=0.8,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train_enc, y_train)

    pred = model.predict(X_val_enc)

    extra_oof[val_idx] = pred

    fold_rmse = np.sqrt(mean_squared_error(y_val, pred))
    extra_fold_rmse.append(fold_rmse)

    print(f"Fold {fold}: RMSE = {fold_rmse:.6f}")

extra_oof_rmse = np.sqrt(mean_squared_error(y, extra_oof))
extra_residual = y.to_numpy() - extra_oof

cat_extra_corr = np.corrcoef(
    cat_residual,
    extra_residual
)[0, 1]

ridge_extra_corr = np.corrcoef(
    ridge_residual,
    extra_residual
)[0, 1]

print("\nFold RMSEs:")
print(extra_fold_rmse)

print(f"\nMean fold RMSE: {np.mean(extra_fold_rmse):.6f}")
print(f"OOF RMSE:       {extra_oof_rmse:.6f}")

print(f"\nCatBoost OOF RMSE: {cat_oof_rmse:.6f}")
print(f"ExtraTrees difference: {extra_oof_rmse - cat_oof_rmse:+.6f}")

print(f"\nCatBoost/ExtraTrees residual correlation: {cat_extra_corr:.6f}")
print(f"Ridge/ExtraTrees residual correlation:    {ridge_extra_corr:.6f}")

Fold 1: RMSE = 1161.380795
Fold 2: RMSE = 1105.296561
Fold 3: RMSE = 1170.035758
Fold 4: RMSE = 1190.965926
Fold 5: RMSE = 1146.011534

Fold RMSEs:
[np.float64(1161.3807950257515), np.float64(1105.296561255975), np.float64(1170.0357580347907), np.float64(1190.9659262900818), np.float64(1146.011534331851)]

Mean fold RMSE: 1154.738115
OOF RMSE:       1155.090074

CatBoost OOF RMSE: 1075.086399
ExtraTrees difference: +80.003675

CatBoost/ExtraTrees residual correlation: 0.925875
Ridge/ExtraTrees residual correlation:    0.888614


In [ ]:
# Validate whether the CatBoost + Ridge ensemble improvement survives fold-level out-of-sample weight selection.

cat_arr = np.asarray(cat_oof)
ridge_arr = np.asarray(ridge_oof)
y_arr = np.asarray(y)

meta_pred = np.zeros(len(y_arr))
meta_weights = []

for fold, (_, val_idx) in enumerate(kf.split(X), start=1):

    train_meta_idx = np.setdiff1d(
        np.arange(len(y_arr)),
        val_idx
    )

    P_train = np.column_stack([
        cat_arr[train_meta_idx],
        ridge_arr[train_meta_idx]
    ])

    y_train_meta = y_arr[train_meta_idx]

    def meta_rmse(w):
        pred = P_train @ np.array([1 - w[0], w[0]])
        return np.sqrt(mean_squared_error(y_train_meta, pred))

    result = minimize(
        meta_rmse,
        x0=np.array([0.04]),
        bounds=[(0, 1)],
        method="SLSQP"
    )

    ridge_weight = result.x[0]
    cat_weight = 1 - ridge_weight

    val_pred = (
        cat_weight * cat_arr[val_idx]
        + ridge_weight * ridge_arr[val_idx]
    )

    meta_pred[val_idx] = val_pred
    meta_weights.append(ridge_weight)

    fold_rmse = np.sqrt(
        mean_squared_error(y_arr[val_idx], val_pred)
    )

    cat_fold_rmse = np.sqrt(
        mean_squared_error(y_arr[val_idx], cat_arr[val_idx])
    )

    print(
        f"Fold {fold}: "
        f"Ridge weight = {ridge_weight:.6f} | "
        f"CatBoost RMSE = {cat_fold_rmse:.6f} | "
        f"Blend RMSE = {fold_rmse:.6f} | "
        f"Gain = {cat_fold_rmse - fold_rmse:+.6f}"
    )

nested_blend_rmse = np.sqrt(
    mean_squared_error(y_arr, meta_pred)
)

print("\nRidge weights by held-out fold:")
print(meta_weights)

print(f"\nMean Ridge weight: {np.mean(meta_weights):.6f}")
print(f"Nested blend OOF RMSE: {nested_blend_rmse:.6f}")
print(f"CatBoost OOF RMSE:     {cat_oof_rmse:.6f}")
print(
    f"Nested blend improvement: "
    f"{cat_oof_rmse - nested_blend_rmse:+.6f}"
)

Fold 1: Ridge weight = 0.039600 | CatBoost RMSE = 1065.990810 | Blend RMSE = 1065.863911 | Gain = +0.126899
Fold 2: Ridge weight = 0.005083 | CatBoost RMSE = 1053.887357 | Blend RMSE = 1053.760105 | Gain = +0.127251
Fold 3: Ridge weight = 0.043655 | CatBoost RMSE = 1085.198997 | Blend RMSE = 1085.174177 | Gain = +0.024820
Fold 4: Ridge weight = 0.064932 | CatBoost RMSE = 1107.592529 | Blend RMSE = 1108.409793 | Gain = -0.817263
Fold 5: Ridge weight = 0.046366 | CatBoost RMSE = 1061.909675 | Blend RMSE = 1061.969586 | Gain = -0.059911

Ridge weights by held-out fold:
[np.float64(0.0396), np.float64(0.005083342157032118), np.float64(0.04365470338233395), np.float64(0.06493226207256798), np.float64(0.046365624369013494)]

Mean Ridge weight: 0.039927
Nested blend OOF RMSE: 1075.211468
CatBoost OOF RMSE:     1075.086399
Nested blend improvement: -0.125069


In [ ]:
# Aim: Determine whether CatBoost's price-related error persists within products, rather than being only a between-product effect.

error_diag = X[[
    "product_code",
    "store_code",
    "product_price"
]].copy()

error_diag["actual"] = y_arr
error_diag["prediction"] = cat_arr
error_diag["residual"] = error_diag["actual"] - error_diag["prediction"]
error_diag["abs_residual"] = np.abs(error_diag["residual"])

# Global price quintiles
error_diag["price_quintile"] = pd.qcut(
    error_diag["product_price"],
    q=5,
    labels=["Q1", "Q2", "Q3", "Q4", "Q5"]
)

print("Global price-quintile error:")
print(
    error_diag
    .groupby("price_quintile", observed=True)
    .agg(
        count=("actual", "size"),
        mean_price=("product_price", "mean"),
        mean_residual=("residual", "mean"),
        mae=("abs_residual", "mean"),
        rmse=("residual", lambda x: np.sqrt(np.mean(x**2)))
    )
)

# Product-level median price learned from the full product history
# This is diagnostic only, not a feature/modeling step.
product_price_median = (
    error_diag
    .groupby("product_code")["product_price"]
    .transform("median")
)

error_diag["price_deviation_from_product"] = (
    error_diag["product_price"] - product_price_median
)

error_diag["price_abs_deviation_from_product"] = (
    np.abs(error_diag["price_deviation_from_product"])
)

print("\nWithin-product price deviation:")
print(
    error_diag[
        ["price_deviation_from_product",
         "price_abs_deviation_from_product",
         "residual",
         "abs_residual"]
    ].corr(method="spearman")
)

print("\nAbsolute within-product price deviation quintiles:")
error_diag["price_dev_quintile"] = pd.qcut(
    error_diag["price_abs_deviation_from_product"],
    q=5,
    labels=["Q1", "Q2", "Q3", "Q4", "Q5"],
    duplicates="drop"
)

print(
    error_diag
    .groupby("price_dev_quintile", observed=True)
    .agg(
        count=("actual", "size"),
        mean_dev=("price_abs_deviation_from_product", "mean"),
        mean_residual=("residual", "mean"),
        mae=("abs_residual", "mean"),
        rmse=("residual", lambda x: np.sqrt(np.mean(x**2)))
    )
)

Global price-quintile error:
                count  mean_price  mean_residual          mae         rmse
price_quintile                                                            
Q1               1364   54.463262     -25.556433   299.227883   407.760826
Q2               1363  101.357997       2.193248   536.350034   704.927549
Q3               1365  138.669707       8.109530   731.921178   961.712495
Q4               1362  177.278884     -14.956284   955.499244  1251.467709
Q5               1364  230.624875      11.285112  1253.630937  1620.212233

Within-product price deviation:
                                  price_deviation_from_product  price_abs_deviation_from_product  residual  abs_residual
price_deviation_from_product                          1.000000                         -0.000511 -0.030618      0.004184
price_abs_deviation_from_product                     -0.000511                          1.000000  0.006709      0.117904
residual                                          

In [ ]:
# Aim: Test whether CatBoost predictions are systematically compressed toward the center of the target distribution.

calibration_diag = pd.DataFrame({
    "actual": y_arr,
    "prediction": cat_arr
})

calibration_diag["residual"] = (
    calibration_diag["actual"] - calibration_diag["prediction"]
)

calibration_diag["abs_residual"] = (
    np.abs(calibration_diag["residual"])
)

calibration_diag["prediction_bin"] = pd.qcut(
    calibration_diag["prediction"],
    q=10,
    labels=[
        "P01", "P02", "P03", "P04", "P05",
        "P06", "P07", "P08", "P09", "P10"
    ]
)

calibration_summary = (
    calibration_diag
    .groupby("prediction_bin", observed=True)
    .agg(
        count=("actual", "size"),
        mean_prediction=("prediction", "mean"),
        mean_actual=("actual", "mean"),
        mean_residual=("residual", "mean"),
        mae=("abs_residual", "mean"),
        rmse=("residual", lambda x: np.sqrt(np.mean(x**2)))
    )
)

print(calibration_summary)

print("\nOverall prediction/target correlation:")
print(
    calibration_diag[
        ["prediction", "actual"]
    ].corr(method="spearman")
)

print("\nResidual vs prediction correlation:")
print(
    calibration_diag[
        ["prediction", "residual", "abs_residual"]
    ].corr(method="spearman")
)

                count  mean_prediction  mean_actual  mean_residual          mae         rmse
prediction_bin                                                                              
P01               682       274.476441   283.665997       9.189556   131.558217   176.301675
P02               682       684.429177   634.930909     -49.498267   252.573286   313.270716
P03               682      1090.075449  1074.909208     -15.166241   388.743094   498.802973
P04               681      1535.184171  1532.999853      -2.184318   531.749088   670.538500
P05               682      1881.378122  1903.069282      21.691159   709.339624   876.029197
P06               682      2307.354952  2314.274809       6.919857   787.500225   978.120108
P07               681      2755.566701  2729.295198     -26.271502   950.329522  1187.548708
P08               682      3116.627271  3101.097317     -15.529955  1054.709123  1322.780181
P09               682      3620.128469  3635.040411      14.911942  13

In [ ]:
# Aim: Identify whether unexplained error is concentrated in specific category × store-format × price regimes.

interaction_diag = pd.DataFrame({
    "product_category_clean": X["product_category_clean"].values,
    "store_format": X["store_format"].values,
    "product_price": X["product_price"].values,
    "actual": y_arr,
    "prediction": cat_arr
})

interaction_diag["residual"] = (
    interaction_diag["actual"] - interaction_diag["prediction"]
)

interaction_diag["abs_residual"] = (
    np.abs(interaction_diag["residual"])
)

interaction_diag["price_quintile"] = pd.qcut(
    interaction_diag["product_price"],
    q=5,
    labels=["Q1", "Q2", "Q3", "Q4", "Q5"]
)

interaction_summary = (
    interaction_diag
    .groupby(
        [
            "product_category_clean",
            "store_format",
            "price_quintile"
        ],
        observed=True
    )
    .agg(
        count=("actual", "size"),
        mean_price=("product_price", "mean"),
        mean_residual=("residual", "mean"),
        mae=("abs_residual", "mean"),
        rmse=("residual", lambda x: np.sqrt(np.mean(x**2)))
    )
    .reset_index()
)

# Keep only reasonably populated groups so tiny groups do not dominate the diagnosis.
interaction_summary = interaction_summary[
    interaction_summary["count"] >= 20
].copy()

interaction_summary = interaction_summary.sort_values(
    "rmse",
    ascending=False
)

print("Highest-error structural regimes:")
print(
    interaction_summary.head(25).to_string(index=False)
)

print("\nNumber of retained groups:", len(interaction_summary))

print("\nCorrelation between group mean price and group RMSE:")
print(
    interaction_summary[
        ["mean_price", "rmse"]
    ].corr(method="spearman")
)

Highest-error structural regimes:
product_category_clean         store_format price_quintile  count  mean_price  mean_residual         mae        rmse
 fruits and vegetables Flagship Hypermarket             Q5     22  225.555455     832.596432 1922.132319 2441.426736
    health and hygiene Standard Supermarket             Q5     35  227.181429     240.545661 1470.739772 2002.717477
           snack foods Flagship Hypermarket             Q4     23  175.424783    -133.483592 1591.010721 1961.398545
           soft drinks Standard Supermarket             Q5     27  223.575926     807.646973 1521.665516 1956.826788
          frozen foods Standard Supermarket             Q5     97  235.992474     204.011730 1465.443114 1837.620443
 fruits and vegetables Standard Supermarket             Q5    139  229.634317     154.817686 1438.238468 1791.223455
         starchy foods Standard Supermarket             Q4     22  176.695909     793.111892 1445.892836 1740.923940
           snack foods        

In [ ]:
# Aim: Test whether the highest-error category × store-format × price regimes have stable residual patterns across validation folds.

stability_diag = interaction_diag.copy()

stability_diag["fold"] = -1

for fold_id, (_, val_idx) in enumerate(kf.split(X), start=1):
    stability_diag.loc[val_idx, "fold"] = fold_id

# Identify the 10 highest-RMSE structural regimes from the previous diagnostic.
top_regimes = (
    interaction_summary
    .head(10)[
        [
            "product_category_clean",
            "store_format",
            "price_quintile"
        ]
    ]
    .copy()
)

# Keep only observations belonging to those regimes.
stability_top = stability_diag.merge(
    top_regimes,
    on=[
        "product_category_clean",
        "store_format",
        "price_quintile"
    ],
    how="inner"
)

fold_stability = (
    stability_top
    .groupby(
        [
            "product_category_clean",
            "store_format",
            "price_quintile",
            "fold"
        ],
        observed=True
    )
    .agg(
        count=("actual", "size"),
        mean_residual=("residual", "mean"),
        mae=("abs_residual", "mean"),
        rmse=("residual", lambda x: np.sqrt(np.mean(x**2)))
    )
    .reset_index()
)

print(
    fold_stability
    .sort_values(
        [
            "product_category_clean",
            "store_format",
            "price_quintile",
            "fold"
        ]
    )
    .to_string(index=False)
)

product_category_clean         store_format price_quintile  fold  count  mean_residual         mae        rmse
                breads Standard Supermarket             Q5     1      7     795.042542 1341.087038 1553.133115
                breads Standard Supermarket             Q5     2      6     483.357230 1692.965736 1902.561559
                breads Standard Supermarket             Q5     3      6    -506.372693 1784.409709 1878.250161
                breads Standard Supermarket             Q5     4      4    -447.114378  849.608876  942.210348
                breads Standard Supermarket             Q5     5      8    -197.975695 1457.716272 1757.552490
                 dairy Standard Supermarket             Q5     1     26     319.154651 1645.411629 1909.806494
                 dairy Standard Supermarket             Q5     2     22    -558.862401 1081.617566 1354.977337
                 dairy Standard Supermarket             Q5     3     21    -553.765013 1193.164314 1518.350707
 

In [ ]:
# Rebuild the log-target OOF experiment using the exact locked 11-feature model matrix and prevent accidental extra columns.

log_features = [
    "product_code",
    "product_weight_kg",
    "fat_content",
    "shelf_visibility",
    "product_price",
    "store_code",
    "store_age_years",
    "store_size",
    "store_location_tier",
    "store_format",
    "product_category_clean"
]

X_log = X[log_features].copy()

# Ensure CatBoost categorical columns have explicit string values for missing entries.
for col in cat_features_model:
    X_log[col] = (
        X_log[col]
        .astype(object)
        .where(X_log[col].notna(), "Missng")
    )

log_oof = np.full(len(y), np.nan)

log_params = {
    "boosting_type": "Ordered",
    "iterations": 300,
    "learning_rate": 0.025,
    "depth": 4,
    "l2_leaf_reg": 1,
    "random_strength": 1.0,
    "loss_function": "RMSE",
    "random_seed": 42
}

for fold_id, (train_idx, val_idx) in enumerate(kf.split(X_log), start=1):

    X_train = X_log.iloc[train_idx]
    X_val = X_log.iloc[val_idx]

    y_train_log = np.log1p(y.iloc[train_idx])

    model = CatBoostRegressor(
        **log_params,
        verbose=False,
        allow_writing_files=False
    )

    model.fit(
        X_train,
        y_train_log,
        cat_features=cat_features_model
    )

    log_oof[val_idx] = np.expm1(
        model.predict(X_val)
    )

    fold_rmse = np.sqrt(
        mean_squared_error(
            y.iloc[val_idx],
            log_oof[val_idx]
        )
    )

    print(f"Fold {fold_id}: RMSE = {fold_rmse:.6f}")

log_oof_rmse = np.sqrt(
    mean_squared_error(y, log_oof)
)

log_residual = y_arr - log_oof

print(f"\nLog-target OOF RMSE: {log_oof_rmse:.6f}")

print("\nResidual correlation with raw-target CatBoost:")
print(
    pd.DataFrame({
        "cat_residual": cat_residual,
        "log_residual": log_residual
    }).corr(method="spearman")
)

print("\nAbsolute-residual correlation:")
print(
    pd.DataFrame({
        "cat_abs_residual": np.abs(cat_residual),
        "log_abs_residual": np.abs(log_residual)
    }).corr(method="spearman")
)

Fold 1: RMSE = 1118.941025
Fold 2: RMSE = 1088.012621
Fold 3: RMSE = 1106.012332


In [23]:
# Aim: Rebuild the exact locked feature matrix and run the first leakage-safe Lasso benchmark.

from sklearn.linear_model import Lasso
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

lasso_features = [
    "product_code",
    "product_weight_kg",
    "fat_content",
    "shelf_visibility",
    "product_price",
    "store_code",
    "store_age_years",
    "store_size",
    "store_location_tier",
    "store_format",
    "product_category_clean"
]

X_lasso = X[lasso_features].copy()

lasso_categorical = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format"
]

lasso_numeric = [
    "product_weight_kg",
    "shelf_visibility",
    "product_price",
    "store_age_years"
]

lasso_oof = np.full(len(y), np.nan)

for fold_id, (train_idx, val_idx) in enumerate(kf.split(X_lasso), start=1):

    X_train = X_lasso.iloc[train_idx]
    X_val = X_lasso.iloc[val_idx]
    y_train = y.iloc[train_idx]

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "cat",
                Pipeline([
                    (
                        "imputer",
                        SimpleImputer(strategy="most_frequent")
                    ),
                    (
                        "onehot",
                        OneHotEncoder(
                            handle_unknown="ignore",
                            sparse_output=True
                        )
                    )
                ]),
                lasso_categorical
            ),
            (
                "num",
                Pipeline([
                    (
                        "imputer",
                        SimpleImputer(strategy="median")
                    ),
                    (
                        "scaler",
                        StandardScaler()
                    )
                ]),
                lasso_numeric
            )
        ]
    )

    lasso_model = Lasso(
        alpha=1.0,
        max_iter=10000
    )

    lasso_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", lasso_model)
    ])

    lasso_pipeline.fit(X_train, y_train)

    lasso_oof[val_idx] = lasso_pipeline.predict(X_val)

    fold_rmse = np.sqrt(
        mean_squared_error(
            y.iloc[val_idx],
            lasso_oof[val_idx]
        )
    )

    print(f"Fold {fold_id}: RMSE = {fold_rmse:.6f}")

lasso_oof_rmse = np.sqrt(
    mean_squared_error(y, lasso_oof)
)

lasso_residual = y_arr - lasso_oof

print(f"\nLasso OOF RMSE: {lasso_oof_rmse:.6f}")

print("\nCatBoost vs Lasso residual correlation:")
print(
    pd.DataFrame({
        "cat_residual": cat_residual,
        "lasso_residual": lasso_residual
    }).corr(method="spearman")
)

Fold 1: RMSE = 1131.002431
Fold 2: RMSE = 1103.064666
Fold 3: RMSE = 1143.059926
Fold 4: RMSE = 1170.314831
Fold 5: RMSE = 1119.860231


NameError: name 'y_arr' is not defined

In [24]:
# Aim: Finish the Lasso OOF evaluation using the already-computed predictions.

y_arr = np.asarray(y)

lasso_residual = y_arr - lasso_oof

lasso_oof_rmse = np.sqrt(
    mean_squared_error(y_arr, lasso_oof)
)

print(f"Lasso OOF RMSE: {lasso_oof_rmse:.6f}")

print("\nCatBoost vs Lasso residual correlation:")
print(
    pd.DataFrame({
        "cat_residual": cat_residual,
        "lasso_residual": lasso_residual
    }).corr(method="spearman")
)

Lasso OOF RMSE: 1133.683276

CatBoost vs Lasso residual correlation:
                cat_residual  lasso_residual
cat_residual        1.000000        0.922863
lasso_residual      0.922863        1.000000
